# When Foundation Models Fall Short: From Off-the-Shelf AI to Problem-Specific Adaptation

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#4A148C;color:#FFFFFF;border-radius:6px;padding:18px 20px;margin:6px 0;width:100%;box-sizing:border-box;font-family:inherit;line-height:1.5"><div style="font-size:2em;font-weight:bold;line-height:1.2;margin-bottom:6px">When Foundation Models Fall Short</div><p><strong>From Off-the-Shelf AI to Problem-Specific Adaptation</strong></p>
<p>AMLD 2027 · EPFL, 11 March 2027 &nbsp;·&nbsp; SDS 2027 · Zurich, 1 June 2027</p>
<p><strong>The question:</strong> what can modern AI solve out of the box, when do its limitations
start to matter, and what should we do next?</p>
<p><strong>At the end of this notebook you will be able to:</strong></p>
<ol>
<li>Judge when a vision foundation model is good enough out of the box, and recognise
   the signs that it is not.</li>
<li>Adapt it with cheap, incomplete supervision (scribbles, missing labels,
   pseudo-labels, explanations, prompts), and explain what each approach uses and assumes.</li>
<li>Compare approaches on accuracy, annotation effort, reliability and compute, and
   decide what to do next on a new problem, including your own images.</li>
</ol>
<p>Levels: 🟢 beginner · 🟡 intermediate (the default path) · 🔴 advanced, optional.
&nbsp; ✍️ = you write a few lines &nbsp; 🖍️ = you draw &nbsp; 👥 = compare with your neighbour.</p>
<p><strong>Run <code>setup_check.ipynb</code> first.</strong> It was shared with you the week
before: it puts the models, images and precomputed files in your Google Drive, so nothing is downloaded
in the room. If you have not run it, do it now (about 5 minutes), then come back here.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>Load the cached models and data; see the question and outcomes.</p></div>"""))

**You need** basic ML (training, validation, evaluation) and some Python, no computer-vision background, and
this notebook in Colab with **Runtime → Change runtime type → T4 GPU**. Every step also runs on a CPU, just
slower, and each training cell offers a precomputed checkpoint.

**The plan.** Block 1 (sections 1–3): a foundation model segments pets almost perfectly, then meets a
specialised task and falls short. Block 2 (sections 4–10): the ways out, from scribbles to prompting the
model directly, all compared on the same hidden ground truth, then on your own images.

## 0. Setup & the question

This cell detects Colab and the GPU, fixes the seed, creates the `RESULTS` dictionary every section writes
into, and defines `use()`, so your own code never blocks the notebook. Run it once.

In [ ]:
# @title Setup (run me first) { display-mode: "form" }
import ast
import inspect
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NB_SMOKE") == "1"  # automated test: tiny data, CPU, seconds. Never set this by hand.
SEED = 42

# Where the cache lives. setup_check.ipynb filled MyDrive/fm-workshop-cache/ in Colab; the smoke run uses the
# small fixtures committed in data/smoke/; a local checkout uses data/cache/ (scripts/prepare_data.py).
if IN_COLAB and not SMOKE:
    CACHE_DIR = Path("/content/drive/MyDrive/fm-workshop-cache")
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception as err:  # noqa: BLE001 - one plain line, then a plain message below
        print(f"(Drive mount error: {type(err).__name__}: {err})")
    if not CACHE_DIR.is_dir():
        print(
            "⚠️ Google Drive is not mounted or the cache folder is missing. Run this cell again and click "
            "'Allow' in the pop-up. If the folder is still missing, run setup_check.ipynb first (5 minutes)."
        )
    if (CACHE_DIR / "models").is_dir():
        os.environ["HF_HUB_OFFLINE"] = "1"  # nothing is downloaded in the room (N-030)
    DATA_DIR, PRECOMPUTED = CACHE_DIR / "data", CACHE_DIR / "precomputed"
    SPECIALISED_JSON = CACHE_DIR / "specialised.json"
else:
    CACHE_DIR = Path("data/smoke" if SMOKE else "data/cache")
    DATA_DIR = CACHE_DIR if SMOKE else CACHE_DIR / "workshop"
    PRECOMPUTED = CACHE_DIR / "precomputed"
    SPECIALISED_JSON = Path("data/specialised.json")

# Model weights: plain files in <cache>/models/<org>--<name>/, written by setup_check.ipynb (no symlinks,
# so they survive Drive). `.fm-complete.json` is written last, once the folder is verified. No folder
# (smoke run, local checkout): the models load from the Hugging Face hub id instead.
MODELS_DIR = CACHE_DIR / "models"
MODEL_DONE = ".fm-complete.json"


def model_folder(repo_id: str) -> Path | None:
    """The cached plain-file folder of a Hugging Face model, or None when setup_check has not made one."""
    folder = MODELS_DIR / repo_id.replace("/", "--")
    return folder if (folder / MODEL_DONE).is_file() else None


def weights_file(folder: Path) -> Path:
    """The weight file inside a model folder (timm: model.safetensors)."""
    default = folder / "model.safetensors"
    return default if default.is_file() else sorted(folder.glob("*.safetensors"))[0]


if IN_COLAB:
    try:
        import pytorch_grad_cam  # noqa: F401 - the only package Colab lacks (SPEC §5); installed if missing
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q grad-cam")  # noqa: F821 - %pip, inside a cell

import torch

if SMOKE:
    DEVICE = "cpu"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
    print(
        "ℹ️ No GPU in this session, so everything runs on the CPU: fine, but slower. For a GPU, use "
        "Runtime → Change runtime type → T4 GPU, then run this cell again. Every training cell also "
        "offers our precomputed checkpoint."
    )


def seed_everything(seed: int = SEED) -> None:
    """Fix every random source, so re-running a section from its first cell gives the same result (N-013)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

RESULTS: dict = {"use_fallbacks": {}}  # every section writes its named numbers here (SPEC §9)


def _is_stub(fn) -> bool:
    """True when the function body is still the `...` placeholder (optionally after a docstring)."""
    try:
        tree = ast.parse(inspect.getsource(fn).strip() or "pass")
    except (OSError, TypeError, SyntaxError):
        return False
    node = tree.body[0]
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return False
    def const(n, kind):
        is_const = isinstance(n, ast.Expr) and isinstance(n.value, ast.Constant)
        return is_const and isinstance(n.value.value, kind)

    body = [n for n in node.body if not const(n, str)]  # drop the docstring
    return all(const(n, type(...)) for n in body)


def _same(a, b) -> bool:
    """Compare two results of any common type (numbers, arrays, tensors, dicts, sequences)."""
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        a = a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else a
        b = b.detach().cpu().numpy() if isinstance(b, torch.Tensor) else b
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b, strict=True))
    try:
        return bool(np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=1e-5))
    except (TypeError, ValueError):
        return a == b


def use(fn, reference, check=None, name: str | None = None):
    """Return the function the notebook continues with: yours if it matches the reference, else the reference.

    `check(f)` runs `f` on a small play example and returns what to compare (any numbers, arrays or dicts).
    Nothing here is mandatory: an untouched stub (`...` body) silently selects the reference (N-029).
    """
    name = name or fn.__name__
    if _is_stub(fn):
        RESULTS["use_fallbacks"][name] = True
        print(f"ℹ️ {name}: using the reference solution. Fill the stub above to use your own version.")
        return reference
    if check is None:
        RESULTS["use_fallbacks"][name] = False
        print(f"✅ {name}: using your version (no automatic check for this one).")
        return fn
    try:
        yours, theirs = check(fn), check(reference)
    except Exception as err:  # noqa: BLE001 - the participant's code may fail in any way; keep going
        RESULTS["use_fallbacks"][name] = True
        print(f"❌ {name}: your version raised {type(err).__name__}: {err}. Continuing with the reference.")
        return reference
    if _same(yours, theirs):
        RESULTS["use_fallbacks"][name] = False
        print(f"✅ {name}: matches the reference. Continuing with your version.")
        return fn
    RESULTS["use_fallbacks"][name] = True
    print(f"❌ {name}: {yours} vs reference {theirs}. Continuing with the reference.")
    return reference


if not SPECIALISED_JSON.is_file():
    raise SystemExit(f"{SPECIALISED_JSON} is missing. Run setup_check.ipynb first, then run this cell again.")
SPECIALISED: dict = json.loads(SPECIALISED_JSON.read_text())  # the specialised task; never hard-coded
SPECIALISED["dir"] = DATA_DIR / "specialised"
PETS: dict = {"key": "pets", "name": "Oxford-IIIT Pet", "task": "pet vs background"}
PETS |= {"licence": "CC BY-SA 4.0", "dir": DATA_DIR / "pets"}

print(f"Colab: {IN_COLAB} · smoke mode: {SMOKE} · device: {DEVICE} · seed: {SEED}")
print(f"cache: {CACHE_DIR} · specialised task: {SPECIALISED['name']} ({SPECIALISED['task']})")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: how use() works</b><p>Some cells ask you to fill a function whose body is <code>...</code> (the first is <code>my_iou</code> in section 1). The line
after it, <code>iou = use(my_iou, reference=_ref_iou, ...)</code>, runs your version and ours on a small play example:</p>
<ul>
<li>Stub untouched: the notebook continues with ours.</li>
<li>✅ Yours matches: it continues with yours.</li>
<li>❌ Yours differs or crashes: it prints both results and continues with ours.</li>
</ul>
<p>Writing the code is where the learning happens; skipping it never breaks anything.</p></div>"""))

### The theme

This cell sets the colours used in every figure: violet means "it works", orange "it fails, we adapt",
green is the hidden ground truth and teal marks scribbles, the same in every plot.

In [ ]:
# @title Theme and image helpers { display-mode: "form" }
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image

THEME = {
    "works": "#6A1B9A",  # violet: the foundation model does the job
    "fails": "#E65100",  # orange: it falls short / we adapt
    "truth": "#2E7D32",  # green: hidden ground truth, evaluation only
    "scribble": "#00838F",  # teal: strokes a person drew
    "background": "#9E9E9E",  # grey overlay
    "banner": "#4A148C",
}
mpl.rcParams.update(
    {
        "figure.dpi": 100,
        "figure.facecolor": "white",
        "axes.titlesize": 10,
        "axes.titlecolor": "#311B92",
        "axes.prop_cycle": mpl.cycler(color=[THEME[k] for k in ("works", "fails", "truth", "scribble")]),
    }
)


def hex_rgb(colour: str) -> np.ndarray:
    """'#RRGGBB' -> float array in [0, 1]."""
    return np.array([int(colour[i : i + 2], 16) for i in (1, 3, 5)]) / 255


def overlay(image: np.ndarray, mask: np.ndarray, colour: str = THEME["truth"], alpha: float = 0.55):
    """Blend a binary mask in one theme colour over an RGB uint8 image; returns an RGB uint8 image."""
    out = image.astype(float) / 255
    out[mask] = (1 - alpha) * out[mask] + alpha * hex_rgb(colour)
    return (out * 255).astype(np.uint8)


def read_png(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))


def load_item(dataset: dict, item: dict) -> dict:
    """Read one image with its full mask (evaluation only!), scribbles, visible labels and FOV."""
    root = dataset["dir"]
    out = {
        "name": item["name"],
        "split": item["split"],
        "image": np.array(Image.open(root / item["image"]).convert("RGB")),
        "mask": read_png(root / item["mask"]) > 127,
        "scribbles": read_png(root / item["scribbles"]),
        "visible": read_png(root / item["visible"]) > 127,
        "fov": read_png(root / item["fov"]) > 127 if "fov" in item else None,
    }
    out["hidden"] = out["mask"] & ~out["visible"]
    return out


def load_index(dataset: dict) -> dict:
    """Attach the dataset's index.json (facts + one entry per image) and return it."""
    index_file = dataset["dir"] / "index.json"
    if not index_file.is_file():
        raise SystemExit(f"{index_file} is missing. Run setup_check.ipynb first, then run this cell again.")
    dataset["index"] = json.loads(index_file.read_text())
    dataset["items"] = dataset["index"]["items"]
    dataset["fit"] = [it for it in dataset["items"] if it["split"] == "fit"]
    dataset["held"] = [it for it in dataset["items"] if it["split"] == "held"]
    return dataset["index"]

### The data

This cell reads the two datasets (pets, and the specialised task) and shows two images of each with the
ground truth in green and its share of the pixels in the title; the ground truth is for evaluation only.

In [ ]:
# @title Load both datasets and show two examples of each { display-mode: "form" }
for ds in (PETS, SPECIALISED):
    idx = load_index(ds)
    print(
        f"{ds['name']}: {len(ds['items'])} images ({len(ds['fit'])} fit, {len(ds['held'])} held-out) · "
        f"{idx['task']}"
    )
PRECOMPUTED_HEADS = json.loads((PRECOMPUTED / "heads" / "index.json").read_text())["heads"]
print(f"precomputed: {len(PRECOMPUTED_HEADS)} head checkpoints in {PRECOMPUTED}")

fig, axes = plt.subplots(2, 4, figsize=(12, 6.4))
for k, ds in enumerate((PETS, SPECIALISED)):
    for j, item in enumerate(ds["fit"][:2]):
        it = load_item(ds, item)
        axes[k, 2 * j].imshow(it["image"])
        axes[k, 2 * j].set_title(f"{ds['name']}: {it['name']}")
        axes[k, 2 * j + 1].imshow(overlay(it["image"], it["mask"]))
        axes[k, 2 * j + 1].set_title(f"ground truth · {it['mask'].mean():.0%} of the pixels")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>Look at the images above. Which task do you expect the foundation model to solve out of the box, and which
one will give it trouble? Write one reason; section 2 will tell you whether you were right.</p>
<p><strong>Q1.</strong> 🟢 What is different about the specialised task that could matter for a model trained on everyday
photos? Look at the size, shape and contrast of the objects, not the domain name.</p>
<details>
<summary>One possible answer</summary>
<p>Pets are big and clear; the specialised targets are thin, faint and small.</p>
</details></div>"""))

### The models

This cell defines two wrappers: `dino.features(image, long_side)` gives a grid of image features and
`sam.probability(image, points, labels)` gives a foreground map from point prompts.

In [ ]:
# @title Model loaders { display-mode: "form" }
import timm
import torch.nn.functional as F

DINO_ID = "vit_small_patch16_dinov3.lvd1689m"
DINO_SOURCE = "timm/vit_small_patch16_dinov3.lvd1689m"
DINO_LICENCE = "DINOv3 License (Meta)"
SAM_ID = "facebook/sam2.1-hiera-small"  # SPEC §5: SAM 2.1 hiera-small, Apache-2.0, no account needed
SAM_LICENCE = "Apache-2.0"


class Dino:
    """Frozen DINOv3 ViT-S/16: an image in, a grid of 384-dim patch features out."""

    def __init__(self, device: str = DEVICE):
        t = time.monotonic()
        self.device = device
        folder = model_folder(DINO_SOURCE)  # plain files on Drive, else the hub id
        extra = {"pretrained_cfg_overlay": {"file": str(weights_file(folder))}} if folder else {}
        model = timm.create_model(DINO_ID, pretrained=True, num_classes=0, dynamic_img_size=True, **extra)
        self.model = model.eval().to(device)
        cfg = timm.data.resolve_data_config({}, model=self.model)
        self.mean = torch.tensor(cfg["mean"]).view(1, 3, 1, 1).to(device)
        self.std = torch.tensor(cfg["std"]).view(1, 3, 1, 1).to(device)
        self.n_prefix = self.model.num_prefix_tokens
        self.dim = self.model.num_features
        dt = time.monotonic() - t
        print(f"DINOv3 ViT-S/16 · {DINO_SOURCE} · {DINO_LICENCE} · {dt:.0f} s")

    @torch.no_grad()
    def features(self, image: np.ndarray, long_side: int) -> torch.Tensor:
        """Patch features [D, h, w] for the image resized so its long side is `long_side` (multiple of 16)."""
        x = torch.from_numpy(np.ascontiguousarray(image)).permute(2, 0, 1).float().unsqueeze(0) / 255
        h, w = image.shape[:2]
        s = long_side / max(h, w)
        H, W = max(16, round(h * s / 16) * 16), max(16, round(w * s / 16) * 16)
        x = F.interpolate(x.to(self.device), size=(H, W), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        f = self.model.forward_features(x)[0, self.n_prefix :]
        return f.reshape(H // 16, W // 16, -1).permute(2, 0, 1).contiguous().float()


class Sam:
    """SAM 2.1 (promptable segmentation): an image plus points or a box in, a foreground probability out."""

    def __init__(self, device: str = DEVICE):
        from transformers import Sam2Model, Sam2Processor

        t = time.monotonic()
        self.device = device
        src = str(model_folder(SAM_ID) or SAM_ID)  # plain files on Drive, else the hub id
        self.proc = Sam2Processor.from_pretrained(src)
        self.model = Sam2Model.from_pretrained(src).eval().to(device)
        self._cache: tuple[np.ndarray, torch.Tensor] | None = None  # (the last image, its embeddings)
        print(f"SAM 2.1 hiera-small · {SAM_ID} · {SAM_LICENCE} · {time.monotonic() - t:.0f} s")

    @torch.no_grad()
    def probability(
        self,
        image: np.ndarray,
        points: list[list[float]] | None = None,
        labels: list[int] | None = None,
        box: list[float] | None = None,
    ) -> np.ndarray:
        """Foreground probability (H x W float) for points (x, y; label 1 = fg, 0 = bg) and/or a box."""
        kw = {}
        if points:
            kw["input_points"], kw["input_labels"] = [[points]], [[labels]]
        if box is not None:
            kw["input_boxes"] = [[box]]
        inputs = self.proc(images=image, return_tensors="pt", **kw)
        if self._cache is None or self._cache[0] is not image:
            self._cache = (image, self.model.get_image_embeddings(inputs["pixel_values"].to(self.device)))
        out = self.model(
            image_embeddings=self._cache[1],
            input_points=inputs["input_points"].to(self.device) if points else None,
            input_labels=inputs["input_labels"].to(self.device) if points else None,
            input_boxes=inputs["input_boxes"].to(self.device) if box is not None else None,
            multimask_output=False,
        )
        masks = out.pred_masks.cpu()
        logits = self.proc.post_process_masks(masks, inputs["original_sizes"], binarize=False)[0]
        return torch.sigmoid(logits[0, 0]).numpy()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: the first slow step</b><p>Loading and the first call of a model are the slow steps: PyTorch compiles kernels the first time a model
sees a new image size. On a T4 this takes seconds; on a CPU it is much slower. Later calls are faster.</p></div>"""))

This cell loads DINOv3 and SAM from the cache and prints one line each: name · source · licence · load time.

In [ ]:
# @title Load DINOv3 and SAM { display-mode: "form" }
dino = Dino()
sam = Sam()
RESULTS["models"] = {"dino": DINO_SOURCE, "sam": SAM_ID, "device": DEVICE}

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFF8E1;border-left:6px solid #5D4037;border-radius:6px;padding:12px 14px;margin:6px 0;color:#5D4037;font-family:inherit;line-height:1.5"><b>🧾 If something goes wrong</b><ul>
<li>"Google Drive is not mounted" or "... is missing" → run the setup cell again and click Allow; if the
  folder is still missing, run <code>setup_check.ipynb</code> first.</li>
<li>"No GPU in this session" → Runtime → Change runtime type → T4 GPU, then run the setup cell again.</li>
<li>A cell is slow on a CPU → normal; training cells have <code>TRAIN_YOURSELF = False</code> to load our checkpoint.</li>
<li>A cell with a <code>...</code> stub → fill it or skip it; <code>use()</code> picks yours or ours and the notebook goes on.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Recipe and sources</b></summary><ul>
<li>Feature extractor: <strong>DINOv3</strong> ViT-S/16 (Meta, 2025, via <code>timm</code>).</li>
<li>Promptable segmenter: <strong>SAM 2.1</strong> hiera-small (Meta).</li>
<li>Scribble idea: train a tiny classifier on frozen DINO patch features, using only the pixels a person
  marked.</li>
</ul></details></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li>When a cell needs data → <code>PETS</code> and <code>SPECIALISED</code> are loaded from the cache, with ground truth for
  evaluation only.</li>
<li>When a cell needs a model → <code>dino</code> and <code>sam</code> are already loaded, nothing is downloaded.</li>
<li>When you skip or fail a <code>...</code> stub → <code>use()</code> continues with our version, nothing breaks.</li>
<li>When you read a figure → violet works, orange falls short, green is ground truth, teal is a scribble.</li>
</ul></div>"""))

## 1. Out of the box: pets

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>See a foundation model segment pets almost perfectly, zero-shot (with no training on this task).</p></div>"""))

Here SAM, DINOv3 and a linear head (explained below the first figure) meet the easy task; keep your
numbers, because section 2 runs the same three cells on the specialised task.

This helper cell defines how to turn a person's hint into a SAM prompt, and `show_masks()`, the figure used
for every comparison in the notebook (prediction in violet, ground truth in green, prompts in teal).

In [ ]:
# @title Prompts and figure helpers { display-mode: "form" }
from safetensors import safe_open
from safetensors.torch import load_file

_FEATURE_FILES: dict = {}


def click_point(item: dict) -> list[float]:
    """One (x, y) where a person would click: a pixel of the first foreground scribble stroke."""
    ys, xs = np.nonzero(item["scribbles"] == 1)
    rng = np.random.default_rng([SEED, len(ys)])
    k = int(rng.integers(len(ys)))
    return [float(xs[k]), float(ys[k])]


def box_prompt(item: dict) -> list[float]:
    """The box a person would draw around the object: the full mask's bounding box, as x0, y0, x1, y1."""
    ys, xs = np.nonzero(item["mask"])
    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]


def show_masks(panels: list[tuple[str, np.ndarray]], prompts: dict | None = None, width: float = 3.2):
    """One row of RGB panels (title, image). `prompts` = {"point": [x, y], "box": [x0, y0, x1, y1]}."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width))
    axes = np.atleast_1d(axes)
    for ax, (title, img) in zip(axes, panels, strict=True):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    if prompts:
        for ax in axes:
            if prompts.get("point") is not None:
                x, y = prompts["point"]
                ax.plot(x, y, marker="*", markersize=14, color=THEME["scribble"], markeredgecolor="white")
            if prompts.get("box") is not None:
                x0, y0, x1, y1 = prompts["box"]
                rect = mpl.patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=2)
                rect.set_edgecolor(THEME["scribble"])
                ax.add_patch(rect)
    plt.tight_layout()
    plt.show()


def features_for(dataset: dict, item: dict, long_side: int) -> torch.Tensor:
    """DINOv3 patch features [384, h, w]: from the precomputed file when it exists, else computed now."""
    prefix = "pets" if dataset is PETS else "specialised"
    path = PRECOMPUTED / "features" / f"{prefix}_{long_side}.safetensors"
    if path.is_file():
        if path not in _FEATURE_FILES:
            _FEATURE_FILES[path] = safe_open(path, framework="pt")
        store = _FEATURE_FILES[path]
        if item["name"] in store.keys():
            return store.get_tensor(item["name"]).to(DEVICE).float()
    return dino.features(item["image"], long_side)


def pca_rgb(feat: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Project the 384-dim patch features onto their 3 main directions and show them as red, green, blue."""
    d, h, w = feat.shape
    x = feat.reshape(d, h * w).T.cpu().numpy().astype(np.float64)
    x = x - x.mean(0)
    _, _, vt = np.linalg.svd(x, full_matrices=False)
    rgb = x @ vt[:3].T
    rgb = (rgb - rgb.min(0)) / (rgb.max(0) - rgb.min(0) + 1e-9)
    img = torch.from_numpy(rgb.reshape(h, w, 3)).permute(2, 0, 1)[None].float()
    img = F.interpolate(img, size=shape, mode="bilinear", align_corners=False)[0].permute(1, 2, 0)
    return (img.numpy() * 255).astype(np.uint8)

This cell loads the held-out pets, the evaluation set every approach in this notebook is scored on (12 of
them on a CPU, to keep cells short), and fixes the resolution DINOv3 works at.

In [ ]:
# @title Load the held-out pets { display-mode: "form" }
PETS["feat_side"] = 256 if SMOKE else PETS["index"]["feature_long_side"]
PETS["eval"] = PETS["held"] if (SMOKE or DEVICE == "cuda") else PETS["held"][:12]
PETS_EVAL = [load_item(PETS, it) for it in PETS["eval"]]
print(f"evaluation set: {len(PETS_EVAL)} held-out pets · DINOv3 features at {PETS['feat_side']} px long side")

### 1.1 SAM: one click, one box


In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The next cell gives SAM a single click on a pet, then a box around it. Before you run it: will one
click be enough for a clean mask of the whole animal, and what could go wrong (fur vs floor, the tail,
a second animal)? Say it to your neighbour or write it down.</p>
<p>Then run the next cell and look at the three panels.</p>
<p><strong>Q2.</strong> 🟡 Compare the one-click mask and the box mask with the ground truth.</p>
<ul>
<li>Where do they differ, if at all?</li>
<li>Is the difference at the object's boundary, or is a whole part missing or added?</li>
<li>Try another pet: change the index in <code>demo = PETS_EVAL[0]</code> (try 1, 2, ...).</li>
</ul></div>"""))

This cell runs SAM twice on one held-out pet, once with a single click and once with a box, and shows
both masks next to the ground truth, so you can see what a promptable model gives for free.

In [ ]:
# Q2 — SAM with one click and with one box on a held-out pet
demo = PETS_EVAL[0]  # try 1, 2, ...
point, box = click_point(demo), box_prompt(demo)
t0 = time.monotonic()
sam_point = sam.probability(demo["image"], points=[point], labels=[1]) >= 0.5
sam_box = sam.probability(demo["image"], box=box) >= 0.5
print(f"{demo['name']}: two SAM calls in {time.monotonic() - t0:.1f} s on {DEVICE}")
show_masks(
    [
        ("SAM, one click", overlay(demo["image"], sam_point, THEME["works"])),
        ("SAM, one box", overlay(demo["image"], sam_box, THEME["works"])),
        ("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])),
    ],
    prompts={"point": point, "box": box},
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: the three tools</b><ul>
<li><strong>SAM</strong> (Segment Anything) is <em>promptable</em>. You give it an image and a hint (a click, a box on a pet);
  it returns the mask of the object under the hint. Nothing is trained: it learnt "what an object is" from
  a billion masks of everyday photos.</li>
<li><strong>DINOv3</strong> is a <em>feature extractor</em>. It cuts the image into 16 × 16 px patches and returns 384 numbers
  per patch, learnt without labels. Patches with similar numbers look alike to the model (two cat ears,
  say).</li>
<li>A <strong>linear head</strong> is the tiny classifier that turns those features into a mask: the features segment
  nothing on their own, and the head learns a new class from a handful of masks in seconds (section 1.4).</li>
</ul></div>"""))

### 1.2 One number: IoU

Looking at masks is the honest way to judge a segmentation; one number lets us compare approaches on a
hundred images. The whole workshop uses one metric, **IoU** (intersection over union):

`IoU = |pred ∩ truth| / |pred ∪ truth|`

- Example: the predicted mask and the true mask each cover 100 pixels and share 80. The union is 120
  pixels, so IoU = 80 / 120 = 0.67.
- 1.0 is a perfect match, 0.0 means no overlap. Section 2 adds one extra metric for thin structures.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 2 min)</b><p>Fill <code>my_iou</code>: two boolean arrays in, one float out.</p>
<ul>
<li>Intersection: pixels where both are True.</li>
<li>Union: pixels where at least one is True.</li>
<li>Empty union: return 1.0 (two empty masks agree).</li>
</ul>
<p>The cell below checks your version on this 4 × 4 play example (1 = True) and continues with ours if you
skip it:</p>
<pre><code>pred   truth
0000   0110
0111   0110
0111   0110
0000   0000
</code></pre>
<p>Intersection 4, union 8, so the answer is 0.5.</p></div>"""))

In [ ]:
# @title Reference solution (open only if stuck) { display-mode: "form" }
def _ref_iou(pred: np.ndarray, truth: np.ndarray) -> float:
    """Reference: |pred ∧ truth| / |pred ∨ truth|."""
    inter, union = float((pred & truth).sum()), float((pred | truth).sum())
    return inter / union if union else 1.0


_play_pred = np.zeros((4, 4), bool)
_play_pred[1:3, 1:4] = True  # 6 pixels
_play_truth = np.zeros((4, 4), bool)
_play_truth[0:3, 1:3] = True  # 6 pixels, 4 shared -> IoU = 4 / 8

In [ ]:
# ✍️ YOUR CODE
def my_iou(pred: np.ndarray, truth: np.ndarray) -> float:
    """Intersection over union of two boolean masks of the same shape; 1.0 when both are empty.
    e.g. 0.62"""
    ...


iou = use(my_iou, reference=_ref_iou, check=lambda f: round(float(f(_play_pred, _play_truth)), 4), name="iou")

This cell scores SAM's one-click and one-box masks on every evaluation pet with the IoU you just
defined and keeps the mean as this task's zero-shot number, for section 2 to compare.

In [ ]:
# @title Score SAM on every held-out pet { display-mode: "form" }
t0 = time.monotonic()
zero_shot_all = {"point": [], "box": []}
for it in PETS_EVAL:
    pt = click_point(it)
    by_point = sam.probability(it["image"], points=[pt], labels=[1]) >= 0.5
    by_box = sam.probability(it["image"], box=box_prompt(it)) >= 0.5
    zero_shot_all["point"].append(iou(by_point, it["mask"]))
    zero_shot_all["box"].append(iou(by_box, it["mask"]))
zero_shot = {k: float(np.mean(v)) for k, v in zero_shot_all.items()}
RESULTS["pets_zero_shot_iou"] = zero_shot["box"]  # the headline: SAM's best single prompt, one box
RESULTS["pets_zero_shot_point_iou"] = zero_shot["point"]
RESULTS["pets_zero_shot_box_iou"] = zero_shot["box"]
RESULTS["pets_zero_shot_seconds"] = round(time.monotonic() - t0, 1)
print(
    f"SAM zero-shot on {len(PETS_EVAL)} held-out pets · mean IoU one click {zero_shot['point']:.3f} · "
    f"one box {zero_shot['box']:.3f} · {RESULTS['pets_zero_shot_seconds']} s"
)

A mean hides the spread, so this cell shows the **worst, median and best** held-out pet by one-box IoU:
violet where mask and truth agree, orange for a false alarm (extra mask), green for what SAM missed.

In [ ]:
# @title Worst, median and best pet by IoU { display-mode: "form" }
order = np.argsort(zero_shot_all["box"])
panels = []
for label, pos in (("worst", order[0]), ("median", order[len(order) // 2]), ("best", order[-1])):
    it = PETS_EVAL[int(pos)]
    pred = sam.probability(it["image"], box=box_prompt(it)) >= 0.5
    img = overlay(it["image"], pred & it["mask"], THEME["works"])
    img = overlay(img, pred & ~it["mask"], THEME["fails"])
    img = overlay(img, ~pred & it["mask"], THEME["truth"])
    panels.append((f"{label} · IoU {zero_shot_all['box'][int(pos)]:.2f}", img))
show_masks(panels)

### 1.3 What DINOv3 sees

This cell keeps the **3 main directions (a PCA)** of each patch's 384 numbers and shows them as red, green
and blue, so the same colour means the model sees those patches as similar (both cat ears, say).

In [ ]:
# @title Show the patch features as colours { display-mode: "form" }
fig_items = [load_item(PETS, it) for it in PETS["fit"][:3]]
panels = []
for it in fig_items:
    feat = features_for(PETS, it, PETS["feat_side"])
    title = f"DINOv3 PCA · {feat.shape[2]}×{feat.shape[1]} patches"
    panels += [(it["name"], it["image"]), (title, pca_rgb(feat, it["image"].shape[:2]))]
show_masks(panels, width=2.6)

**Q3.** 🟡 Look at the PCA images above.

- Which pixels share a colour: the whole animal, parts of it (head, fur, legs), or the floor too?
- Does the colour follow the object's outline?
- How fine is the grid behind it?

<details><summary>What to look for</summary>

The colours are not classes: they are the 3 directions along which the features vary most in this image.
Typically the animal separates from the background in one direction and its parts (face vs body) in
another. The blocky edges show the 16 px grid: the features know <i>what</i> is where, at patch resolution,
not <i>exactly where</i> the boundary is. Whether that is enough depends on the size of what you want to
segment.

</details>

### 1.4 A linear head on five pets

**How the head works.** Each patch gets one score from its 384 feature numbers `f`:

`score = w · f + b` (384 weights + 1 bias = 385 parameters), the same `w` and `b` for every patch (a
"1 × 1 convolution"). Score above 0: the patch is "pet"; below 0: "background".

A sigmoid `σ` squeezes the score into a probability `p = σ(score)` between 0 and 1 (score 2.2 gives
p = 0.90, score 0 gives 0.5), so "score above 0" and "p above 0.5" say the same thing. The per-patch map is
then upsampled to the image.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The next cells train a classifier with 385 parameters on the patch features of just <strong>five</strong> pets with
full masks, then score it on the held-out ones. Will its IoU land above, near or below SAM's one-click
number from 1.2? Write down a number before you run.</p></div>"""))

This helper cell defines the head used for every training step in the notebook (that 1 × 1 convolution on
standardised patch features, with a balanced loss, where each class gets half the total weight, that
ignores pixels labelled −1) plus `load_head()` for our precomputed checkpoints.

In [ ]:
# @title Linear head: train, predict, load a checkpoint { display-mode: "form" }
def train_head(
    feats: list[torch.Tensor], labels: list[np.ndarray], steps: int, lr: float = 0.05, balanced: bool = True
) -> tuple[dict, float]:
    """Linear head (384 -> 1) on standardised patch features. Labels per pixel: 1 fg, 0 bg, -1 ignored.

    Returns the head's tensors (weight, bias, feat_mean, feat_std) and the training seconds.
    """
    torch.manual_seed(SEED)
    allf = torch.cat([f.flatten(1) for f in feats], 1)
    mu, sd = allf.mean(1), allf.std(1) + 1e-6
    feats = [(f - mu.view(-1, 1, 1)) / sd.view(-1, 1, 1) for f in feats]
    labs = [torch.from_numpy(np.asarray(lab, dtype=np.int8)).to(DEVICE) for lab in labels]
    n_pos = sum(int((lab == 1).sum()) for lab in labs)
    n_neg = sum(int((lab == 0).sum()) for lab in labs)
    if balanced:
        w_pos, w_neg = 0.5 / max(n_pos, 1), 0.5 / max(n_neg, 1)
    else:
        w_pos = w_neg = 1.0 / max(n_pos + n_neg, 1)
    head = torch.nn.Conv2d(feats[0].shape[0], 1, 1).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    t0 = time.monotonic()
    for _ in range(steps):
        opt.zero_grad()
        loss = 0.0
        for f, lab in zip(feats, labs, strict=True):
            logit = F.interpolate(head(f[None]), size=lab.shape, mode="bilinear", align_corners=False)[0, 0]
            w = torch.where(lab == 1, w_pos, torch.where(lab == 0, w_neg, 0.0))
            bce = F.binary_cross_entropy_with_logits(logit, (lab == 1).float(), reduction="none")
            loss = loss + (bce * w).sum()
        loss.backward()
        opt.step()
    tensors = {
        "weight": head.weight.detach().cpu(),
        "bias": head.bias.detach().cpu(),
        "feat_mean": mu.cpu(),
        "feat_std": sd.cpu(),
    }
    return tensors, time.monotonic() - t0


@torch.no_grad()
def predict_head(tensors: dict, feat: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Apply a head to one feature grid; returns the boolean foreground mask at `shape` (logit > 0)."""
    dev = feat.device
    f = (feat - tensors["feat_mean"].to(dev).view(-1, 1, 1)) / tensors["feat_std"].to(dev).view(-1, 1, 1)
    logit = F.conv2d(f[None], tensors["weight"].to(dev), tensors["bias"].to(dev))
    logit = F.interpolate(logit, size=shape, mode="bilinear", align_corners=False)[0, 0]
    return (logit > 0).cpu().numpy()


def load_head(dataset: dict, regime: str, long_side: int | None = None) -> tuple[dict, dict]:
    """Our precomputed checkpoint for (dataset, regime): its tensors and its index entry."""
    long_side = long_side or dataset["feat_side"]
    key = "pets" if dataset is PETS else SPECIALISED["key"]
    wanted = (key, regime, long_side)
    heads = PRECOMPUTED_HEADS
    entry = next((h for h in heads if (h["dataset"], h["regime"], h["long_side"]) == wanted), None)
    if entry is None:
        raise SystemExit(f"no precomputed head for {wanted}. Run setup_check.ipynb first.")
    return load_file(PRECOMPUTED / entry["file"]), entry

This cell trains the head on the full masks of five fit pets (only sections 1 and 2 use full masks); set
`TRAIN_YOURSELF = False` and run it again to load our checkpoint instead.

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed head "full5" instead of training
N_TRAIN_PETS, HEAD_STEPS = 5, 30 if SMOKE else 200
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

train_items = [load_item(PETS, it) for it in PETS["fit"][:N_TRAIN_PETS]]
train_feats = [features_for(PETS, it, PETS["feat_side"]) for it in train_items]
if TRAIN_YOURSELF:
    train_labels = [it["mask"].astype(np.int8) for it in train_items]  # full masks: allowed in section 1 only
    head_full5, head_sec = train_head(train_feats, train_labels, HEAD_STEPS)
    head_src = f"trained here on {len(train_items)} pets, {HEAD_STEPS} steps"
else:
    head_full5, entry = load_head(PETS, "full5")
    head_sec = entry["train_sec"]
    head_src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} pets)"
print(f"linear head · {head_src} · {head_sec:.1f} s on {DEVICE}")
if head_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

This cell runs the head you just trained on one training pet and shows each step (patches, one probability
per patch, the upsampled map, the final mask), then works through one patch by hand.

In [ ]:
# @title The linear head step by step on one pet { display-mode: "form" }
hx = train_items[0]
hfeat = features_for(PETS, hx, PETS["feat_side"])  # [384, h, w]: 384 numbers per patch
dev = hfeat.device
with torch.no_grad():
    hp = {k: v.to(dev) for k, v in head_full5.items()}
    hz = (hfeat - hp["feat_mean"].view(-1, 1, 1)) / hp["feat_std"].view(-1, 1, 1)
    hscore = F.conv2d(hz[None], hp["weight"], hp["bias"])[0, 0]  # w·f + b, one score per patch
    hprob = torch.sigmoid(hscore)
    hup = F.interpolate(hprob[None, None], size=hx["mask"].shape, mode="bilinear", align_corners=False)[0, 0]
gh, gw = hprob.shape
py, px = np.unravel_index(int(hprob.argmax()), hprob.shape)  # the most pet-like patch
f_vec, w_vec = hz[:, py, px], hp["weight"].reshape(-1)
f_head = ", ".join(f"{v:.2f}" for v in f_vec[:4].tolist())
dot, bias = float((w_vec * f_vec).sum()), float(hp["bias"])
total, p_patch = float(hscore[py, px]), float(hprob[py, px])
print(f"patch (row {py}, col {px}) · f = [{f_head}, … 384 numbers]")
print(f"w · f + b = {dot:.2f} + ({bias:.2f}) = {total:.2f} → p = σ({total:.2f}) = {p_patch:.2f}")

prob_cmap = mpl.colors.LinearSegmentedColormap.from_list("prob", ["#FFFFFF", THEME["works"]])
H, W = hx["mask"].shape
fig, axes = plt.subplots(1, 4, figsize=(11.6, 3))
axes[0].imshow(hx["image"])
axes[0].vlines(np.arange(0, W + 1, W / gw), 0, H, colors="white", linewidths=0.4)
axes[0].hlines(np.arange(0, H + 1, H / gh), 0, W, colors="white", linewidths=0.4)
cell_box = plt.Rectangle((px * W / gw, py * H / gh), W / gw, H / gh, fill=False, ec=THEME["scribble"], lw=2)
axes[0].add_patch(cell_box)
axes[0].set_title("1 · image cut into patches", fontsize=9)
axes[1].imshow(hprob.cpu().numpy(), cmap=prob_cmap, vmin=0, vmax=1, interpolation="nearest")
axes[1].add_patch(plt.Rectangle((px - 0.5, py - 0.5), 1, 1, fill=False, ec=THEME["scribble"], lw=2))
axes[1].set_title("2 · one p per patch (white 0, violet 1)", fontsize=9)
axes[2].imshow(hup.cpu().numpy(), cmap=prob_cmap, vmin=0, vmax=1)
axes[2].set_title("3 · upsampled to the image", fontsize=9)
axes[3].imshow(overlay(hx["image"], predict_head(head_full5, hfeat, hx["mask"].shape), THEME["works"]))
axes[3].set_title("4 · p > 0.5 → pet", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

This cell scores the head on the same evaluation pets as SAM, keeps the mean IoU, and shows three held-out
pets with the head's mask next to the ground truth.

In [ ]:
# @title Score the head on the held-out pets { display-mode: "form" }
t0 = time.monotonic()
head_preds = [
    predict_head(head_full5, features_for(PETS, it, PETS["feat_side"]), it["mask"].shape) for it in PETS_EVAL
]
head_ious = [iou(pred, it["mask"]) for pred, it in zip(head_preds, PETS_EVAL, strict=True)]
RESULTS["pets_head_iou"] = float(np.mean(head_ious))
RESULTS["pets_head"] = {
    "iou": RESULTS["pets_head_iou"],
    "n_train": len(train_items),
    "train_sec": round(float(head_sec), 2),
    "infer_sec": round(time.monotonic() - t0, 1),
}
print(
    f"linear head on {len(PETS_EVAL)} held-out pets · mean IoU {RESULTS['pets_head_iou']:.3f} "
    f"(SAM one click: {RESULTS['pets_zero_shot_point_iou']:.3f}, "
    f"one box: {RESULTS['pets_zero_shot_box_iou']:.3f})"
)
panels = []
for it, pred, v in list(zip(PETS_EVAL, head_preds, head_ious, strict=True))[:3]:
    panels += [
        (f"head · IoU {v:.2f}", overlay(it["image"], pred, THEME["works"])),
        ("ground truth", overlay(it["image"], it["mask"], THEME["truth"])),
    ]
show_masks(panels, width=2.6)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8EAF6;border-left:6px solid #1A237E;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1A237E;font-family:inherit;line-height:1.5"><b>🔀 Across approaches</b><p>Two ways to the same mask, scored on the same hidden ground truth:</p>
<ul>
<li><strong>SAM</strong> needs <em>one hint per image</em> and no training.</li>
<li><strong>The head</strong> needs <em>five full masks once</em>, then runs on every image with no hint.</li>
</ul>
<p>Keep both numbers: the comparison table in section 8 starts with them, and section 2 reruns both on the
specialised task.</p></div>"""))

**Q4.** 🟡 Compare the two numbers you got with the prediction you wrote down.

- Which approach would you pick for 10 000 pet photos, and which for 10 photos?
- What does each one ask of a person?

<details><summary>One possible answer</summary>

For 10 photos, a click each is nothing, and SAM's masks come with sharp boundaries. For 10 000, a person
cannot click on each one, while the head, once trained on a few masks, runs with no hint at all. Its
boundaries follow the 16 px patch grid, so it loses a little IoU at the edges, which matters more as the
objects get smaller. Hardware or seed may change the numbers; the trade-off stays.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>👥 Optional, 3 min</b><p>Pick different indices in <code>demo = PETS_EVAL[0]</code> (Q2), so you and your neighbour see different pictures, and
compare.</p>
<ul>
<li>Did one click fail on either of yours?</li>
<li>If so, was the click on an ambiguous spot (fur against a similar floor, a collar, a second animal)?</li>
</ul></div>"""))

This cell clicks on the floor instead of the pet and shows what SAM returns.

In [ ]:
# @title Click on the floor, not the pet { display-mode: "form" }
it = PETS_EVAL[0]
floor_ys, floor_xs = np.nonzero(~it["mask"])
k = int(np.random.default_rng([SEED, len(floor_ys)]).integers(len(floor_ys)))
floor_pt = [float(floor_xs[k]), float(floor_ys[k])]
by_floor = sam.probability(it["image"], points=[floor_pt], labels=[1]) >= 0.5
floor_title = f"SAM, click on the floor · IoU {iou(by_floor, it['mask']):.2f}"
show_masks(
    [
        (floor_title, overlay(it["image"], by_floor, THEME["works"])),
        ("ground truth", overlay(it["image"], it["mask"], THEME["truth"])),
    ],
    prompts={"point": floor_pt},
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: a clean mask is not a correct one</b><p>SAM returned a confident-looking mask for a click on the floor, so always compare a mask with the ground
truth (section 2 shows why).</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li>When you have a few photos and can click each one → SAM with a point or a box gives sharp masks, with no
  training.</li>
<li>When you have thousands of photos and no one to click → train the linear head on a few full masks.</li>
<li>When a mask looks clean → still check it against the ground truth before trusting it.</li>
<li>When the head's edges look blocky (16 px grid) → expect lower IoU on small or thin objects; section 2
  tests it.</li>
</ul></div>"""))

## 2. Where it falls short

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>Run the same tools on the specialised task and probe SAM with escalating prompts.</p></div>"""))

Same three tools, same cells, new task. Section 1 gave you three numbers on pets: SAM with one click,
SAM with one box, and a linear head on five full masks.

Here they meet retina images (a *fundus* is a photo of the back of the eye), where the objects are thin,
branching and low-contrast. Then we push SAM with more and more prompts, and give DINOv3 twice the
resolution, to see whether the limit is the prompt or the patch size.

This helper cell defines what this section measures with: the evaluation region (the fundus, not the
black border), skeletons for the thin-structure metric, clicks spread along a person's scribbles, and a
figure that marks every click.

In [ ]:
# @title Thin-structure helpers: region, skeleton, clicks, figure { display-mode: "form" }
from skimage.morphology import skeletonize


def region(item: dict) -> np.ndarray:
    """Where we score: the field of view if the dataset has one, else the whole image."""
    return item["fov"] if item.get("fov") is not None else np.ones_like(item["mask"])


def skeleton(mask: np.ndarray) -> np.ndarray:
    """The one-pixel-wide centre line of a boolean mask."""
    return skeletonize(np.asarray(mask, bool))


def scribble_points(item: dict, n: int) -> list[list[float]]:
    """n (x, y) clicks along the foreground scribbles: the section-1 click first, then spread out as far as
    possible from each other (farthest-point sampling), the way a person would click along the vessels."""
    ys, xs = np.nonzero(item["scribbles"] == 1)
    pts = np.stack([xs, ys], 1).astype(float)
    chosen = [np.array(click_point(item))]
    dist = np.linalg.norm(pts - chosen[0], axis=1)
    while len(chosen) < min(n, len(pts)):
        k = int(dist.argmax())
        chosen.append(pts[k])
        dist = np.minimum(dist, np.linalg.norm(pts - pts[k], axis=1))
    return [[float(x), float(y)] for x, y in chosen]


def score(pred: np.ndarray, item: dict) -> dict:
    """IoU and clDice of a boolean prediction, both inside the evaluation region."""
    r = region(item)
    return {"iou": iou(pred & r, item["mask"] & r), "cldice": cldice(pred & r, item["mask"] & r)}


def mean_scores(scores: list[dict]) -> dict:
    return {k: float(np.mean([s[k] for s in scores])) for k in scores[0]}


def show_clicks(panels: list[tuple[str, np.ndarray, list | None]], width: float = 2.6):
    """One row of panels (title, RGB image, clicks or None[, box]); stars = clicks, frame = box."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width))
    for ax, (title, img, pts, *extra) in zip(np.atleast_1d(axes), panels, strict=True):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
        if extra and extra[0] is not None:  # an optional box (x0, y0, x1, y1)
            bx0, by0, bx1, by1 = extra[0]
            ax.add_patch(plt.Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False, lw=2,
                                       ec=THEME["scribble"]))
        for x, y in pts or []:
            ax.plot(x, y, marker="*", markersize=10, color=THEME["scribble"], markeredgecolor="white")
    plt.tight_layout()
    plt.show()

This cell loads the held-out retina images (1024 px crops of a 3504 × 2336 fundus), the evaluation set for
the rest of the notebook, and prints how much of the fundus the vessels cover; keep that number, it comes
back as a baseline.

In [ ]:
# @title Load the held-out retina images { display-mode: "form" }
SPECIALISED["feat_side"] = 256 if SMOKE else SPECIALISED["feature_long_side"]
SPECIALISED["eval"] = SPECIALISED["held"] if (SMOKE or DEVICE == "cuda") else SPECIALISED["held"][:12]
SPEC_EVAL = [load_item(SPECIALISED, it) for it in SPECIALISED["eval"]]
fg_share = float(np.mean([it["mask"][region(it)].mean() for it in SPEC_EVAL]))
print(
    f"evaluation set: {len(SPEC_EVAL)} held-out images · {SPECIALISED['task']} · the foreground covers "
    f"{fg_share:.0%} of the evaluation region · DINOv3 features at {SPECIALISED['feat_side']} px long side"
)

Before any theory, look at the images: this cell shows one retina photo, its vessels, and a zoom with the
16 px grid DINOv3 sees, and measures how wide a vessel is next to one patch.

In [ ]:
# @title Meet the retina images: vessels next to the DINOv3 patch grid { display-mode: "form" }
peek = SPEC_EVAL[0]
ph, pw = peek["mask"].shape
patch_px = 16 * max(ph, pw) / SPECIALISED["feat_side"]  # one DINOv3 patch, in image pixels
centre_line = skeleton(peek["mask"] & region(peek))
vessel_px = float(peek["mask"].sum() / max(centre_line.sum(), 1))  # area ÷ centre-line length = mean width
win = int(min(8 * patch_px, ph, pw))  # the zoom window: about 8 × 8 patches
y0, x0 = (ph - win) // 2, (pw - win) // 2
zoom = (slice(y0, y0 + win), slice(x0, x0 + win))
truth_img = overlay(peek["image"], peek["mask"], THEME["truth"])

fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.2))
axes[0].imshow(peek["image"])
axes[0].set_title("fundus photo: bright round patch = optic disc,\nblack ring = outside the view", fontsize=8)
axes[1].imshow(truth_img)
axes[1].set_title("vessels (ground truth, green)", fontsize=9)
axes[2].imshow(truth_img[zoom])
grid = np.arange(0, win + 1, patch_px)
axes[2].vlines(grid, 0, win, colors="white", linewidths=0.8)
axes[2].hlines(grid, 0, win, colors="white", linewidths=0.8)
axes[2].set_xlim(0, win)
axes[2].set_ylim(win, 0)
axes[2].set_title(f"zoom: each white square is one {patch_px:.0f} × {patch_px:.0f} px patch", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
print(
    f"{peek['name']}: vessels are about {vessel_px:.1f} px wide on average, one patch is {patch_px:.0f} px, "
    f"so a typical vessel fills about {vessel_px / patch_px:.0%} of a patch's width"
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: why thin structures are hard</b><p>Each tool has its own trouble with the vessels you just saw.</p>
<ul>
<li><strong>DINOv3 looks at 16 × 16 px patches.</strong> A vessel a few pixels wide fills a fraction of a patch (see the
  zoom), so the patch is mostly "background with a dark line in it". The head can only say "vessel" or
  "not" for the whole square.</li>
<li><strong>SAM learnt from a billion masks of <em>objects</em>:</strong> compact things with a clear inside and outside, like a
  pet. A vessel tree has no inside, and one click is ambiguous: this vessel? the whole tree? the eye?</li>
</ul></div>"""))

### 2.1 SAM: one click, one box, again

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>Before you run the next cell (the section-1 cell on a retina image: one click on a vessel, then a box
around all of them), guess: what will SAM segment from one click on a thin line? And what is "the object"
inside a box that covers the whole fundus?</p>
<p>Then run it and compare the panels with the pets in section 1.</p>
<p><strong>Q5.</strong> 🟡 What did SAM segment from the click, and from the box: a vessel, part of the tree, a blob, the
whole fundus? Use the printed shares: the true vessels cover a small share of the fundus, so a mask that
marks far more is not tracing vessels. Try two other images by changing the index (on a CPU session there
are 12 held-out images).</p>
<pre><code class="language-python">demo = SPEC_EVAL[0]   # try 1, 2, ...
</code></pre></div>"""))

In [ ]:
# Q5 — SAM with one click and with one box on a held-out specialised image
demo = SPEC_EVAL[0]  # try 1, 2, ...
point, box = click_point(demo), box_prompt(demo)
t0 = time.monotonic()
sam_point = sam.probability(demo["image"], points=[point], labels=[1]) >= 0.5
sam_box = sam.probability(demo["image"], box=box) >= 0.5
print(f"{demo['name']}: two SAM calls in {time.monotonic() - t0:.1f} s on {DEVICE}")
shares = {"click mask": sam_point, "box mask": sam_box, "true vessels": demo["mask"]}
marked = " · ".join(f"{name} {m[region(demo)].mean():.0%}" for name, m in shares.items())
print(f"share of the fundus marked · {marked}")
show_masks(
    [
        ("SAM, one click", overlay(demo["image"], sam_point, THEME["fails"])),
        ("SAM, one box", overlay(demo["image"], sam_box, THEME["fails"])),
        ("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])),
    ],
    prompts={"point": point, "box": box},
)

### 2.2 A second number for thin structures: clDice

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: clDice</b><p><strong>Why IoU misleads on thin vessels.</strong> IoU counts pixels, and thick vessels hold most of them.</p>
<ul>
<li>Draw every vessel twice as wide as it is: nothing is missed, yet IoU drops to about 0.5.</li>
<li>Miss all the capillaries: they hold few pixels, so IoU barely moves.</li>
</ul>
<p><strong>clDice</strong> compares <em>centre lines</em> instead: <code>tprec</code> (how much of what you drew is real vessel) and <code>tsens</code>
(how much of the real vessel you drew), combined by harmonic mean.</p>
<ol>
<li><code>tprec</code> = prediction centre line inside the truth ÷ prediction centre line (invented branches lower it).</li>
<li><code>tsens</code> = true centre line inside the prediction ÷ true centre line (missed branches lower it).</li>
<li>clDice = 2 · tprec · tsens ÷ (tprec + tsens). Example: tprec 1.0 and tsens 0.25 give 0.40.</li>
</ol></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Where clDice comes from</b></summary><p>clDice is from Shit et al., "clDice: a novel topology-preserving loss function for tubular structure
segmentation", CVPR 2021.</p></details></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 3 min)</b><p>Fill <code>my_cldice</code>: two boolean masks in, one float out.</p>
<ul>
<li><code>skeleton(mask)</code> gives the centre line.</li>
<li><code>tprec</code> = pixels of <code>skeleton(pred)</code> inside <code>truth</code>, divided by the pixels of <code>skeleton(pred)</code>.</li>
<li><code>tsens</code> = the same with the roles swapped.</li>
<li>Return 0.0 when either skeleton is empty.</li>
</ul>
<p>The cell checks your version on two play masks (a good trace and a blob that covers everything) and
continues with ours if you skip it. Stuck? Open the collapsed "Reference solution" cell above it.</p></div>"""))

In [ ]:
# @title Reference solution (open only if stuck) { display-mode: "form" }
def _ref_cldice(pred: np.ndarray, truth: np.ndarray) -> float:
    """Reference: harmonic mean of topology precision and topology sensitivity."""
    sk_pred, sk_truth = skeleton(pred), skeleton(truth)
    if not sk_pred.any() or not sk_truth.any():
        return 0.0
    tprec = float((sk_pred & truth).sum() / sk_pred.sum())
    tsens = float((sk_truth & pred).sum() / sk_truth.sum())
    return 2 * tprec * tsens / (tprec + tsens) if tprec + tsens else 0.0

In [ ]:
# ✍️ YOUR CODE
def my_cldice(pred: np.ndarray, truth: np.ndarray) -> float:
    """clDice of two boolean masks: 2·tprec·tsens / (tprec + tsens); 0.0 when either skeleton is empty.
    e.g. 0.71"""
    ...


_play_truth = np.zeros((12, 12), bool)
_play_truth[2:4, :] = True  # a horizontal vessel, 2 px wide
_play_truth[:, 9:11] = True  # crossed by a vertical one
_play_trace = np.zeros((12, 12), bool)
_play_trace[1:5, :] = True  # the horizontal vessel, drawn too wide; the vertical one missed
_play_blob = np.ones((12, 12), bool)  # "everything is vessel"
cldice = use(
    my_cldice,
    reference=_ref_cldice,
    check=lambda f: [round(float(f(p, _play_truth)), 4) for p in (_play_trace, _play_blob)],
    name="cldice",
)
print(
    f"play example · IoU / clDice of the wide trace: {iou(_play_trace, _play_truth):.2f} / "
    f"{cldice(_play_trace, _play_truth):.2f} · of the blob: {iou(_play_blob, _play_truth):.2f} / "
    f"{cldice(_play_blob, _play_truth):.2f}"
)

### 2.3 Escalating prompts: 1 → 5 → 20 clicks, and a box

If one click is ambiguous, maybe more clicks tell SAM what we mean. Each step tells SAM something new:

1. **1 point**: one spot. SAM must guess: this vessel, the tree, or the eye?
2. **5 points**: five spots along different vessels. It can look for one thing that contains all five.
3. **20 points**: dense along the tree. More information, but still only *where*, never *what kind*.
4. **box**: a box around the whole fundus says "everything in here".

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>Write down three guesses before you run the ladder.</p>
<ul>
<li>Will IoU rise steadily from 1 to 20 clicks, jump at some point, or stay flat?</li>
<li>Will the box beat 20 clicks?</li>
<li>Where will the grey line sit? It scores a mask that says "everything is foreground": if vessels cover 12%
  of the fundus, overlap 12% ÷ union 100% gives IoU 0.12. The share printed above is yours.</li>
</ul></div>"""))

This cell runs the ladder on every evaluation image (one SAM image encoding, four cheap prompt decodings)
and prints mean IoU, clDice and the share of the fundus marked per setting, where a result near the
"everything" line means the prompt taught SAM nothing about vessels. The last figure draws the four
prompts (teal stars and box) on the demo image next to SAM's mask for each.

In [ ]:
# @title Run the SAM prompt ladder on every evaluation image { display-mode: "form" }
LADDER = {"1 point": 1, "5 points": 5, "20 points": 20, "box": None}  # setting -> number of clicks
t0 = time.monotonic()
ladder_scores = {name: [] for name in LADDER}
ladder_cover = {name: [] for name in LADDER}
ladder_demo = {}  # the demo image's masks and clicks, for the figure below
sam_box_masks = []  # every image's box mask, for the gallery in 2.6
everything = []
for it in SPEC_EVAL:
    r = region(it)
    for name, n in LADDER.items():
        pts = scribble_points(it, n) if n else None
        if n:
            pred = sam.probability(it["image"], points=pts, labels=[1] * len(pts)) >= 0.5
        else:
            pred = sam.probability(it["image"], box=box_prompt(it)) >= 0.5
        ladder_scores[name].append(score(pred, it))
        ladder_cover[name].append(float(pred[r].mean()))
        if it is demo:
            ladder_demo[name] = (pred, pts, box_prompt(it) if not n else None)
    sam_box_masks.append(pred)  # the last setting is the box
    everything.append(score(r.copy(), it))
ladder_mean = {name: mean_scores(s) for name, s in ladder_scores.items()}
everything_mean = mean_scores(everything)
spec_sec = round(time.monotonic() - t0, 1)

RESULTS["sam_prompt_ladder"] = {name: v["iou"] for name, v in ladder_mean.items()}
RESULTS["sam_prompt_ladder_cldice"] = {name: v["cldice"] for name, v in ladder_mean.items()}
RESULTS["specialised_zero_shot_iou"] = ladder_mean["box"]["iou"]  # same headline as pets: one box
RESULTS["specialised_zero_shot_point_iou"] = ladder_mean["1 point"]["iou"]
RESULTS["specialised_zero_shot_box_iou"] = ladder_mean["box"]["iou"]
RESULTS["specialised_everything_iou"] = everything_mean["iou"]
RESULTS["specialised_zero_shot_seconds"] = spec_sec

print(f"SAM prompt ladder on {len(SPEC_EVAL)} held-out images · {spec_sec} s on {DEVICE}")
print(f"{'setting':<12}{'IoU':>7}{'clDice':>8}{'marked':>8}")
for name, v in ladder_mean.items():
    print(f"{name:<12}{v['iou']:>7.3f}{v['cldice']:>8.3f}{np.mean(ladder_cover[name]):>8.0%}")
print(f"{'everything':<12}{everything_mean['iou']:>7.3f}{everything_mean['cldice']:>8.3f}{1:>8.0%}")

fig, ax = plt.subplots(figsize=(6.4, 3))
names = list(ladder_mean)
ax.bar(names, [ladder_mean[k]["iou"] for k in names], color=THEME["fails"], label="specialised, SAM")
ax.axhline(everything_mean["iou"], color=THEME["background"], ls="--", label='"everything is foreground"')
ax.axhline(RESULTS["pets_zero_shot_iou"], color=THEME["works"], ls=":", label="pets, SAM one box (section 1)")
ax.set_ylabel("mean IoU")
ax.set_ylim(0, 1)
ax.set_title("More prompts, same model")
ax.legend(fontsize=8, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

show_clicks(
    [(f"SAM, {k}", overlay(demo["image"], m, THEME["fails"]), p, b) for k, (m, p, b) in ladder_demo.items()]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]), None)]
)

**Q6.** 🟡 Compare the ladder with your prediction. Read the table row by row: IoU, clDice, and *marked*
(the share of the fundus SAM marked). How to read a row: IoU 0.30 with 70% marked, against a grey line at
0.29, is a big mask that is barely better than marking everything.

- Which rows beat the grey line by more than 0.05 IoU?
- Did more clicks bring the masks closer to the vessels, or just make them bigger?
- What does that say about what SAM thinks an object is?

<details><summary>What to look for</summary>

- Read the IoU next to the grey line, not on its own. A mask that marks most of the fundus scores the
  "everything" baseline, whatever the prompt.
- If more clicks push "marked" up while IoU stays near the line, SAM answers "the object that contains all
  these points". For a vessel tree that object is the eye.
- If clicks do help, look where: usually along the thick vessels near the clicks, rarely the capillaries
  in between.
- A prompt says *where*, not *what kind of thing*. Section 7 comes back to prompting.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: over-trusting SAM&#x27;s confident masks</b><p>SAM returns a crisp mask for every prompt. Its own confidence score is high for a mask of the whole eye,
because the whole eye <em>is</em> a good object. Nothing in the output says "this is the wrong kind of thing".</p>
<ul>
<li>Score against a baseline that knows nothing (the grey line).</li>
<li>Always look at the mask.</li>
</ul></div>"""))

### 2.4 🔴 Optional: What DINOv3 sees here

Same 3-colour view as section 1.3 (a PCA of the 384 numbers per patch, painted red, green and blue); this
cell shows it on retina images, to see whether the features separate vessels from background before any
training.

In [ ]:
# @title DINOv3 PCA colours on three retina images { display-mode: "form" }
fig_items = [load_item(SPECIALISED, it) for it in SPECIALISED["fit"][:3]]
panels = []
for it in fig_items:
    feat = features_for(SPECIALISED, it, SPECIALISED["feat_side"])
    title = f"DINOv3 PCA · {feat.shape[2]}×{feat.shape[1]} patches"
    panels += [(it["name"], it["image"]), (title, pca_rgb(feat, it["image"].shape[:2]))]
show_masks(panels, width=2.6)

**Q7.** 🔴 Do the thick vessels share a colour at all, across the three images? Which of the optic disc
(bright round patch), the dark border and the vessels stands out most from its surroundings? Compare with
the pets, where one direction separated animal from floor.

<details><summary>What to look for</summary>

- The colours are the directions of largest variation. On a fundus that is rarely the vessels: it is the
  optic disc, the illumination falling off towards the edge, the border.
- Thick vessels may show as a faint trace of blocks; thin ones vanish inside the 16 px squares.
- That does not mean the features hold no vessel information (a trained head can find a direction for it).
  It is just not the first thing the model "sees".

</details>

### 2.5 The linear head, and the resolution it looks at

The head from section 1.4, trained the same way on the full masks of five retina images (the last time
training sees a full mask: a perfect vessel map costs an expert 3 to 5 hours per fundus). We train it
twice, on DINOv3 features at two resolutions:

- **512 px long side**: each 16 px patch covers 32 × 32 pixels of the 1024 px crop.
- **1024 px long side**: each patch covers 16 × 16 pixels. Four times the patches, about 16 times the
  attention cost (see the go-deeper below the chart).

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Why 512 and not 518?</b></summary><p>DINOv2 papers use 518 px: 37 patches of 14 px. DINOv3's patches are 16 px, so 512 px (32 patches) is the
same idea.</p></details></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>With <em>perfect</em> labels, write two guesses.</p>
<ul>
<li>Will the head beat SAM's one-box IoU from the 2.3 table?</li>
<li>Will doubling the resolution add more than 0.05 to IoU, to clDice, to both or to neither?</li>
</ul></div>"""))

This cell trains one head per resolution on the five fit images and prints the seconds it took; if it is
slow, set `TRAIN_YOURSELF = False` and run it again to load our checkpoints (trained the same way on all
fit images).

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "full" heads instead of training
N_TRAIN_SPEC, HEAD_STEPS = 5, 30 if SMOKE else 200
SIDES = SPECIALISED["feature_long_sides"]  # [512, 1024]: the two resolutions we compare
SIDES = [s // 4 for s in SIDES] if SMOKE else SIDES  # smoke thumbnails are 256 px
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

spec_train = [load_item(SPECIALISED, it) for it in SPECIALISED["fit"][:N_TRAIN_SPEC]]
spec_heads = {}  # long side -> (head tensors, training seconds)
for side in SIDES:
    if TRAIN_YOURSELF:
        feats = [features_for(SPECIALISED, it, side) for it in spec_train]
        labels = [np.where(region(it), it["mask"], -1).astype(np.int8) for it in spec_train]  # border ignored
        spec_heads[side] = train_head(feats, labels, HEAD_STEPS)
        src = f"trained here on {len(spec_train)} images, {HEAD_STEPS} steps"
    else:
        tensors, entry = load_head(SPECIALISED, "full", side)
        spec_heads[side] = (tensors, entry["train_sec"])
        src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
    print(f"linear head @ {side} px · {src} · {spec_heads[side][1]:.1f} s on {DEVICE}")
if sum(sec for _, sec in spec_heads.values()) > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoints next time.")

This cell scores both heads on the evaluation images with IoU and clDice and zooms into the centre of the
demo image to compare the two resolutions on the thinnest vessels.

In [ ]:
# @title Score both heads and zoom into the thinnest vessels { display-mode: "form" }
t0 = time.monotonic()
head_preds, res_scores = {}, {}
for side, (tensors, _) in spec_heads.items():
    head_preds[side] = [
        predict_head(tensors, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
    ]
    res_scores[side] = mean_scores([score(p, it) for p, it in zip(head_preds[side], SPEC_EVAL, strict=True)])
lo, hi = min(SIDES), max(SIDES)
RESULTS["specialised_head_iou"] = res_scores[hi]["iou"]
RESULTS["specialised_head"] = {
    "iou": res_scores[hi]["iou"],
    "cldice": res_scores[hi]["cldice"],
    "long_side": hi,
    "n_train": len(spec_train),
    "train_sec": round(float(spec_heads[hi][1]), 2),
    "infer_sec": round(time.monotonic() - t0, 1),
}
RESULTS["resolution_effect"] = {f"{side} px": v for side, v in res_scores.items()}
for side, v in res_scores.items():
    print(f"linear head @ {side:>4} px · mean IoU {v['iou']:.3f} · clDice {v['cldice']:.3f}")
print(f"SAM one box · mean IoU {ladder_mean['box']['iou']:.3f} · clDice {ladder_mean['box']['cldice']:.3f}")

k = next(i for i, it in enumerate(SPEC_EVAL) if it is demo)
h, w = demo["mask"].shape
zoom = (slice(h // 4, 3 * h // 4), slice(w // 4, 3 * w // 4))  # the central quarter
show_masks(
    [
        (
            f"head @ {s} px · IoU {score(head_preds[s][k], demo)['iou']:.2f}",
            overlay(demo["image"], head_preds[s][k], THEME["fails"])[zoom],
        )
        for s in SIDES
    ]
    +[("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])[zoom])],
    width=3.0,
)

This cell puts the section-1 numbers next to this section's, the same three tools on both tasks, so the gap
is one picture.

In [ ]:
# @title Pets vs the specialised task, side by side { display-mode: "form" }
rows = {
    "SAM, one click": (RESULTS["pets_zero_shot_point_iou"], RESULTS["specialised_zero_shot_point_iou"]),
    "SAM, one box": (RESULTS["pets_zero_shot_box_iou"], RESULTS["specialised_zero_shot_box_iou"]),
    "linear head": (RESULTS["pets_head_iou"], RESULTS["specialised_head_iou"]),
}
print(f"{'tool':<16}{'pets IoU':>10}{'specialised IoU':>17}")
for name, (p, s) in rows.items():
    print(f"{name:<16}{p:>10.3f}{s:>17.3f}")
print(f"{'everything':<16}{'':>10}{RESULTS['specialised_everything_iou']:>17.3f}")

x = np.arange(len(rows))
fig, ax = plt.subplots(figsize=(6.4, 3))
ax.bar(x - 0.2, [v[0] for v in rows.values()], 0.4, color=THEME["works"], label=PETS["name"])
ax.bar(x + 0.2, [v[1] for v in rows.values()], 0.4, color=THEME["fails"], label=SPECIALISED["name"])
ax.axhline(RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--",
           label='specialised, "everything is foreground"')
ax.set_xticks(x, list(rows))
ax.set_ylabel("mean IoU on held-out images")
ax.set_ylim(0, 1)
ax.set_title("Same tools, two tasks")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Resolution is not free: the attention arithmetic</b></summary><p>Four times the patches costs about 16 times the attention, so when the structure is thinner than a patch,
check the resolution before anything else.</p>
<p>Self-attention compares every patch with every other, so the cost grows with the <em>square</em> of the patches.</p>
<ol>
<li>512 px side: 32 × 32 = 1 024 patches, about 1 million patch pairs.</li>
<li>1024 px side: 64 × 64 = 4 096 patches, about 17 million pairs: 4× the patches, about 16× the attention
   cost.</li>
<li>The whole 3504 × 2336 fundus at 16 px: about 219 × 146 = 32 000 patches, about 1 billion pairs. That is
   why we work on crops and precompute the features.</li>
</ol></details></div>"""))

### 2.6 The failure gallery

Numbers average the failures away; the gallery shows them. The next cell sorts the evaluation images from
worst to best (by the head's IoU at the higher resolution) and shows three at a time: the image, SAM with a
box, the head, and the ground truth.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Question</b><p>Run the next cell: it draws the failure gallery from the predictions you already have, with each image's
IoU and clDice in the titles.</p>
<p><strong>Q8.</strong> 🟡 Read the IoU and clDice in each title. For the worst image, is the failure the same kind for SAM
and for the head? Pick from: missed thin vessels, vessels drawn too wide, the optic disc or the border
marked as vessel, whole regions marked. Page with the start index:</p>
<pre><code class="language-python">GALLERY_START = 0   # 0 = the three worst; try 3, 6, ...
</code></pre></div>"""))

In [ ]:
# Q8 — the failure gallery, worst images first
GALLERY_START = 0  # 0 = the three worst; try 3, 6, ...
if not isinstance(GALLERY_START, int) or not 0 <= GALLERY_START < len(SPEC_EVAL):
    raise SystemExit(f"GALLERY_START must be a whole number from 0 to {len(SPEC_EVAL) - 1}.")
gallery = [score(p, it) for p, it in zip(head_preds[hi], SPEC_EVAL, strict=True)]
order = np.argsort([s["iou"] for s in gallery])
for k in order[GALLERY_START : GALLERY_START + 3]:
    it, sam_s, head_s = SPEC_EVAL[k], ladder_scores["box"][k], gallery[k]
    show_masks(
        [
            (it["name"], it["image"]),
            (
                f"SAM box · IoU {sam_s['iou']:.2f} · clDice {sam_s['cldice']:.2f}",
                overlay(it["image"], sam_box_masks[k], THEME["fails"]),
            ),
            (
                f"head @ {hi} px · IoU {head_s['iou']:.2f} · clDice {head_s['cldice']:.2f}",
                overlay(it["image"], head_preds[hi][k], THEME["fails"]),
            ),
            ("ground truth", overlay(it["image"], it["mask"], THEME["truth"])),
        ],
        width=2.8,
    )

Secondary visual: this cell shows the typical (median) and the best image of the same ranking, to see what
an IoU of, say, 0.3 versus 0.7 looks like next to the worst ones.

In [ ]:
# @title For contrast: the typical and the best image of the ranking { display-mode: "form" }
for label, k in (("typical (median)", order[len(order) // 2]), ("best", order[-1])):
    it, sam_s, head_s = SPEC_EVAL[k], ladder_scores["box"][k], gallery[k]
    show_masks(
        [
            (f"{label} · {it['name']}", it["image"]),
            (
                f"SAM box · IoU {sam_s['iou']:.2f} · clDice {sam_s['cldice']:.2f}",
                overlay(it["image"], sam_box_masks[k], THEME["fails"]),
            ),
            (
                f"head @ {hi} px · IoU {head_s['iou']:.2f} · clDice {head_s['cldice']:.2f}",
                overlay(it["image"], head_preds[hi][k], THEME["fails"]),
            ),
            ("ground truth", overlay(it["image"], it["mask"], THEME["truth"])),
        ],
        width=2.8,
    )

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>👥 Pairs, 5 min: whose image fails worst?</b><p>Each of you picks a different <code>GALLERY_START</code> (change that value only, not <code>demo</code>) and finds the worst
mask you can.</p>
<ul>
<li>Show it to your neighbour and name the failure in three words ("misses thin vessels", "marks the disc",
  "the whole eye").</li>
<li>Is it the same failure for SAM and for the head?</li>
</ul></div>"""))

<details><summary>What people usually find</summary>

- SAM tends to fail *globally*: it marks a whole region or one thick vessel, because it seeks an object.
- The head fails *locally*: it finds the thick vessels, draws them as wide as a patch, and loses the
  capillaries, because it can only label 16 px squares.
- On the whole fundus both get worse, since every vessel is now thinner than a patch (optional 2.9 below).

</details>

### 2.7 A checklist for a new problem

**Q9.** 🟡 Use what is on screen as a checklist: the "everything is foreground" line in the ladder, the
512 versus 1024 px chart, and IoU next to clDice. A colleague segments cracks in concrete and reports:

- SAM with one box: IoU 0.31.
- "Everything is foreground": IoU 0.28.
- Linear head at 512 px: IoU 0.35; at 1024 px: IoU 0.48.
- IoU 0.48 but clDice 0.22.

Which two or three signs from this section tell you the foundation model is not good enough out of the box?

<details><summary>One possible answer</summary>

- SAM's 0.31 sits near the 0.28 baseline that knows nothing: it learnt little about cracks.
- Going from 512 to 1024 px moves the head by 0.13: the cracks are thinner than a patch, so the
  representation, not the labels, is the bottleneck.
- IoU 0.48 next to clDice 0.22: a pixel metric and a structure metric disagree, so the mask misses the
  branches.
- And always: the mask looks crisp and confident, so look at it next to the truth.

</details>

### 2.8 🔴 Optional: what if the model paints only the thick trunk?

Back to the two numbers of 2.2, now on a real retina image: three fake predictions next to the truth, the
playful one being a model that paints only the thick trunk: violet marks the true centre line the
prediction covers, green the part it misses, grey is the prediction.

In [ ]:
# @title Playful test: what if the model paints only the thick trunk? { display-mode: "form" }
from scipy import ndimage as ndi

sim = SPEC_EVAL[0]
sim_truth = sim["mask"] & region(sim)
sim_line = skeleton(sim_truth)
sim_width = float(sim_truth.sum() / max(sim_line.sum(), 1))  # mean vessel width in px
sim_dist = ndi.distance_transform_edt(sim_truth)  # distance of each vessel pixel to the vessel edge
sim_cut = np.percentile(sim_dist[sim_truth], 60)  # keep the 40 % thickest vessel pixels
fakes = {
    "trunk only": ndi.binary_dilation(sim_dist >= sim_cut, iterations=max(int(sim_cut), 1)) & sim_truth,
    "2× too wide": ndi.binary_dilation(sim_truth, iterations=max(int(round(sim_width / 2)), 1)) & region(sim),
    '"everything"': region(sim),
}
fat = max(1, max(sim_truth.shape) // 400)  # draw the 1 px centre line a few px thick so it is visible


def trace_panel(pred: np.ndarray) -> np.ndarray:
    """The prediction in grey; the true centre line violet where covered, green where missed."""
    img = overlay(sim["image"], pred, THEME["background"])
    img = overlay(img, ndi.binary_dilation(sim_line & pred, iterations=fat), THEME["works"], alpha=1.0)
    return overlay(img, ndi.binary_dilation(sim_line & ~pred, iterations=fat), THEME["truth"], alpha=1.0)


panels = [
    (
        "truth and its centre line",
        overlay(
            overlay(sim["image"], sim_truth, THEME["truth"]),
            ndi.binary_dilation(sim_line, iterations=fat),
            "#FFFFFF",
            alpha=1.0,
        ),
    )
]
for name, pred in fakes.items():
    found = float((sim_line & pred).sum() / max(sim_line.sum(), 1))
    panels.append(
        (
            f"{name} · IoU {iou(pred, sim_truth):.2f} · clDice {cldice(pred, sim_truth):.2f}\n"
            f"true centre line covered: {found:.0%}",
            trace_panel(pred),
        )
    )
show_masks(panels, width=3.0)

### 2.9 🔴 Optional: the whole fundus

The gallery shows 1024 px crops. This cell runs SAM with a box and the head on the whole fundus of the demo
image shrunk to 1024 px (vessels only 1–3 px wide), next to the truth.

In [ ]:
# @title The failure gallery on the whole fundus { display-mode: "form" }
# the whole fundus of the demo image, shrunk so its long side is 1024 px (smoke: 256 px)
meta = next(m for m in SPECIALISED["eval"] if m["name"] == demo["name"])
root = SPECIALISED["dir"]
full = {
    "image": np.array(Image.open(root / meta["full"]["image"]).convert("RGB")),
    "mask": read_png(root / meta["full"]["mask"]) > 127,
    "fov": read_png(root / meta["full"]["fov"]) > 127,
}
full_sam = sam.probability(full["image"], box=box_prompt(full)) >= 0.5
full_feat = dino.features(full["image"], max(full["image"].shape[:2]))
full_head = predict_head(spec_heads[hi][0], full_feat, full["mask"].shape)
full_masks = {"SAM box": full_sam, f"head @ {hi} px": full_head}
full_s = {name: score(m, full) for name, m in full_masks.items()}
show_masks(
    [(f"whole fundus · {meta['name']}", full["image"])]
    + [
        (f"{name} · IoU {full_s[name]['iou']:.2f} · clDice {full_s[name]['cldice']:.2f}",
         overlay(full["image"], m, THEME["fails"]))
        for name, m in full_masks.items()
    ]
    + [("ground truth", overlay(full["image"], full["mask"], THEME["truth"]))],
    width=2.8,
)
RESULTS["specialised_full_view"] = full_s

**Got a photo of something thin or specialised?** Section 9 runs SAM, the ladder and the head on it.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li><strong>Objects thinner than a 16 px patch</strong> → raise the resolution first, and check how far the numbers move.</li>
<li><strong>Score close to the "everything is foreground" line</strong> → the model learnt nothing about the task; always
  compare with a baseline that knows nothing, and look at the mask next to the truth.</li>
<li><strong>More clicks make SAM's mask bigger, not thinner</strong> → stop prompting: a prompt says <em>where</em>, not <em>what
  kind of thing</em>.</li>
<li><strong>IoU and clDice disagree</strong> → trust the structure metric on thin objects, and look at the masks.</li>
</ul></div>"""))

## 3. Hide the labels: the naive baseline

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>Train a head with "missing = background" and watch it fail differently.</p></div>"""))

A perfect vessel map of one fundus takes an expert 3 to 5 hours, so real label files are rarely complete.

Picture an annotator who traced the big vessels, ran out of time halfway through the tree, and saved the
file. Nothing in it says "I stopped here": every pixel they did not paint is stored as background, exactly
like the pixels that really are background.

We make such a file ourselves by hiding about half of the vessels from the ground truth, train the
section-2 head on it the obvious way, and score it on the vessels we hid. From here on the full masks are
used for scoring only.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: incomplete supervision</b><p>The cheap labels of the rest of the notebook are of two kinds, both leaving pixels nobody looked at:
<strong>missing labels</strong> (this section and section 5) and <strong>scribbles</strong> (section 4).</p>
<p>What should a model believe about those pixels? The obvious answer, and what a plain loss does, is
"background".</p></div>"""))

This helper cell defines how we hide labels, the same way `scripts/prepare_data.py` built the cached files:
cut the ground-truth skeleton into segments at its junctions, give every foreground pixel to its nearest
segment, then hide a random share of the segments.

In [ ]:
# @title Missing labels: cut into segments, hide a share { display-mode: "form" }
import zlib

from scipy import ndimage as ndi


def label_units(mask: np.ndarray, thin: bool) -> np.ndarray:
    """Label map of the units we hide (0 = background): skeleton segments for thin structures or a single
    object, connected components otherwise. Every foreground pixel gets the label of its nearest segment."""
    lab, n = ndi.label(mask)
    if not thin and n >= 2:
        return lab
    sk = skeleton(mask)
    neighbours = ndi.convolve(sk.astype(np.uint8), np.ones((3, 3), np.uint8), mode="constant") - sk
    seg, _ = ndi.label(sk & ~(sk & (neighbours > 2)), structure=np.ones((3, 3)))  # cut at junctions
    if seg.max() == 0:
        return np.where(mask, 1, 0)
    _, (iy, ix) = ndi.distance_transform_edt(seg == 0, return_indices=True)
    return np.where(mask, seg[iy, ix], 0)


def hide_units(mask: np.ndarray, fraction: float, rng: np.random.Generator, thin: bool) -> np.ndarray:
    """The 'missing labels' file: the ground truth with ~fraction of its units removed. Returns the visible
    mask; the hidden foreground is mask & ~visible."""
    lab = label_units(mask, thin)
    ids = np.arange(1, int(lab.max()) + 1)
    rng.shuffle(ids)
    k = int(round(len(ids) * fraction))
    hidden_ids = ids[: max(1, k)] if len(ids) >= 2 else ids[:0]
    return mask & ~np.isin(lab, hidden_ids)


def hide_rng(name: str) -> np.random.Generator:
    """One fixed random generator per image, so the same segments are hidden on every run (N-013)."""
    return np.random.default_rng([SEED, zlib.crc32(f"{name}/hide".encode())])


def part_recall(preds: list[np.ndarray], items: list[dict], part: str) -> float:
    """Mean over images of the share of item[part] ("mask", "visible", "hidden"), inside the evaluation
    region, that the prediction marks. Uses `hidden_recall` from 3.2."""
    pairs = zip(preds, items, strict=True)
    return float(np.mean([hidden_recall(p, it[part] & region(it)) for p, it in pairs]))


def two_colour(
    image: np.ndarray, good: np.ndarray, bad: np.ndarray, bad_colour: str = THEME["fails"]
) -> np.ndarray:
    """Overlay `good` in violet ("works") and `bad` in `bad_colour` (default orange, "fails") on one image."""
    return overlay(overlay(image, good, THEME["works"]), bad, bad_colour)

### 3.1 Make a half-finished label file

This cell hides about half of the vessel segments (the share comes from the dataset config) of the five
retina training images of section 2.5 and shows one of them: the full truth we keep for scoring, the labels
training may use, and what the label file now wrongly calls background.

In [ ]:
# @title Hide half of the vessel segments and look at the result { display-mode: "form" }
MISSING = SPECIALISED["missing_labels"]  # {"hidden_fraction": 0.5, "unit": ...}: never hard-coded here
miss_train = [load_item(SPECIALISED, it) for it in SPECIALISED["fit"][:N_TRAIN_SPEC]]
same_as_cache = []
for it in miss_train:
    rng = hide_rng(it["name"])
    visible = hide_units(it["mask"], MISSING["hidden_fraction"], rng, SPECIALISED["thin_structures"])
    same_as_cache.append(float((visible == it["visible"]).mean()))
    it["visible"], it["hidden"] = visible, it["mask"] & ~visible  # from now on: what training may see
hidden_share = float(np.mean([it["hidden"].sum() / max(it["mask"].sum(), 1) for it in miss_train]))
print(
    f"{len(miss_train)} training images · {hidden_share:.0%} of the vessel pixels hidden "
    f"(asked for {MISSING['hidden_fraction']:.0%} of the segments) · pixels identical to the cached labels "
    f"our checkpoints used: {np.mean(same_as_cache):.2%}"
)

ex = miss_train[0]
show_masks(
    [
        (ex["name"], ex["image"]),
        ("ground truth · evaluation only", overlay(ex["image"], ex["mask"], THEME["truth"])),
        ("labels training sees: vessel", overlay(ex["image"], ex["visible"], THEME["scribble"])),
        ('hidden: the file says "background"', overlay(ex["image"], ex["hidden"], THEME["fails"])),
    ],
    width=2.8,
)

### 3.2 A metric for the missing vessels: recall

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: recall and precision</b><p><strong>Recall</strong> = found ÷ there, <strong>precision</strong> = right ÷ marked; IoU mixes them, and here we want them apart
(our question is recall on the hidden vessels).</p>
<ul>
<li>Recall: 400 hidden vessel pixels, the prediction marks 120 of them: 120 ÷ 400 = 30%.</li>
<li>Precision: it marked 150 pixels and 120 are vessels: 120 ÷ 150 = 80%.</li>
</ul></div>"""))

This helper cell defines `hidden_recall(pred, hidden)`: the share of the `hidden` pixels that the prediction
marks (0.0 when nothing is hidden). Every recall below uses it.

In [ ]:
# @title hidden_recall: the share of the hidden pixels found { display-mode: "form" }
def hidden_recall(pred: np.ndarray, hidden: np.ndarray) -> float:
    """Share of the `hidden` pixels that `pred` marks as foreground; 0.0 when `hidden` is empty."""
    hidden = np.asarray(hidden, bool)
    return float(np.asarray(pred, bool)[hidden].mean()) if hidden.any() else 0.0

### 3.3 The naive baseline: missing = background

The naive recipe is what you get by passing the label file straight into the textbook loss: every pixel not
painted as vessel is a background target, and plain binary cross-entropy counts every pixel equally.

Two things change from section 2.5, the labels and the loss weighting. The hidden labels cause the gap in
the chart below; the plain loss only makes the head believe them more strictly, and 3.4 separates the two.
Everything else is the same: the same five images, DINOv3 features, linear head and steps.

A playful extreme first, to have a scale: this cell scores a fake model that believes the label file
perfectly (its prediction is the file itself), and it comes back as the grey bars in the chart below.

In [ ]:
# @title Playful test: a model that believes the label file perfectly { display-mode: "form" }
believer = [it["visible"] for it in miss_train]  # its prediction is the file itself
b_rec_hidden = part_recall(believer, miss_train, "hidden")
b_rec_visible = part_recall(believer, miss_train, "visible")
b_pairs = list(zip(believer, miss_train, strict=True))
b_precision = float(np.mean([hidden_recall(it["mask"], b & region(it)) for b, it in b_pairs]))
b_iou = float(np.mean([iou(b & region(it), it["mask"] & region(it)) for b, it in b_pairs]))
print(
    f"believes the file perfectly · recall on hidden vessels {b_rec_hidden:.0%} · on visible vessels "
    f"{b_rec_visible:.0%} · precision {b_precision:.0%} · IoU {b_iou:.2f}"
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The head was told that about half of the vessels in these five images are background. Write two guesses.</p>
<ul>
<li>On those hidden vessels, will its recall be near the section-2 head's, about half of it, or close to
  zero (like the believer above)?</li>
<li>On new images, will it mark more pixels than the section-2 head, or fewer?</li>
</ul></div>"""))

This cell trains the naive head on the incomplete labels only (the fundus border is ignored, as in 2.5) and
prints the seconds it took; if it is slow, set `TRAIN_YOURSELF = False` and run it again to load our
checkpoint, trained the same way on all fit images.

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "hidden_bg" head instead of training
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

side = SPECIALISED["feat_side"]
miss_feats = [features_for(SPECIALISED, it, side) for it in miss_train]
# missing = background: the label file as it is. The hidden ground truth never enters training (L-005).
naive_labels = [np.where(region(it), it["visible"], -1).astype(np.int8) for it in miss_train]
if TRAIN_YOURSELF:
    naive_head, naive_sec = train_head(miss_feats, naive_labels, HEAD_STEPS, balanced=False)
    src = f"trained here on {len(miss_train)} images, {HEAD_STEPS} steps, plain cross-entropy"
else:
    naive_head, entry = load_head(SPECIALISED, "hidden_bg", side)
    naive_sec = entry["train_sec"]
    src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
print(f"naive head @ {side} px · {src} · {naive_sec:.1f} s on {DEVICE}")
if naive_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

This cell scores the naive head and the section-2 head (full masks) on the vessel pixels we hid, the ones
training saw, and every vessel of the held-out images, and plots the recalls next to the believer's.

In [ ]:
# @title Score both heads on hidden, visible and held-out vessels { display-mode: "form" }
full_head = spec_heads[side][0]  # section 2.5: same features, trained on full masks
HEADS = {"full masks (section 2)": full_head, "naive: missing = background": naive_head}
t0 = time.monotonic()
naive_report = {}
for name, tensors in HEADS.items():
    fit_preds = [
        predict_head(tensors, f, it["mask"].shape) for f, it in zip(miss_feats, miss_train, strict=True)
    ]
    held_preds = [
        predict_head(tensors, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
    ]
    held = mean_scores([score(p, it) for p, it in zip(held_preds, SPEC_EVAL, strict=True)])
    # precision = the share of the marked pixels that are true vessels
    held_pairs = zip(held_preds, SPEC_EVAL, strict=True)
    precision = [hidden_recall(it["mask"], p & region(it)) for p, it in held_pairs]
    naive_report[name] = {
        "recall_hidden_fit": part_recall(fit_preds, miss_train, "hidden"),
        "recall_visible_fit": part_recall(fit_preds, miss_train, "visible"),
        "recall_held": part_recall(held_preds, SPEC_EVAL, "mask"),
        "precision_held": float(np.mean(precision)),
        "iou_held": held["iou"],
        "cldice_held": held["cldice"],
        "fit_preds": fit_preds,
        "held_preds": held_preds,
    }
infer_sec = round(time.monotonic() - t0, 1)

naive = naive_report["naive: missing = background"]
RESULTS["naive_recall_hidden_fg"] = naive["recall_hidden_fit"]  # recall on the vessels hidden in training
RESULTS["naive_head"] = {k: v for k, v in naive.items() if not k.endswith("_preds")} | {
    "long_side": side,
    "n_train": len(miss_train),
    "hidden_share": hidden_share,
    "train_sec": round(float(naive_sec), 2),
    "infer_sec": infer_sec,
}

GROUPS = {
    "recall_hidden_fit": "hidden vessels\n(told: background)",
    "recall_visible_fit": "visible vessels\n(told: vessel)",
    "recall_held": "held-out images\n(all vessels)",
}
print(f"{'head':<30}{'recall: hidden':>15}{'visible':>9}{'held-out':>10}   held-out IoU / clDice / precision")
for name, r in naive_report.items():
    print(
        f"{name:<30}{r['recall_hidden_fit']:>15.3f}{r['recall_visible_fit']:>9.3f}{r['recall_held']:>10.3f}"
        f"   {r['iou_held']:.3f} / {r['cldice_held']:.3f} / {r['precision_held']:.3f}"
    )

x = np.arange(len(GROUPS))
fig, ax = plt.subplots(figsize=(6.4, 3))
# the believer (its prediction is the file) only exists on the training images: no bar for the held-out group
believer_bars = {0: b_rec_hidden, 1: b_rec_visible}
ax.bar(
    [x[i] - 0.27 for i in believer_bars], list(believer_bars.values()), 0.26, color=THEME["background"],
    label="believer: the file itself",
)
for shift, name, colour in zip((0.0, 0.27), HEADS, (THEME["works"], THEME["fails"]), strict=True):
    ax.bar(x + shift, [naive_report[name][g] for g in GROUPS], 0.26, color=colour, label=name)
ax.set_xticks(x, list(GROUPS.values()), fontsize=8)
ax.set_ylabel("recall (share of vessel pixels found)")
ax.set_ylim(0, 1.3)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_title("Which vessels does each head still find?")
ax.legend(fontsize=8, frameon=False, loc="upper right", ncol=1)
plt.tight_layout()
plt.show()

This cell shows the naive head's worst, typical (median) and best training image by recall on the hidden
vessels: violet is a hidden vessel pixel it still found, green one it missed.

In [ ]:
# @title Worst, typical and best training image by hidden recall { display-mode: "form" }
per_image = [hidden_recall(p, it["hidden"]) for p, it in zip(naive["fit_preds"], miss_train, strict=True)]
rank = np.argsort(per_image)
panels = []
for label, k in (("worst", rank[0]), ("typical (median)", rank[len(rank) // 2]), ("best", rank[-1])):
    it, pred = miss_train[k], naive["fit_preds"][k]
    found, missed = it["hidden"] & pred, it["hidden"] & ~pred
    panels.append(
        (
            f"{label} · found {int(found.sum())} of {int(it['hidden'].sum())} hidden px = {per_image[k]:.0%}",
            two_colour(it["image"], found, missed, THEME["truth"]),
        )
    )
show_masks(panels, width=3.4)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Question</b><p><strong>Q10.</strong> 🟡 Look at the worst, typical and best images above: which hidden vessels did the naive head
miss (green), and which did it still find (violet)? Then run the next cell, which draws the naive head's
predictions on the held-out demo image of section 2 next to the section-2 head's (no new training, no model
calls): does it fail the same way (too wide, the disc, the whole eye) or differently?</p></div>"""))

In [ ]:
# Q10 — the naive head on the held-out demo image, next to the section-2 head
k = next(i for i, e in enumerate(SPEC_EVAL) if e is demo)
show_masks(
    [
        (
            f"{name} · IoU {score(r['held_preds'][k], demo)['iou']:.2f}",
            overlay(demo["image"], r["held_preds"][k], THEME["fails"]),
        )
        for name, r in naive_report.items()
    ]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]))],
    width=3.0,
)

This cell shows two look-alike patches from a training image, one the annotator painted (target 1) and one
they never reached (target 0), which is all the head sees of the problem.

In [ ]:
# @title Picture: two look-alike patches, two targets { display-mode: "form" }
pic_side = int(2 * 16 * max(miss_train[0]["mask"].shape) / SPECIALISED["feat_side"])  # 2 × 2 patches, in px
pic_half = pic_side // 2
pic_pair = None
for pic_item in miss_train:
    pic_gray = pic_item["image"].astype(float).mean(axis=2)
    pic_h, pic_w = pic_gray.shape
    cands = []
    for pic_part in ("hidden", "visible"):
        pys, pxs = np.nonzero(skeleton(pic_item[pic_part]))
        inside = (pys >= pic_half) & (pys < pic_h - pic_half) & (pxs >= pic_half) & (pxs < pic_w - pic_half)
        cands.append(np.stack([pys[inside], pxs[inside]], 1))
    if len(cands[0]) and len(cands[1]):
        pic_pair = (pic_item, cands)
        break
if pic_pair is None:
    print("no hidden and visible vessel segment far enough from the border to draw the picture")
else:
    pic_item, (pic_hid, pic_vis) = pic_pair
    pic_gray = pic_item["image"].astype(float).mean(axis=2)

    def pic_crop(arr: np.ndarray, centre: np.ndarray) -> np.ndarray:
        return arr[centre[0] - pic_half : centre[0] + pic_half, centre[1] - pic_half : centre[1] + pic_half]

    hid_centre = pic_hid[len(pic_hid) // 2]
    vis_pool = pic_vis[:: max(1, len(pic_vis) // 300)]
    gaps = [float(np.abs(pic_crop(pic_gray, c) - pic_crop(pic_gray, hid_centre)).mean()) for c in vis_pool]
    vis_centre = vis_pool[int(np.argmin(gaps))]  # the visible patch that looks most like the hidden one
    fig, axes = plt.subplots(1, 2, figsize=(4.4, 2.4))
    for ax, centre, title, colour in (
        (axes[0], vis_centre, "target 1: painted", THEME["scribble"]),
        (axes[1], hid_centre, "target 0: never reached", THEME["background"]),
    ):
        ax.imshow(pic_crop(pic_item["image"], centre))
        ax.set_title(title, fontsize=9, color=colour)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(colour)
            spine.set_linewidth(3)
    fig.suptitle("same features, two targets: the head can only say p ≈ 0.5", fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: why missing = background fails, step by step</b><ol>
<li>The annotator traces vessels A, B and C, then stops. Vessels D and E are never painted.</li>
<li>The file stores D and E as 0, the same value as real background.</li>
<li>Training sees patches that look like A with target 1, and patches that look like D (the same features)
   with target 0.</li>
<li>For look-alike patches, plain cross-entropy settles on the share that was labelled. If half of the
   vessels like A are hidden, p comes out near 0.5.</li>
<li>p near 0.5 sits on the threshold, so small differences tip many vessels to "background". The head keeps
   the clearest, thickest vessels and drops the rest.</li>
</ol>
<p>Consequence: recall on the hidden vessels falls, while precision stays high.</p></div>"""))

<details><summary>What to look for</summary>

- The naive head did what the labels told it: it kept the clearest vessels, on the hidden segments, on new
  images, and often even on the visible segments it was shown.
- Compare the precision column: what it does mark is mostly right. The section-2 head failed by being too
  generous (vessels as wide as a patch); this one fails by being too stingy.
- Look at the held-out IoU of the two heads: it can be close, because IoU rewards the precision the naive
  head gained as much as it punishes the recall it lost. One number would have hidden the failure.
- Same features, same head: the only thing that changed is what we told it about the unlabelled pixels.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Pitfall: a precise model looks trustworthy</b></summary><p>Every vessel the naive head marks is a real vessel, so a quick look at a few predictions, or a precision
score, says "works". The failure is in what is <em>not</em> marked, and you can only measure that against labels
that are complete.</p>
<ul>
<li>On your own data, keep a small, fully labelled evaluation set, even if training uses cheap labels.</li>
</ul></details></div>"""))

### 3.4 🔴 Optional: is it the labels, or the loss?

Plain cross-entropy is one of the two changes from section 2.5: the heads of sections 1 and 2 used the
balanced loss.

**Q11.** 🔴 Train the same naive labels with the balanced loss of section 2.5 and compare its recall on the
hidden vessels with the plain head's. Does reweighting fix "missing = background"? What did it fix?

```python
head, sec = train_head(miss_feats, naive_labels, HEAD_STEPS, balanced=True)
```

This cell trains that balanced variant on the same incomplete labels (seconds) and prints its recall next
to the plain head's.

In [ ]:
# Q11 — the same "missing = background" labels with the balanced loss of section 2.5
balanced_head, balanced_sec = train_head(miss_feats, naive_labels, HEAD_STEPS, balanced=True)
bal_fit = [
    predict_head(balanced_head, f, it["mask"].shape) for f, it in zip(miss_feats, miss_train, strict=True)
]
bal_hidden, bal_visible = (part_recall(bal_fit, miss_train, part) for part in ("hidden", "visible"))
RESULTS["naive_balanced_recall_hidden_fg"] = bal_hidden
print(f"balanced head · trained in {balanced_sec:.1f} s on {DEVICE}")
print(f"{'loss':<12}{'recall: hidden':>15}{'visible':>9}")
print(f"{'plain':<12}{naive['recall_hidden_fit']:>15.3f}{naive['recall_visible_fit']:>9.3f}")
print(f"{'balanced':<12}{bal_hidden:>15.3f}{bal_visible:>9.3f}")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Q11: what to look for</b></summary><ul>
<li>Reweighting usually recovers a lot of the hidden vessels. It does not fix the labels, it fixes the
  <em>proportion</em>: missing = background told the head that vessels are about half as common as they are, and
  the balanced loss stops caring about proportions.</li>
<li>It works here for two reasons that need not hold on your data: we hid segments <em>at random</em>, so the hidden
  ones look like the visible ones, and a linear head is too simple to memorise which segment was hidden.</li>
<li>If annotators systematically skip the thin capillaries, the head learns "thin = background", and no class
  weight undoes that.</li>
<li>Section 5 fixes the cause instead: tell the loss which pixels are unknown.</li>
</ul></details></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li><strong>A label file with only vessel and background</strong> → ask how it was made: an unpainted pixel may mean
  "unknown", not "background".</li>
<li><strong>A head trained with "missing = background"</strong> → expect a stingy model (precision stays high, recall on
  what was left out collapses), so also measure recall on a small, fully labelled set.</li>
<li><strong>A balanced loss that wins the recall back</strong> → do not trust it for systematic gaps: it works when the
  gaps are random, not when thin vessels are skipped.</li>
<li><strong>Three failures so far</strong> (SAM finds the eye, the head is limited by its 16 px patches, naive training
  believes the file) → none of these is a model-size problem we have tested; each needs different
  supervision. This section's recall on the hidden vessels is the baseline to beat; section 4 starts on how.</li>
</ul></div>"""))

## 4. The adaptation ladder

In section 3 the head believed a label file that said "background" wherever nobody
had looked. Now the ways out, and the first is a ladder: give the foundation model a little
more task information at each rung, and look at what changes. Higher is not automatically better, and
each rung fails in its own way, so we look at the masks, not only at the numbers.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>zero-shot → one-shot → scribbles: see how predictions change at each rung.</p></div>"""))

The next cell draws what the model receives at each rung, with its annotation cost: the input is the only
thing that changes.

In [ ]:
# @title What each rung receives { display-mode: "form" }
def effort_minutes(n_strokes: int = 0, n_full_masks: int = 0) -> float:
    """Annotation minutes under the dataset's stated assumptions (seconds per stroke, hours per full mask)."""
    cost = SPECIALISED["annotation_effort"]
    return n_strokes * cost["seconds_per_stroke"] / 60 + n_full_masks * 60 * float(
        np.mean(cost["hours_per_full_mask"])
    )


_shot0 = load_item(SPECIALISED, SPECIALISED["fit"][0])
_scribs = [load_item(SPECIALISED, m) for m in SPECIALISED["fit"][:N_TRAIN_SPEC]]
_n_str = sum(m.get("n_fg_strokes", 0) + m.get("n_bg_strokes", 0) for m in SPECIALISED["fit"][:N_TRAIN_SPEC])


def _cost(minutes: float) -> str:
    if minutes < 1:
        return f"{minutes * 60:.0f} s"
    return f"{minutes:.0f} min" if minutes < 90 else f"{minutes / 60:.0f} h"


_ex = _scribs[0]
_thick = {v: ndi.binary_dilation(_ex["scribbles"] == v, iterations=2) for v in (1, 2)}
_strokes = overlay(_ex["image"], _thick[2], THEME["background"], 0.9)
_strokes = overlay(_strokes, _thick[1], THEME["scribble"], 0.9)
_x0, _y0, _x1, _y1 = box_prompt(demo)
_box = mpl.patches.Rectangle((_x0, _y0), _x1 - _x0, _y1 - _y0, fill=False, lw=2, ec=THEME["scribble"])
_c0, _c1 = _cost(effort_minutes(n_strokes=1)), _cost(effort_minutes(n_full_masks=1))
_c2 = _cost(effort_minutes(n_strokes=_n_str))
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
axes[0].imshow(demo["image"])
axes[0].add_patch(_box)
axes[0].set_title(f"Rung 0 · zero-shot\n1 image + 1 box · ≈ {_c0}")
axes[1].imshow(overlay(_shot0["image"], _shot0["mask"], THEME["truth"]))
axes[1].set_title(f"Rung 1 · one-shot\n1 image + its complete mask · ≈ {_c1}")
axes[2].imshow(_strokes)
axes[2].set_title(f"Rung 2 · scribbles\n{len(_scribs)} images + {_n_str} strokes · ≈ {_c2}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: what each rung tells the model</b><ul>
<li><strong>Rung 0 · zero-shot</strong> (section 2): SAM gets the image and one box, and nothing about the task. It
  returns its best guess of "the object": on the retina, the eye or one thick vessel.</li>
<li><strong>Rung 1 · one-shot:</strong> one image with a complete vessel mask. On frozen DINOv3 features it becomes a
  <em>prototype</em>: the mean feature of its vessel patches and of its background patches. Every patch of a new
  image goes to the nearer one. This rung has <strong>two variants</strong>: the prototype mask itself, and the same
  similarity map choosing clicks for SAM. That is why the table in 4.4 has four rows for three rungs.</li>
<li><strong>Rung 2 · scribbles:</strong> strokes on some vessels and some background in a few images; everything else is
  <em>unlabelled</em>. The section-2 head is trained with a loss that counts only the labelled pixels (4.3).</li>
<li><strong>Cost:</strong> one complete mask takes hours, one scribble seconds.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Go deeper: where the rungs come from</b></summary><ul>
<li>Clicks from a similarity map: PerSAM, Zhang et al., ICLR 2024.</li>
<li>Partial cross-entropy on scribbles: Tang et al., ECCV 2018.</li>
<li>The recipe of the MIT "DINOv2 + scribbles" work, here on DINOv3.</li>
</ul></details></div>"""))

This helper cell defines the tools of the ladder: one colour per approach (kept for the rest of the
notebook), prototypes, clicks from a similarity map, error and difference maps, and a report of the
numbers we compare.

In [ ]:
# @title Ladder helpers: colours, prototypes, clicks, error and difference maps { display-mode: "form" }
APPROACH_COLOURS = {  # one colour per approach (L-003); later sections add theirs and keep these
    "zero_shot": "#E65100",
    "one_shot": "#C2185B",
    "one_shot_sam": "#F06292",
    "scribbles": "#3949AB",
}
APPROACH_NAMES = {
    "zero_shot": "zero-shot: SAM, one box",
    "one_shot": "one-shot: nearest prototype",
    "one_shot_sam": "one-shot: prototype → SAM clicks",
    "scribbles": "scribbles: partial cross-entropy",
}
APPROACH_SHORT = {
    "zero_shot": "zero-shot",
    "one_shot": "prototype",
    "one_shot_sam": "proto → SAM",
    "scribbles": "scribbles",
}


def patch_share(mask: np.ndarray, grid: tuple[int, int]) -> torch.Tensor:
    """Share of each feature patch covered by a boolean pixel mask: [h, w] floats in [0, 1]."""
    m = torch.from_numpy(np.asarray(mask, np.float32))[None, None]
    return F.interpolate(m, size=grid, mode="area")[0, 0].to(DEVICE)


def prototypes(feat: torch.Tensor, w_fg: torch.Tensor, w_bg: torch.Tensor) -> tuple[torch.Tensor, ...]:
    """Foreground and background prototypes: the weighted means of the unit-length patch features."""
    f = F.normalize(feat.flatten(1), dim=0)  # [D, h*w]
    return tuple(F.normalize(f @ w.flatten().to(f), dim=0) for w in (w_fg, w_bg))


@torch.no_grad()
def prototype_score(feat: torch.Tensor, protos: tuple[torch.Tensor, torch.Tensor]) -> torch.Tensor:
    """Per patch: cosine similarity to the foreground prototype minus to the background one ([h, w])."""
    return torch.einsum("dhw,d->hw", F.normalize(feat, dim=0), protos[0] - protos[1])


def score_to_mask(s: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """Upsample a patch score map to the image and mark the pixels nearer the foreground prototype."""
    up = F.interpolate(s[None, None], size=shape, mode="bilinear", align_corners=False)[0, 0]
    return (up > 0).cpu().numpy()


def heat_rgb(s: torch.Tensor, shape: tuple[int, int]) -> np.ndarray:
    """A score map as an RGB image: purple = like the foreground prototype, orange = like the background."""
    up = F.interpolate(s[None, None], size=shape, mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
    up = 0.5 + 0.5 * up / (np.abs(up).max() + 1e-9)
    return (mpl.colormaps["PuOr"](up)[..., :3] * 255).astype(np.uint8)


def top_points(s: torch.Tensor, keep: torch.Tensor, k: int, shape: tuple[int, int], largest: bool = True):
    """k (x, y) clicks at the centres of the patches that score highest (or lowest) inside `keep`, at least
    3 patches apart: how a similarity map becomes SAM prompts (the PerSAM idea)."""
    s = (s if largest else -s).clone()
    s[keep < 0.5] = -float("inf")
    (h, w), (H, W) = s.shape, shape
    pts = []
    for _ in range(k):
        idx = int(s.argmax())
        i, j = divmod(idx, w)
        if not torch.isfinite(s[i, j]):
            break
        pts.append([(j + 0.5) * W / w, (i + 0.5) * H / h])
        s[max(0, i - 2) : i + 3, max(0, j - 2) : j + 3] = -float("inf")
    return pts


def error_map(image: np.ndarray, pred: np.ndarray, item: dict) -> np.ndarray:
    """Inside the evaluation region: vessels found in violet, false alarms in orange, missed vessels in
    green (the ground truth the prediction did not cover)."""
    r = region(item)
    truth, p = item["mask"] & r, pred & r
    out = overlay(image, truth & ~p, THEME["truth"])
    out = overlay(out, p & truth, THEME["works"])
    return overlay(out, p & ~truth, THEME["fails"])


def diff_map(image: np.ndarray, prev: np.ndarray, new: np.ndarray, k_prev: str, k_new: str, item: dict):
    """Where two rungs disagree: pixels only the earlier rung marks in its colour, only the later rung in
    its colour, both in grey."""
    r = region(item)
    prev, new = prev & r, new & r
    out = overlay(image, prev & new, THEME["background"])
    out = overlay(out, prev & ~new, APPROACH_COLOURS[k_prev])
    return overlay(out, new & ~prev, APPROACH_COLOURS[k_new])


def rung_report(preds: list[np.ndarray]) -> dict:
    """Mean IoU, clDice, recall, precision and marked share of the evaluation region, over SPEC_EVAL."""
    out = mean_scores([score(p, it) for p, it in zip(preds, SPEC_EVAL, strict=True)])
    out["recall"] = part_recall(preds, SPEC_EVAL, "mask")
    pairs = zip(preds, SPEC_EVAL, strict=True)
    out["precision"] = float(np.mean([hidden_recall(it["mask"], p & region(it)) for p, it in pairs]))
    out["marked"] = float(np.mean([p[region(it)].mean() for p, it in zip(preds, SPEC_EVAL, strict=True)]))
    return out


def image_numbers(pred: np.ndarray, item: dict) -> dict:
    """IoU, clDice, recall, precision and marked share of one prediction on one image."""
    r = region(item)
    p = pred & r
    return score(pred, item) | {
        "recall": hidden_recall(pred, item["mask"] & r),
        "precision": hidden_recall(item["mask"], p),
        "marked": float(p[r].mean()),
    }


def show_extremes(preds: list[np.ndarray]) -> None:
    """Error maps of the worst, the median and the best held-out image (by IoU), with their numbers."""
    nums = [image_numbers(p, it) for p, it in zip(preds, SPEC_EVAL, strict=True)]
    order = np.argsort([n["iou"] for n in nums])
    picks = [("worst", order[0]), ("typical (median)", order[len(order) // 2]), ("best", order[-1])]
    show_masks(
        [
            (
                f"{label} · IoU {nums[k]['iou']:.2f}\nrecall {nums[k]['recall']:.2f} · "
                f"precision {nums[k]['precision']:.2f}\nmarked {nums[k]['marked']:.0%}",
                error_map(SPEC_EVAL[k]["image"], preds[k], SPEC_EVAL[k]),
            )
            for label, k in picks
        ],
        width=3.3,
    )

### 4.1 Rung 0: zero-shot, from section 2

Nothing new to run: the next cell measures section 2's SAM-with-one-box masks with the numbers every rung
is compared on.

- **IoU, clDice:** as before: overlap, and overlap along the centre line.
- **Recall:** of the true vessel pixels, the share the mask finds.
- **Precision:** of the pixels the mask marks, the share that really are vessel.
- **Marked:** the share of the fundus the mask covers.
- **Error-map colours:** violet = vessel found, orange = false alarm, green = vessel missed.

In [ ]:
# @title Score rung 0 and show its worst, typical and best image { display-mode: "form" }
k_demo = next(i for i, it in enumerate(SPEC_EVAL) if it is demo)  # the section-2 demo image
ladder_preds = {"zero_shot": sam_box_masks}  # section 2.3: SAM, one box, per evaluation image
ladder_report = {"zero_shot": rung_report(sam_box_masks)}
v = ladder_report["zero_shot"]
print(
    f"zero-shot (SAM, one box) on {len(SPEC_EVAL)} held-out images · IoU {v['iou']:.3f} · clDice "
    f"{v['cldice']:.3f} · recall {v['recall']:.3f} · precision {v['precision']:.3f} · marks {v['marked']:.0%}"
)
show_extremes(sam_box_masks)

### 4.2 Rung 1: one annotated image

Someone spends an afternoon tracing every vessel of **one** training image. That complete mask is all
this rung may use: the held-out ground truth stays for scoring. No training, just one mean per class:

1. Cut the annotated image into 16 px patches; DINOv3 gives one feature vector per patch.
2. Average the vessel patches into a vessel prototype, the background patches into a background prototype.
3. For a new image, score every patch: `cos(feature, vessel) − cos(feature, background)`.
4. Upsample the scores to pixels and mark those above 0.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>You saw SAM's box on the three images above (its "marked" share and its error maps). Before running the
prototype, write one guess for each:</p>
<ul>
<li>Will the prototype mark more of the fundus than SAM, or less?</li>
<li>Will it find the thin vessels, or only the thick ones?</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn: swap the annotated image</b><ul>
<li>Run the next cell: it builds the two prototypes from one annotated image, scores every evaluation image
  (seconds), and shows the similarity map and the mask on the demo image.</li>
<li>Then set <code>ONE_SHOT_IMAGE</code> to 1, 2, … and run it again, to answer Q12.</li>
</ul></div>"""))

In [ ]:
# Q12 — one-shot: the prototypes of one annotated image, applied to every held-out image
ONE_SHOT_IMAGE = 0  # which training image is the annotated one; try 1, 2, ...
if not isinstance(ONE_SHOT_IMAGE, int) or not 0 <= ONE_SHOT_IMAGE < len(SPECIALISED["fit"]):
    raise SystemExit(f"ONE_SHOT_IMAGE must be a whole number from 0 to {len(SPECIALISED['fit']) - 1}.")
side = SPECIALISED["feat_side"]
shot = load_item(SPECIALISED, SPECIALISED["fit"][ONE_SHOT_IMAGE])
shot_feat = features_for(SPECIALISED, shot, side)
grid, r = tuple(shot_feat.shape[1:]), region(shot)
# the rung's supervision: one complete mask of one *training* image (L-005)
protos = prototypes(shot_feat, patch_share(shot["mask"] & r, grid), patch_share(~shot["mask"] & r, grid))
t0 = time.monotonic()
proto_scores = [prototype_score(features_for(SPECIALISED, it, side), protos) for it in SPEC_EVAL]
ladder_preds["one_shot"] = [
    score_to_mask(s, it["mask"].shape) for s, it in zip(proto_scores, SPEC_EVAL, strict=True)
]
ladder_report["one_shot"] = rung_report(ladder_preds["one_shot"])
proto_sec = round(time.monotonic() - t0, 1)
v = ladder_report["one_shot"]
print(
    f"one-shot prototype from {shot['name']} · IoU {v['iou']:.3f} · clDice {v['cldice']:.3f} · recall "
    f"{v['recall']:.3f} · precision {v['precision']:.3f} · marks {v['marked']:.0%} · "
    f"{proto_sec} s on {DEVICE}"
)
print(f"for scale: the section-2 head on five complete masks · IoU {RESULTS['specialised_head_iou']:.3f}")
show_masks(
    [
        (f"the one annotated image · {shot['name']}", overlay(shot["image"], shot["mask"], THEME["truth"])),
        ("similarity: vessel − background", heat_rgb(proto_scores[k_demo], demo["mask"].shape)),
        (
            f"one-shot prototype · IoU {score(ladder_preds['one_shot'][k_demo], demo)['iou']:.2f}",
            overlay(demo["image"], ladder_preds["one_shot"][k_demo], THEME["fails"]),
        ),
        ("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"])),
    ],
    width=2.8,
)

**Q12.** 🟡 Look at the similarity map and the IoU you got:

- Which parts of the held-out image look most like the annotated vessels (purple), and which like the
  background (orange)?
- Pick another annotated image: does the held-out IoU move much?
- What does that say about relying on a single example?

<details><summary>What to look for</summary>

The purple follows the thick vessels and the dark, textured parts of the fundus, and the mask is as wide
as a 16 px patch: the prototype has the same resolution limit as the section-2 head. Compare the two IoUs
printed above: one example and a mean, against five complete masks and a trained head. More labels do not
automatically give a better mask.

If the IoU moves when you swap the annotated image, this rung depends on *which* example you happened to
annotate. With one image, a lucky or unlucky choice is the whole training set.

</details>

### 🔴 Optional: the same map, pointing SAM

The same similarity map can also point SAM at the right places (rung 1, second variant). The cell below
runs under "Run all" even if you skip the reading: its row appears in the table of 4.4. On every
evaluation image:

1. The five patches most like a vessel become positive clicks.
2. The five most like background become negative clicks.
3. SAM turns the ten clicks into a mask (one SAM image encoding per image: seconds, also on a CPU).

In [ ]:
# @title Prototype → SAM clicks on every evaluation image { display-mode: "form" }
N_PROTO_CLICKS = 5  # positive clicks, and as many negative ones, per image
t0 = time.monotonic()
ladder_preds["one_shot_sam"], proto_clicks = [], []
for it, s in zip(SPEC_EVAL, proto_scores, strict=True):
    keep = patch_share(region(it), tuple(s.shape))
    pos = top_points(s, keep, N_PROTO_CLICKS, it["mask"].shape, largest=True)
    neg = top_points(s, keep, N_PROTO_CLICKS, it["mask"].shape, largest=False)
    prob = sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg))
    ladder_preds["one_shot_sam"].append(prob >= 0.5)
    proto_clicks.append((pos, neg))
ladder_report["one_shot_sam"] = rung_report(ladder_preds["one_shot_sam"])
proto_sam_sec = round(time.monotonic() - t0, 1)
v = ladder_report["one_shot_sam"]
print(
    f"one-shot prototype → SAM clicks · IoU {v['iou']:.3f} · clDice {v['cldice']:.3f} · recall "
    f"{v['recall']:.3f} · precision {v['precision']:.3f} · marks {v['marked']:.0%} · {proto_sam_sec} s"
)
pos, neg = proto_clicks[k_demo]
demo_sam = overlay(demo["image"], ladder_preds["one_shot_sam"][k_demo], THEME["fails"])
show_clicks(
    [
        ("clicks chosen as vessel", demo["image"], pos),
        ("clicks chosen as background", demo["image"], neg),
        ("SAM from those clicks", demo_sam, pos),
        ("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]), None),
    ],
    width=2.8,
)

**Q13.** 🔴 Same annotated image, same similarity map, two ways to use it. Compare the prototype mask
with the SAM mask:

- Which one draws wider vessels, and which loses thin branches?
- Did the "vessel" clicks land on vessels?

<details><summary>What to look for</summary>

The prototype labels *every* patch, so it covers the whole tree at patch resolution. SAM only sees ten
clicks and does what section 2 showed: it looks for the object that contains the positive ones and
avoids the negative ones. Its edges are sharper, but it can lose whole branches that no click touched,
or grow into a region around a thick vessel.

If a "vessel" click landed on the optic disc or a dark spot, the mask follows it: SAM trusts its prompts.
Two uses of the same information can fail in opposite directions.

</details>

### ☕ Before the break

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Question for the coffee break</b><p>No code, just think about it over coffee.</p>
<p><strong>Q14.</strong> 🟡 You have 50 fundus images and a 4-hour annotation budget. Perfect labels cost 3 to 5 hours per
image, and the cheap ones lie: the label file says "background" wherever nobody looked. Without buying more
labels, what could you tell the training about a pixel nobody labelled? Think of at least two options before
4.3.</p>
<details>
<summary>A hint, for after the break</summary>
<p>Three families of answers come back in block 2:</p>
<ul>
<li>Say "unknown" instead of "background", and let the loss skip those pixels (sections 4 and 5).</li>
<li>Guess the missing labels with another model, for example SAM prompted from the labels you have, and train
  on the guesses (section 6).</li>
<li>Skip training and give the cheap labels to the foundation model directly as prompts (section 7).</li>
</ul>
</details></div>"""))

### 4.3 Rung 2: scribbles

Welcome back. The annotator from section 3 again, with a new instruction: "don't trace everything;
draw a few strokes along some vessels and a few on clear background, in each of five images, then stop".
Every other pixel is *unlabelled*, and this time the stroke map says so, so the loss can skip it.

Three label encodings meet here, and the number 0 means something different in each:

| | 0 | 1 | 2 | −1 |
|---|---|---|---|---|
| Section-3 file | background (or just unpainted) | vessel | – | – |
| Stroke map (this rung) | unlabelled | vessel stroke | background stroke | – |
| Training targets | background | vessel | – | ignore this pixel |

The next cell shows the strokes on one training image (vessel in teal, background in grey, thickened for
the figure only) and, in the last panel, what the loss will actually see.

In [ ]:
# @title Show the scribbles of one training image { display-mode: "form" }
scrib_train = [load_item(SPECIALISED, it) for it in SPECIALISED["fit"][:N_TRAIN_SPEC]]
labelled = float(np.mean([(it["scribbles"] > 0)[region(it)].mean() for it in scrib_train]))
scrib_meta = SPECIALISED["fit"][:N_TRAIN_SPEC]
n_strokes = sum(m.get("n_fg_strokes", 0) + m.get("n_bg_strokes", 0) for m in scrib_meta)
print(
    f"{len(scrib_train)} training images · {n_strokes} strokes · {labelled:.2%} of the fundus pixels "
    f"labelled · ≈ {effort_minutes(n_strokes=n_strokes):.1f} min at "
    f"{SPECIALISED['annotation_effort']['seconds_per_stroke']} s per stroke"
)
ex = scrib_train[0]
thick = {v: ndi.binary_dilation(ex["scribbles"] == v, iterations=2) for v in (1, 2)}
strokes = overlay(overlay(ex["image"], thick[2], THEME["background"], 0.9), thick[1], THEME["scribble"], 0.9)
counted = ndi.binary_dilation(ex["scribbles"] > 0, iterations=2)  # thickened for display only
loss_sees = np.where(counted[..., None], strokes, (ex["image"] * 0.25).astype(np.uint8))
show_masks(
    [
        (ex["name"], ex["image"]),
        ("scribbles: vessel · background", strokes),
        ("what the loss sees (dark = ignored)", loss_sees),
        ("ground truth · evaluation only", overlay(ex["image"], ex["mask"], THEME["truth"])),
    ],
    width=2.8,
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: partial cross-entropy</b><p>The loss of section 2 adds a cross-entropy term for every pixel. The partial version adds it only over
the set S of pixels a person labelled:</p>
<p><code>L = −(1/|S|) · Σ_{i∈S} [ y_i·log p_i + (1−y_i)·log(1−p_i) ]</code></p>
<ul>
<li>A vessel stroke says y = 1, a background stroke says y = 0.</li>
<li>An unlabelled pixel is not in S: it adds nothing. In code it is a target of <code>-1</code>, which <code>train_head</code>
  already skips (it has skipped the fundus border that way since section 2).</li>
<li>In section 3 the unpainted pixels were targets of <code>0</code>: a thin vessel nobody painted counted as a
  "background" error.</li>
</ul></div>"""))

This cell holds our version of the function you can write below. Open it only if you get stuck.

In [ ]:
# @title Reference solution (open only if stuck) { display-mode: "form" }
def _ref_scribble_targets(scribbles: np.ndarray, region: np.ndarray, bg_beyond_px: float | None = None):
    """Reference: strokes as given, unlabelled ignored (or background when far from every vessel stroke)."""
    targets = np.full(scribbles.shape, -1, np.int8)
    if bg_beyond_px is not None:
        targets[ndi.distance_transform_edt(scribbles != 1) > bg_beyond_px] = 0
    targets[scribbles == 2] = 0
    targets[scribbles == 1] = 1
    targets[~np.asarray(region, bool)] = -1
    return targets

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 3 min): choose which pixels to ignore</b><p>Fill <code>my_scribble_targets</code>: it turns a stroke map into training targets (rows 2 and 3 of the table above).</p>
<ul>
<li>In: 0 unlabelled, 1 vessel stroke, 2 background stroke. Out: 1, 0, or −1 for "ignore".</li>
<li>Pixels outside <code>region</code> are always −1.</li>
<li>Start with the default, <code>bg_beyond_px=None</code>: every unlabelled pixel is ignored.</li>
<li><strong>Ignore the number setting of <code>bg_beyond_px</code> until 4.6.</strong> There, unlabelled pixels farther than that
  from every vessel stroke count as background; <code>ndi.distance_transform_edt(scribbles != 1)</code> gives each
  pixel's distance to the nearest vessel stroke.</li>
</ul>
<p>The cell checks your version on a 6 × 6 play map with both settings and continues with ours if you skip
it or leave the number setting out.</p></div>"""))

In [ ]:
# ✍️ YOUR CODE
def my_scribble_targets(scribbles: np.ndarray, region: np.ndarray, bg_beyond_px: float | None = None):
    """Targets (int8) from a scribble map: 1 vessel stroke, 0 background stroke, -1 ignored. Outside
    `region`: -1. If bg_beyond_px is a number, unlabelled pixels farther than bg_beyond_px from every vessel
    stroke become 0 (background); the others stay -1.
    e.g. on a 3×4 crop, same shape as `scribbles`: [[-1, -1, 1, -1], [0, -1, 1, -1], [0, 0, -1, -1]]"""
    ...


_play_scrib = np.zeros((6, 6), np.uint8)
_play_scrib[1, :4] = 1  # a vessel stroke
_play_scrib[4, 1:3] = 2  # a background stroke
_play_scrib[5, 5] = 2  # a background stroke outside the region
_play_region = np.ones((6, 6), bool)
_play_region[:, 5] = False  # the last column is outside the evaluation region
scribble_targets = use(
    my_scribble_targets,
    reference=_ref_scribble_targets,
    check=lambda f: [np.asarray(f(_play_scrib, _play_region, d)).astype(int) for d in (None, 2)],
    name="scribble_targets",
)
print("play example · targets with bg_beyond_px=2 (−1 = ignored):")
print(scribble_targets(_play_scrib, _play_region, 2))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The head will see labels on fewer than two pixels in a hundred (see the printout above), and no vessel
will ever be called "background". Against the one complete mask of rung 1, write two guesses:</p>
<ul>
<li>Will it find more of the held-out vessels (recall), or fewer?</li>
<li>Will it mark more of the fundus, or less?</li>
</ul></div>"""))

The next cell trains the section-2 head on the scribbles of the five training images, every unlabelled
pixel ignored; with `TRAIN_YOURSELF = False` it loads our checkpoint, trained the same way on all fit
images.

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "scribbles" head instead of training
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

scrib_feats = [features_for(SPECIALISED, it, side) for it in scrib_train]
# the rung's supervision: strokes only; unlabelled pixels and the border are ignored (-1), never "background"
scrib_labels = [scribble_targets(it["scribbles"], region(it)) for it in scrib_train]
if TRAIN_YOURSELF:
    scrib_head, scrib_sec = train_head(scrib_feats, scrib_labels, HEAD_STEPS)
    src = f"trained here on {len(scrib_train)} images, {HEAD_STEPS} steps, partial cross-entropy"
else:
    scrib_head, entry = load_head(SPECIALISED, "scribbles", side)
    scrib_sec = entry["train_sec"]
    src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
print(f"scribble head @ {side} px · {src} · {scrib_sec:.1f} s on {DEVICE}")
if scrib_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

This cell scores the scribble head on the evaluation images with the same five numbers as the other rungs.

In [ ]:
# @title Score the scribble head on the evaluation images { display-mode: "form" }
t0 = time.monotonic()
ladder_preds["scribbles"] = [
    predict_head(scrib_head, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
]
ladder_report["scribbles"] = rung_report(ladder_preds["scribbles"])
scrib_infer = round(time.monotonic() - t0, 1)
v = ladder_report["scribbles"]
print(
    f"scribble head · IoU {v['iou']:.3f} · clDice {v['cldice']:.3f} · recall {v['recall']:.3f} · precision "
    f"{v['precision']:.3f} · marks {v['marked']:.0%} · {scrib_infer} s on {DEVICE}"
)

### 4.4 The ladder, side by side

The next cell puts the four rows (three rungs: rung 1 has two variants) side by side: a table with the
five numbers and the annotation each needed, a bar chart, and the demo image's error maps (violet =
vessel found, orange = false alarm, green = vessel missed).

In [ ]:
# @title Table, bar chart and error maps of all rungs { display-mode: "form" }
effort = {
    "zero_shot": effort_minutes(n_strokes=len(SPEC_EVAL)),  # one box per image you segment
    "one_shot": effort_minutes(n_full_masks=1),
    "one_shot_sam": effort_minutes(n_full_masks=1),
    "scribbles": effort_minutes(n_strokes=n_strokes),
}
effort_text = {
    "zero_shot": f"1 box per image ({len(SPEC_EVAL)} here)",
    "one_shot": "1 complete mask",
    "one_shot_sam": "1 complete mask",
    "scribbles": f"{n_strokes} strokes, {len(scrib_train)} images",
}
RESULTS["ladder"] = {k: ladder_report[k]["iou"] for k in ("zero_shot", "one_shot", "scribbles")}  # SPEC §9
RESULTS["ladder_detail"] = ladder_report
RESULTS["ladder_effort_min"] = {k: round(v, 2) for k, v in effort.items()}
RESULTS["scribbles_head"] = ladder_report["scribbles"] | {
    "long_side": side,
    "n_train": len(scrib_train),
    "n_strokes": n_strokes,
    "labelled_fraction": labelled,
    "train_sec": round(float(scrib_sec), 2),
    "infer_sec": scrib_infer,
}
RESULTS["one_shot"] = ladder_report["one_shot"] | {"image": shot["name"], "infer_sec": proto_sec}

COLUMNS = f"{'IoU':>6}{'clDice':>8}{'recall':>8}{'precis.':>8}{'marked':>8}"
print(f"{'approach':<36}{COLUMNS}   annotation (≈ minutes)")
for k, v in ladder_report.items():
    print(
        f"{APPROACH_NAMES[k]:<36}{v['iou']:>6.3f}{v['cldice']:>8.3f}{v['recall']:>8.3f}{v['precision']:>8.3f}"
        f"{v['marked']:>8.0%}   {effort_text[k]} (≈ {effort[k]:.1f})"
    )
print(f"{'everything is foreground':<36}{RESULTS['specialised_everything_iou']:>6.3f}")

metrics = ["iou", "cldice", "recall", "precision"]
x = np.arange(len(metrics))
width = 0.8 / len(ladder_report)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for n, (k, v) in enumerate(ladder_report.items()):
    shift = (n - (len(ladder_report) - 1) / 2) * width
    ax.bar(x + shift, [v[m] for m in metrics], width, color=APPROACH_COLOURS[k], label=APPROACH_NAMES[k])
ax.axhline(
    RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--", label='IoU of "everything"'
)
ax.set_xticks(x, ["IoU", "clDice", "recall", "precision"])
ax.set_ylim(0, 1)
ax.set_title("The adaptation ladder: different information, different failures")
ax.legend(fontsize=7, frameon=False, loc="upper left", ncol=2)
plt.tight_layout()
plt.show()

show_masks(
    [
        (
            f"{APPROACH_NAMES[k].split(':')[0]} · IoU {score(ladder_preds[k][k_demo], demo)['iou']:.2f}",
            error_map(demo["image"], ladder_preds[k][k_demo], demo),
        )
        for k in ladder_preds
    ]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]))],
    width=2.6,
)

**Q15.** 🟡 Read the table and the error maps rung by rung:

- Which rung has the lowest recall (misses the most, green)? Which has the lowest precision (invents the
  most, orange)? Is it the rung with the lowest IoU?
- You need a screening tool that must not miss a vessel, and a tool whose every mark must be right. Do you
  pick the same rung for both?

<details><summary>What to look for</summary>

The rungs do not line up on one "quality" axis. SAM's box tends to mark a region or the whole eye: high
recall, many false alarms, an IoU near the grey line. The prototype labels by patch, so it draws vessels
as wide as a square and confuses dark, textured background with vessel. SAM from prototype clicks is
sharper but only covers what its clicks touched.

The scribble head saw the fewest labelled pixels, yet from several images, and it was never told a vessel
was background. Its errors are not the section-3 errors: compare its recall with the naive head's.

Which rung is "best" depends on whether missing or inventing a vessel costs you more, and on the minutes
in the last column. Your order may differ from your neighbour's: that is why we read the maps.

</details>

### 🔴 Optional: what changes from one rung to the next

Several of those error maps look alike, so the next cell shows only what changes from one rung to the next
on the demo image (grey = both mark it, a colour = only that one).

In [ ]:
# @title Difference maps: what changes from one rung to the next { display-mode: "form" }
keys = list(ladder_preds)
fig, axes = plt.subplots(1, len(keys) - 1, figsize=(3.0 * (len(keys) - 1), 3.6))
for ax, a, b in zip(axes, keys[:-1], keys[1:], strict=True):
    pa, pb = ladder_preds[a][k_demo], ladder_preds[b][k_demo]
    reg = region(demo)
    differs = float(((pa ^ pb) & reg).sum() / reg.sum())
    ax.imshow(diff_map(demo["image"], pa, pb, a, b, demo))
    ax.set_title(f"{APPROACH_SHORT[a]} → {APPROACH_SHORT[b]}\n{differs:.0%} of fundus differs", fontsize=9)
    ax.axis("off")
    ax.legend(
        handles=[
            mpl.patches.Patch(color=APPROACH_COLOURS[a], label=f"only {APPROACH_SHORT[a]}"),
            mpl.patches.Patch(color=APPROACH_COLOURS[b], label=f"only {APPROACH_SHORT[b]}"),
            mpl.patches.Patch(color=THEME["background"], label="both"),
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.0),
        fontsize=7,
        frameon=False,
        ncol=1,
    )
plt.tight_layout()
plt.show()

### 4.5 A tempting shortcut: which unlabelled pixels to ignore

Ignoring every unlabelled pixel is safe, and wasteful: over 98 % of the fundus says nothing. A tempting
shortcut is "far from every vessel stroke, it's surely background". That is the second setting of your
`scribble_targets`.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: &quot;surely background&quot; is a guess</b><ol>
<li>The strokes cover only some vessel segments. Say vessel B has no stroke.</li>
<li>B is more than 40 px from every stroke, so the rule calls it "background".</li>
<li>The loss now pushes the head to call B-like pixels background: the section-3 mistake, back through the
   side door.</li>
</ol></div>"""))

### 4.6 🔴 Optional: choose a distance

Want to see what the shortcut costs? The next cell tries it on the real scribbles.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn: choose a distance</b><ul>
<li>Set <code>BG_BEYOND_PX</code> (try 20, 80, <code>None</code> = ignore every unlabelled pixel) and run the next cell.</li>
<li>It turns the scribbles into targets with your distance, trains the same head on them (seconds), and
  prints its numbers next to the scribble head of 4.3, which ignored every unlabelled pixel.</li>
<li>It also measures how much true vessel your rule mislabels (the hidden ground truth is used for that
  diagnosis only, never for training).</li>
<li>Try a few distances before you answer Q16.</li>
</ul></div>"""))

In [ ]:
# Q16 — scribbles plus "far from every vessel stroke = background"
BG_BEYOND_PX = 40  # pixels, on the 1024 px crops; try 20, 80, None (= ignore every unlabelled pixel)
if BG_BEYOND_PX is not None and (not isinstance(BG_BEYOND_PX, (int, float)) or BG_BEYOND_PX <= 0):
    raise SystemExit("BG_BEYOND_PX must be a positive number of pixels, or None.")
bg_px = BG_BEYOND_PX / 4 if (SMOKE and BG_BEYOND_PX) else BG_BEYOND_PX  # smoke thumbnails are 4× smaller
far_labels = [scribble_targets(it["scribbles"], region(it), bg_px) for it in scrib_train]
# diagnosis only: how much of the true vessel area the rule called background (never used for training)
far_pairs = list(zip(far_labels, scrib_train, strict=True))
wrong = float(np.mean([((t == 0) & it["mask"]).sum() / max(it["mask"].sum(), 1) for t, it in far_pairs]))
far_labelled = float(np.mean([(t >= 0)[region(it)].mean() for t, it in far_pairs]))
far_head, far_sec = train_head(scrib_feats, far_labels, HEAD_STEPS)
far_preds = [
    predict_head(far_head, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
]
far_report = rung_report(far_preds)
RESULTS["scribbles_bg_beyond"] = far_report | {"bg_beyond_px": BG_BEYOND_PX, "vessel_called_bg": wrong}
print(
    f"rule: unlabelled pixels > {BG_BEYOND_PX} px from every vessel stroke = background · labelled "
    f"{far_labelled:.0%} of the fundus · {wrong:.1%} of the true vessel pixels called background · "
    f"trained in {far_sec:.1f} s"
)
print(f"{'scribble head':<28}{COLUMNS}")
for name, v in {"ignore every unlabelled": ladder_report["scribbles"], "your distance": far_report}.items():
    nums = f"{v['iou']:>6.3f}{v['cldice']:>8.3f}{v['recall']:>8.3f}{v['precision']:>8.3f}"
    print(f"{name:<28}{nums}{v['marked']:>8.0%}")
# what the rule did on one training image: its targets, and the true vessels it called background
tgt, ex17 = far_labels[0], scrib_train[0]
called_bg = (tgt == 0) & ex17["mask"]
show_masks(
    [
        (
            "targets: vessel · background · ignored",
            overlay(
                overlay(ex17["image"], tgt == 0, THEME["background"], 0.5),
                ndi.binary_dilation(tgt == 1, iterations=2),
                THEME["scribble"],
                0.9,
            ),
        ),
        (
            f"true vessels called background (orange): {called_bg.sum() / max(ex17['mask'].sum(), 1):.1%}",
            two_colour(ex17["image"], ex17["mask"] & ~called_bg, called_bg),
        ),
    ],
    width=3.8,
)

**Q16.** 🔴 Compare your distance with the scribble head of 4.3:

- How many true vessel pixels did your rule call background?
- Did recall, precision and IoU on the held-out images move, and in which direction?
- Is there a distance that helps without hurting?

<details><summary>What to look for</summary>

A large distance adds a lot of background that really is background, and a little that is not; a small
distance adds more of both. More "background" targets push the head to mark less, so watch recall fall
(the direction of section 3) and check whether precision rose enough to pay for it. The line "vessel
pixels called background" tells you the price.

The lesson is not a number: every rule that fills in unlabelled pixels is an assumption about where the
annotator did not look, and it is only as good as that assumption. Section 5 makes "unknown" explicit in
the loss; section 6 fills unlabelled pixels with a second model's guesses instead of a distance.

</details>

Want to try the one-shot idea on your own photo? Section 9 runs the whole ladder on your own images.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li><strong>Zero-shot IoU looks fine but the mask covers a whole region</strong> → read the "marked" share and the error
  map, not only IoU.</li>
<li><strong>You have one complete mask</strong> → expect patch-wide vessels, and swap the annotated image to see how
  much the result depends on your choice.</li>
<li><strong>You have cheap scribbles</strong> → train with a loss that ignores unlabelled pixels, and measure what any
  "far from a stroke = background" rule mislabels before you use it (4.5).</li>
<li><strong>Two rungs score alike</strong> → compare their error maps and the minutes of annotation: they differ more
  in how they fail than in one number. Keep the rung colours; they return in section 8.</li>
</ul></div>"""))

## 5. Path 1 · Learn from less

Same trick as 4.3 ("unlabelled = ignored"), now measured on the half-painted label file of section 3. The
naive head failed because the loss got that file as it was, and the file calls every unpainted pixel
"background". Path 1 keeps the same file, features and head, and changes only what the loss is told about
the unpainted pixels:

- **Masked:** "unknown". The loss skips them; the only background it hears about is a few strokes on
  clear background.
- **🔴 PU** (positive-unlabelled: you know some positives, everything else is unknown): "unlabelled, and
  about this share of them is vessel". The loss uses every pixel and corrects for the vessels hiding
  among them.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>treat missing labels as unknown; write the masked loss, then 🔴 a PU-inspired loss.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: the masked loss</b><p>A <strong>masked loss</strong> is the partial cross-entropy of 4.3: only the pixels S that carry a label count, and an
ignored pixel adds nothing and sends no gradient. Here you write it yourself.</p>
<ul>
<li><strong>Example, a thin vessel nobody painted.</strong> Naive (section 3): its pixels are in S with y = 0, so the
  loss punishes the head for calling them vessel. Masked: they are not in S, so nothing punishes it.</li>
<li><strong>Catch:</strong> if S holds almost only vessel, "everything is vessel" scores a perfect zero. Some known
  background must stay in S.</li>
</ul></div>"""))

The next cell defines a training loop that takes the loss as an argument (same linear head, features and
steps as every head so far), a report of the section-3 and section-4 numbers, a view of what the loss
hears, and one colour for each head of this path.

In [ ]:
# @title Path 1 helpers: train with any loss, score a head, view the loss mask { display-mode: "form" }
APPROACH_COLOURS |= {
    "naive": "#BF360C",
    "naive_balanced": "#FF8A65",
    "masked": "#F9A825",
    "pu": "#6D4C41",
}
APPROACH_NAMES |= {
    "naive": "naive: missing = background",
    "naive_balanced": "naive, balanced loss (3.4)",
    "masked": "masked: missing = unknown",
    "pu": "PU: missing = unlabelled",
}


def train_head_with(
    feats: list[torch.Tensor], labels: list[np.ndarray], loss_fn, steps: int, lr: float = 0.05
):
    """The linear head of `train_head`, trained with `loss_fn(logits, labels, ignore_mask)` on every pixel of
    every training image at once. Labels per pixel: 1 vessel, 0 background (or unlabelled), -1 ignored.
    Returns the head's tensors and the training seconds."""
    torch.manual_seed(SEED)
    allf = torch.cat([f.flatten(1) for f in feats], 1)
    mu, sd = allf.mean(1), allf.std(1) + 1e-6
    feats = [(f - mu.view(-1, 1, 1)) / sd.view(-1, 1, 1) for f in feats]
    lab = torch.cat([torch.from_numpy(np.asarray(x, dtype=np.int8)).flatten() for x in labels]).to(DEVICE)
    target, ignore = (lab == 1).float(), lab < 0
    shapes = [np.asarray(x).shape for x in labels]
    head = torch.nn.Conv2d(feats[0].shape[0], 1, 1).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    t0 = time.monotonic()
    for _ in range(steps):
        opt.zero_grad()
        logits = torch.cat(
            [
                F.interpolate(head(f[None]), size=s, mode="bilinear", align_corners=False).flatten()
                for f, s in zip(feats, shapes, strict=True)
            ]
        )
        loss_fn(logits, target, ignore).backward()
        opt.step()
    tensors = {
        "weight": head.weight.detach().cpu(),
        "bias": head.bias.detach().cpu(),
        "feat_mean": mu.cpu(),
        "feat_std": sd.cpu(),
    }
    return tensors, time.monotonic() - t0


def path_report(tensors: dict) -> dict:
    """Recall on the hidden and visible vessels of the training images (section 3) and the held-out
    numbers of section 4 (IoU, clDice, recall, precision, marked share), plus the predictions."""
    fit_preds = [
        predict_head(tensors, f, it["mask"].shape) for f, it in zip(miss_feats, miss_train, strict=True)
    ]
    held_preds = [
        predict_head(tensors, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
    ]
    out = {
        "recall_hidden_fit": part_recall(fit_preds, miss_train, "hidden"),
        "recall_visible_fit": part_recall(fit_preds, miss_train, "visible"),
    }
    return out | rung_report(held_preds) | {"fit_preds": fit_preds, "held_preds": held_preds}


def print_reports(reports: dict) -> None:
    """One line per head: recall on the hidden / visible training vessels, then the held-out numbers."""
    print(f"{'head':<34}{'recall: hidden':>15}{'visible':>9}   held-out IoU  recall  precision  marked")
    for name, r in reports.items():
        print(
            f"{name:<34}{r['recall_hidden_fit']:>15.3f}{r['recall_visible_fit']:>9.3f}   {r['iou']:>12.3f}"
            f"{r['recall']:>8.3f}{r['precision']:>11.3f}{r['marked']:>8.0%}"
        )


def public(r: dict) -> dict:
    """A report without its prediction arrays, for RESULTS."""
    return {k: v for k, v in r.items() if not k.endswith("_preds")}


def loss_view(item: dict, labels: np.ndarray, show_bg: bool = True) -> np.ndarray:
    """What the loss hears on one image: painted vessel in teal, background targets in grey, pixels the loss
    ignores (label -1) dark. Sparse labels are thickened for display only."""
    known = labels >= 0
    if known.mean() < 0.2:
        known = ndi.binary_dilation(known, iterations=4)
    out = np.where(known[..., None], item["image"], (item["image"] * 0.25).astype(np.uint8))
    if show_bg:
        bg = labels == 0
        if bg.mean() < 0.2:
            bg = ndi.binary_dilation(bg, iterations=4)
        out = overlay(out, bg, THEME["background"])
    return overlay(out, labels == 1, THEME["scribble"])

### 5.1 Three ways to read the same label file

The next cell builds what each recipe tells the loss, for the five training images of section 3. The
visible vessels are the same every time; the rest differs:

- **Naive:** every unpainted pixel is a background target.
- **Masked:** unpainted pixels are ignored, except 3–5 background strokes per image (the annotator also
  drew a few strokes on clear background, as in section 4): the loss needs some known background.
- **PU:** every unpainted pixel counts, as "unlabelled".

No ground truth goes in: only the visible vessels, the strokes and the fundus border.

Colour key for the figure: teal = painted vessel, grey = a background target, dark = ignored by the loss,
bright with no colour = counted as "unlabelled" (PU only).

In [ ]:
# @title What each recipe tells the loss { display-mode: "form" }
bg_strokes = [(it["scribbles"] == 2) & region(it) & ~it["visible"] for it in miss_train]
# masked: vessel where painted, background only on the strokes, everything else unknown (-1)
masked_labels = [
    np.where(it["visible"] & region(it), 1, np.where(bg, 0, -1)).astype(np.int8)
    for it, bg in zip(miss_train, bg_strokes, strict=True)
]
# PU: vessel where painted, "unlabelled" (0) everywhere else inside the fundus
pu_labels = [np.where(region(it), it["visible"], -1).astype(np.int8) for it in miss_train]
known = {
    "naive": np.mean([(lab >= 0).mean() for lab in naive_labels]),
    "masked": np.mean([(lab >= 0).mean() for lab in masked_labels]),
}
n_bg_strokes = sum(m.get("n_bg_strokes", 0) for m in SPECIALISED["fit"][: len(miss_train)])
print(
    f"pixels the loss hears about · naive {known['naive']:.1%} (all called background unless painted) · "
    f"masked {known['masked']:.1%} ({n_bg_strokes} background strokes over {len(miss_train)} images) · "
    "PU: all of them, as vessel or unlabelled"
)

ex, k_ex = miss_train[0], 0
show_masks(
    [
        (ex["name"], ex["image"]),
        ("naive: vessel / background", loss_view(ex, naive_labels[k_ex])),
        ("masked: vessel / strokes / ignored", loss_view(ex, masked_labels[k_ex])),
        ("PU: vessel / unlabelled", loss_view(ex, pu_labels[k_ex], show_bg=False)),
    ],
    width=2.8,
)

### 5.2 Write the masked loss

This cell holds our version of the function you can write below. Open it only if you get stuck.

In [ ]:
# @title Reference solution (open only if stuck) { display-mode: "form" }
def _ref_masked_loss(logits: torch.Tensor, labels: torch.Tensor, ignore_mask: torch.Tensor) -> torch.Tensor:
    """Reference: per-pixel BCE, then a mean per known class (an empty class counts as 0)."""
    bce = F.binary_cross_entropy_with_logits(logits, labels.float(), reduction="none")
    pos, neg = ~ignore_mask & (labels == 1), ~ignore_mask & (labels == 0)
    return 0.5 * bce[pos].sum() / pos.sum().clamp(min=1) + 0.5 * bce[neg].sum() / neg.sum().clamp(min=1)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 4 min)</b><p>Fill <code>masked_loss</code>. In: <code>logits</code> (float tensor, any shape), <code>labels</code> (same shape, 1.0 = vessel,
0.0 = background) and <code>ignore_mask</code> (same shape, bool, True = unknown). Out: one scalar tensor. It is the
balanced loss of every head so far, with one change: ignored pixels count for nothing.</p>
<ol>
<li>Take the binary cross-entropy of each pixel: <code>F.binary_cross_entropy_with_logits(..., reduction="none")</code>.</li>
<li>Average it over the known vessel pixels and over the known background pixels, separately.</li>
<li>Return the mean of the two averages.</li>
</ol>
<p><code>L = ½ · mean_known vessel(BCE) + ½ · mean_known background(BCE)</code></p>
<p>The cell checks your version on a 2 × 3 play example and continues with ours if you skip it.</p></div>"""))

In [ ]:
# ✍️ YOUR CODE
def masked_loss(logits: torch.Tensor, labels: torch.Tensor, ignore_mask: torch.Tensor) -> torch.Tensor:
    """Balanced binary cross-entropy over the pixels where `ignore_mask` is False: the mean loss on the
    known vessel pixels and the mean loss on the known background pixels, averaged.
    e.g. tensor(0.4137), one number"""
    ...


_play_logits = torch.tensor([[2.0, -1.0, 0.5], [-2.0, 3.0, 0.0]])
_play_labels = torch.tensor([[1.0, 0.0, 1.0], [0.0, 0.0, 1.0]])
_play_ignore = torch.tensor([[False, False, True], [False, True, False]])  # two unknown pixels
masked_loss_fn = use(
    masked_loss,
    reference=_ref_masked_loss,
    check=lambda f: [
        round(float(f(_play_logits, _play_labels, ig)), 4)
        for ig in (_play_ignore, torch.zeros(2, 3, dtype=bool))
    ],
    name="masked_loss",
)
_with, _without = (
    float(masked_loss_fn(_play_logits, _play_labels, ig))
    for ig in (_play_ignore, torch.zeros_like(_play_ignore))
)
print(f"play example · loss {_with:.4f} with two pixels unknown, {_without:.4f} with none")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>Same label file as the naive head (you saw its hidden-vessel recall in section 3), but the unpainted pixels
are now unknown. Write two guesses:</p>
<ul>
<li>On the hidden vessels of the training images, will its recall be closer to the naive head's or to the
  section-2 head's (full masks)?</li>
<li>On new images, will it mark more of the fundus than the naive head, or less?</li>
</ul></div>"""))

The next cell trains the head with your masked loss (or ours) on the masked labels of 5.1.
With `TRAIN_YOURSELF = False` it loads our "masked" checkpoint instead, which was made with a **stricter
rule**: it was told exactly which pixels next to the hidden vessels are unknown and that the rest is
background. That rule needs the hidden truth, so its numbers are an upper bound for this path, not what
you can do from the half-painted file.

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "masked" head instead of training
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

# the supervision of this path: painted vessels + background strokes; the hidden truth never enters (L-005)
if TRAIN_YOURSELF:
    masked_head, masked_sec = train_head_with(miss_feats, masked_labels, masked_loss_fn, HEAD_STEPS)
    src = f"trained here on {len(miss_train)} images, {HEAD_STEPS} steps, masked loss"
else:
    masked_head, entry = load_head(SPECIALISED, "masked", side)
    masked_sec = entry["train_sec"]
    src = (
        f"our checkpoint {entry['file']} ({len(entry['train_names'])} images). ⚠️ It was told exactly which "
        "pixels next to the hidden vessels are unknown and that the rest is background: a rule that needs "
        "the hidden truth, so read its numbers as an upper bound for this path"
    )
print(f"masked head @ {side} px · {src} · {masked_sec:.1f} s on {DEVICE}")
if masked_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

The next cell scores the masked head next to the heads of sections 2 and 3, on the hidden and visible
vessels of the training images (as in 3.3) and on the held-out images, and plots recalls and precision.

In [ ]:
# @title Score the masked head next to the earlier heads { display-mode: "form" }
t0 = time.monotonic()
path_reports = {APPROACH_NAMES["masked"]: path_report(masked_head)}
masked_infer = round(time.monotonic() - t0, 1)
earlier_heads = {"naive": naive_head}
if "balanced_head" in globals():  # made by the 🔴 Q11 cell of section 3; skipped there = no such row here
    earlier_heads["naive_balanced"] = balanced_head
else:
    print("ℹ️ No balanced naive head (the 🔴 Q11 cell of section 3 was skipped): its row is left out.")
for key, tensors in earlier_heads.items():
    path_reports[APPROACH_NAMES[key]] = path_report(tensors)
path_reports["full masks (section 2)"] = path_report(full_head)
masked = path_reports[APPROACH_NAMES["masked"]]

RESULTS["masked_recall_hidden_fg"] = masked["recall_hidden_fit"]  # same definition as section 3 (SPEC §9)
RESULTS["masked_head"] = public(masked) | {
    "long_side": side,
    "n_train": len(miss_train),
    "known_share": float(known["masked"]),
    "n_bg_strokes": int(n_bg_strokes),
    "trained_here": TRAIN_YOURSELF,
    "train_sec": round(float(masked_sec), 2),
    "infer_sec": masked_infer,
}
print_reports(path_reports)
print(f"section 3 recorded for the naive head · hidden-vessel recall {RESULTS['naive_recall_hidden_fg']:.3f}")

PATH_COLOURS = {APPROACH_NAMES[k]: APPROACH_COLOURS[k] for k in earlier_heads} | {
    APPROACH_NAMES["masked"]: APPROACH_COLOURS["masked"],
    "full masks (section 2)": THEME["works"],
}
BARS = {
    "recall_hidden_fit": "hidden vessels\n(training images)",
    "recall": "recall\n(held-out)",
    "precision": "precision\n(held-out)",
}


def path_chart(names: list[str]) -> None:
    """Grouped bars: recall on the hidden training vessels, held-out recall and precision, per head."""
    x, width = np.arange(len(BARS)), 0.8 / len(names)
    fig, ax = plt.subplots(figsize=(7.2, 3))
    for j, name in enumerate(names):
        shift = (j - (len(names) - 1) / 2) * width
        ax.bar(x + shift, [path_reports[name][b] for b in BARS], width, color=PATH_COLOURS[name], label=name)
    ax.set_xticks(x, list(BARS.values()), fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title("Path 1: what each head finds, and how much of what it marks is right")
    ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()


path_chart(list(PATH_COLOURS))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn: compare the heads on one image</b><ul>
<li>Run the next cell: the hidden vessels each head found on a training image, then the error maps of
  section 4 on the held-out demo image (violet found, orange false alarm, green missed). No new training.</li>
<li>Then set <code>LOOK_IMAGE</code> to 1, 2, … and run it again, to answer Q17.</li>
</ul></div>"""))

In [ ]:
# Q17 — hidden vessels found on a training image, then error maps on the held-out demo image
LOOK_IMAGE = 0  # try 1, 2, ...
if not isinstance(LOOK_IMAGE, int) or not 0 <= LOOK_IMAGE < len(miss_train):
    raise SystemExit(f"LOOK_IMAGE must be a whole number from 0 to {len(miss_train) - 1}.")
it = miss_train[LOOK_IMAGE]
LOOK = [APPROACH_NAMES["naive"], APPROACH_NAMES["masked"]]
look_preds = {name: path_reports[name]["fit_preds"][LOOK_IMAGE] for name in LOOK}
show_masks(
    [
        (
            f"{name.split(':')[0]} · hidden found {hidden_recall(pred, it['hidden']):.0%}",
            two_colour(it["image"], it["hidden"] & pred, it["hidden"] & ~pred),
        )
        for name, pred in look_preds.items()
    ],
    width=3.4,
)
k_demo = next(i for i, e in enumerate(SPEC_EVAL) if e is demo)
show_masks(
    [
        (
            f"{name.split(':')[0]} · IoU {score(path_reports[name]['held_preds'][k_demo], demo)['iou']:.2f}",
            error_map(demo["image"], path_reports[name]["held_preds"][k_demo], demo),
        )
        for name in LOOK
    ],
    width=3.4,
)

**Q17.** 🟡 Compare the naive and the masked head:

- Which hidden vessels does the masked head recover that the naive head missed? Think back to the third
  panel of 5.1: which pixels may it call vessel without any penalty?
- On the held-out image, what did each head pay for what it found? Compare the orange.
- Does that hold on another training image?
- What would the loss do if you also removed the background strokes?

<details><summary>What to look for</summary>

The masked head was never told that a hidden vessel is background, so it treats a vessel-looking patch as
a vessel wherever it is: it finds most of the hidden vessels, the ones the naive head dropped. Every dark
pixel in the third panel of 5.1 is free in this way: the hidden vessels, which is the point, but also all
the background between the strokes, including the background right next to every vessel.

The price shows on the held-out image. The only background the loss heard about is a handful of strokes
far from any vessel, so nothing told it where a vessel ends: the area around each vessel, and sometimes
darker background, turns orange.

Compare the "marked" column: the naive head is too stingy, the masked head is often too generous, and IoU
alone may rank them either way. Both failures come from what the loss was told about the unpainted
pixels, in opposite directions.

Without the background strokes the loss would only hear "vessel": the trivial solution of the Theory, shown
with a fake head below.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: &quot;ignore&quot; is not free</b><p>The trivial solution of 5.1 (the Theory's "Catch"): the training loss can look great while the mask covers
the whole eye. Watch the "marked" share. The next cell shows it with a fake head.</p></div>"""))

In [ ]:
# @title A fake head that always answers "vessel" { display-mode: "form" }
_lab = torch.cat([torch.from_numpy(x.astype(np.int8)).flatten() for x in masked_labels])
_y, _ignored = (_lab == 1).float(), _lab < 0
_logits = torch.full_like(_y, 8.0)  # "vessel" everywhere
_bce = F.binary_cross_entropy_with_logits(_logits, _y, reduction="none")
_with_bg = float(_bce[~_ignored].mean())
_no_bg = float(_bce[~_ignored & (_lab == 1)].mean())  # the same labels with the background strokes removed
print(
    f"mean loss of 'vessel everywhere' on the pixels the masked loss keeps · with the background strokes "
    f"{_with_bg:.3f} · without them {_no_bg:.5f} (≈ 0: a perfect score for a useless head)"
)

### 5.3 🔴 Optional: a PU-inspired loss

The masked head needed background strokes; the PU head needs none. It reads every unpainted pixel as
*unlabelled*: a mix of background and the vessels nobody painted. If you know roughly what share of the
vessels was painted, you know roughly how much vessel hides in that mix, and you can subtract it.

The idea in one line: **count everything as background, then subtract the vessels you know hide in
there.** The **prior** π is the share of fundus pixels that are vessel. We estimate it as (painted share
of the fundus) ÷ (share of the vessels that was painted). The second number comes from the annotator ("I
did about half"; here the dataset config). Example: painted 1.5 % of the fundus, annotator did half →
π = 3 %.

1. Vessel loss on the painted pixels, target "vessel": `L_v = mean_painted(−log p)`.
2. Background loss. Calling every pixel "background" mixes π vessels with (1−π) background, so subtract
   the vessels: `L_b = ( mean_all(−log(1−p)) − π · mean_painted(−log(1−p)) ) / (1−π)`.
3. Balance them: `L = ½·L_v + ½·max(L_b, 0)`. The max stops L_b dipping below zero when the head
   overfits the few painted vessels.

It assumes the painted vessels are a random sample of all vessels: true here, because we hid segments at
random.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Go deeper: where the PU loss comes from</b></summary><ul>
<li>Elkan &amp; Noto, KDD 2008: with labelled positives drawn at random from all positives, plus the share of
  positives, you can estimate the loss you would get with complete labels.</li>
<li>Kiryo et al., "Positive-Unlabeled Learning with Non-Negative Risk Estimator", NeurIPS 2017,
  <a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/1703.00593">arXiv:1703.00593</a>: the clipping at zero in step 3.</li>
<li>We use their non-negative version on a single pixel classifier, which is why we call it PU-<em>inspired</em>.</li>
</ul></details></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 🔴): guess the annotator wrong</b><ul>
<li>Read <code>pu_loss</code> in the next cell, then run it.</li>
<li>The prior rests on one number from the annotator, <code>LABELLED_SHARE</code>. Pretend the annotator was wrong:
  set <code>LABELLED_SHARE</code> to 0.3, then 0.8, and run again.</li>
<li>The cell defines the PU loss, estimates the prior from the painted vessels and <code>LABELLED_SHARE</code>, trains
  the head on the PU labels of 5.1 (seconds) and prints its numbers next to the other heads.</li>
</ul></div>"""))

In [ ]:
# Q18 — a PU-inspired loss: painted vessels + unlabelled pixels + one guess about the annotator
LABELLED_SHARE = 1 - MISSING["hidden_fraction"]  # the annotator's "I did about half"; try 0.3, 0.8
if not isinstance(LABELLED_SHARE, (int, float)) or not 0 < LABELLED_SHARE <= 1:
    raise SystemExit("LABELLED_SHARE must be a number above 0 and at most 1, e.g. 0.5.")


def pu_loss(
    logits: torch.Tensor, labels: torch.Tensor, ignore_mask: torch.Tensor, prior: float
) -> torch.Tensor:
    """Balanced non-negative PU loss: labels 1 = painted vessel, 0 = unlabelled; `prior` = vessel share."""
    z, y = logits[~ignore_mask], labels[~ignore_mask]
    painted = z[y == 1]
    vessel_loss = F.softplus(-painted).mean()  # (1) painted vessels, target "vessel"
    all_as_bg = F.softplus(z).mean()  # every pixel, target "background"
    painted_as_bg = F.softplus(painted).mean()  # painted vessels, target "background"
    bg_loss = (all_as_bg - prior * painted_as_bg) / (1 - prior)  # (2) the true background, estimated
    return 0.5 * vessel_loss + 0.5 * bg_loss.clamp(min=0)  # (3) balanced, clipped at zero


painted_share = sum(int((lab == 1).sum()) for lab in pu_labels) / sum(
    int((lab >= 0).sum()) for lab in pu_labels
)
prior = painted_share / LABELLED_SHARE
if not 0 < prior < 1:
    raise SystemExit(
        f"painted share {painted_share:.3f} ÷ LABELLED_SHARE {LABELLED_SHARE} gives a prior of "
        f"{prior:.2f}, which must lie between 0 and 1: choose a larger LABELLED_SHARE."
    )
pu_head, pu_sec = train_head_with(
    miss_feats, pu_labels, lambda z, y, ig: pu_loss(z, y, ig, prior), HEAD_STEPS
)
t0 = time.monotonic()
path_reports[APPROACH_NAMES["pu"]] = pu = path_report(pu_head)
PATH_COLOURS[APPROACH_NAMES["pu"]] = APPROACH_COLOURS["pu"]
RESULTS["pu_recall_hidden_fg"] = pu["recall_hidden_fit"]
RESULTS["pu_head"] = public(pu) | {
    "long_side": side,
    "n_train": len(miss_train),
    "labelled_share": float(LABELLED_SHARE),
    "prior": float(prior),
    "train_sec": round(float(pu_sec), 2),
    "infer_sec": round(time.monotonic() - t0, 1),
}
print(
    f"PU head · painted {painted_share:.1%} of the fundus ÷ labelled share {LABELLED_SHARE:.2f} → prior "
    f"{prior:.1%} vessel · trained in {pu_sec:.1f} s on {DEVICE}"
)
print_reports(
    {k: path_reports[k] for k in (APPROACH_NAMES["naive"], APPROACH_NAMES["masked"], APPROACH_NAMES["pu"])}
)
path_chart([APPROACH_NAMES["naive"], APPROACH_NAMES["masked"], APPROACH_NAMES["pu"]])

**Q18.** 🔴 Train again with a smaller and a larger `LABELLED_SHARE`. Which way does the head move:

- on recall?
- on the marked share?

<details><summary>What to look for</summary>

The PU head hears about every pixel, like the naive head, but it no longer takes "unpainted" for
"background": it expects a share π of vessel in the mix and stops pushing once the mix looks like that.
Where it lands between the naive and the masked head depends on the run, so read your numbers.

Compare it with the balanced naive head of 3.4: on labels hidden at random the two can land close,
because the damage of "missing = background" is mostly a wrong proportion, and both correct it, one by
reweighting, the other by subtracting the vessels it expects in the mix.

The prior is the lever. A smaller `LABELLED_SHARE` claims more hidden vessel, a larger prior, and the
head marks more; at 1.0 it claims nothing is hidden and moves back towards the naive head. A wrong guess
moves the head in a predictable direction rather than breaking it: with a small fully labelled check set,
you can pick the share that matches it.

</details>

**Q19.** 🟡 The masked head rests on one assumption: the hidden vessels look like the painted ones, because
we hid segments at random. Name a real annotation habit that breaks it. What would the masked head learn
then?

<details><summary>A possible answer</summary>

Annotators skip what is hard or tedious: the thinnest capillaries, the low-contrast periphery, the
vessels inside a bleeding. Then the painted vessels are thick and clear, and the unpainted ones are thin
and faint.

- **Masked head:** never hears "background" on the thin vessels, but never hears "vessel" either. It learns
  "vessel = thick and clear" and still misses most of them.
- **PU head (🔴, if you ran 5.3):** worse off. Its correction assumes the hidden vessels look like the
  painted ones, so it pushes the thin ones towards background. The PU literature calls this a violation of
  the "selected completely at random" assumption.

The practical fix is the one of the ⚠️ in section 3: keep a small fully labelled set and check the recall
on the kind of structure your annotators skip.

</details>

The last cell of the section writes the annotation effort of the three Path 1 heads for the comparison
table of section 8: the painted half of each vessel map, plus seconds per background stroke.

In [ ]:
# @title Annotation effort of the Path 1 heads { display-mode: "form" }
# LABELLED_SHARE is set in the 🔴 PU cell (5.3); if it was skipped, use the annotator's share from the config
_share = LABELLED_SHARE if "LABELLED_SHARE" in globals() else 1 - MISSING["hidden_fraction"]
painted_masks = len(miss_train) * _share
RESULTS["path1_effort_min"] = {
    "naive": round(effort_minutes(n_full_masks=painted_masks), 2),
    "masked": round(effort_minutes(n_strokes=n_bg_strokes, n_full_masks=painted_masks), 2),
    "pu": round(effort_minutes(n_full_masks=painted_masks), 2),
}
print(
    " · ".join(
        f"{APPROACH_NAMES[k].split(':')[0]} {v:.0f} min" for k, v in RESULTS["path1_effort_min"].items()
    )
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway</b><ul>
<li><strong>A label file where unpainted = background</strong> → mark the unpainted pixels "ignore" (−1), so the hidden
  vessels stop being punished.</li>
<li><strong>The loss ignores almost everything</strong> → keep some known background close to the objects, and watch the
  "marked" share, not the training loss: hidden vessels come back, but the orange area grows.</li>
<li><strong>Only positives are labelled and you can estimate their share</strong> → try the PU loss, and sweep
  <code>LABELLED_SHARE</code> to see how much a wrong guess moves recall.</li>
<li><strong>Annotators skip the thin or faint cases</strong> → neither fix works and none adds information about the
  hidden vessels: keep a small fully labelled check set, or try a second model (section 6).</li>
</ul></div>"""))

## 6. Path 2 · Recover supervision

Path 1 kept the label file and changed the loss. Path 2 keeps the loss and fills in the labels.

A second model, the **teacher** (the model whose output we use as labels), guesses every pixel nobody
painted, and the head trains on those guesses as if a person had drawn them. A guess used this way is a
**pseudo-label**. It is like handing your half-done colouring-in to a fast, confident colleague: the page
comes back full, and every confident mistake is now in your training set.

We start from the scribbles of section 4 (a few strokes per image, no more annotation than rung 2). Two
teachers get the job: **SAM**, prompted with clicks taken from the strokes, and **Grad-CAM**, the heat
map of a small classifier that only learnt whether a tile contains a vessel.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>Turn SAM pseudo-labels and Grad-CAM maps into training signal, and know when to distrust them.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: when pseudo-labels work</b><p>A pseudo-label works when the teacher's guesses are mostly right <em>and</em> its mistakes are not systematic.</p>
<ul>
<li>Pets: SAM's mask of a cat is a fine pseudo-label.</li>
<li>Retina: SAM's "vessel" mask is a fat region, and a head trained on it learns the region, not the line.</li>
<li>Guard: keep only the guesses the teacher is sure of and leave the rest unlabelled (6.2). A person's
  strokes always overrule the teacher.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Pseudo-label papers and confirmation bias</b></summary><p>A model trained on its teacher's errors learns to repeat them. The literature calls this <em>confirmation
bias</em>. The usual guard is a confidence threshold.</p>
<ul>
<li>Lee, "Pseudo-Label", ICML 2013 workshop: the original recipe.</li>
<li>FixMatch, Sohn et al., NeurIPS 2020: <a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/2001.07685">arXiv:2001.07685</a></li>
</ul></details></div>"""))

This helper cell adds a colour for each head of this path and defines how to spread SAM clicks along the
strokes of one kind, vessel or background.

In [ ]:
# @title Path 2 helpers: colours, clicks along the strokes { display-mode: "form" }
APPROACH_COLOURS |= {
    "sam_pseudo": "#0277BD",
    "gradcam_pseudo": "#827717",
}
APPROACH_NAMES |= {
    "sam_pseudo": "pseudo-labels: SAM from stroke clicks",
    "gradcam_pseudo": "pseudo-labels: Grad-CAM of a tile classifier",
}


def prob_rgb(prob: np.ndarray) -> np.ndarray:
    """A probability map as RGB on a fixed scale: purple = 1, white = 0.5 (unsure), orange = 0."""
    return (mpl.colormaps["PuOr"](np.clip(prob, 0, 1))[..., :3] * 255).astype(np.uint8)


def stroke_points(item: dict, value: int, n: int) -> list[list[float]]:
    """n (x, y) clicks spread along the strokes of one kind (1 vessel, 2 background) by farthest-point
    sampling from the first stroke pixel: no randomness, no ground truth."""
    ys, xs = np.nonzero(item["scribbles"] == value)
    if len(ys) == 0:
        return []
    pts = np.stack([xs, ys], 1).astype(float)
    chosen = [0]
    dist = np.linalg.norm(pts - pts[0], axis=1)
    while len(chosen) < min(n, len(pts)):
        k = int(dist.argmax())
        chosen.append(k)
        dist = np.minimum(dist, np.linalg.norm(pts - pts[k], axis=1))
    return [[float(x), float(y)] for x, y in pts[chosen]]

### 6.1 SAM fills in the gaps

We take the scribbles of section 4 again: a few vessel strokes and a few background strokes on each of the
same five training images.

- Twenty clicks spread along the vessel strokes go in as "vessel", twenty along the background strokes as
  "not vessel".
- SAM's answer for every pixel is the pseudo-label.
- Only the strokes go in. The ground truth appears in the figure so you can judge the guesses, and never
  in training.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>Section 2 showed what SAM does with clicks on a vessel tree. Now it gets forty clicks per image, half of
them saying "not here". Will its mask follow the vessels, or cover a region around them? Write one guess.</p></div>"""))

How to read the printout and the last panel of the next cell:

- **Probability spread:** the 5 %, 50 % and 95 % points of SAM's probability over the fundus pixels. Near 0
  or 1 means sure, near 0.5 means unsure.
- **Marked:** the share of fundus pixels the pseudo-label calls vessel (section 2 printed how much is
  really vessel).
- **IoU, clDice:** overlap with the hidden truth, as in section 2. Diagnosis only.
- **Colours:** violet = a real vessel found, orange = a vessel SAM invented, green = a vessel SAM left out.
  Every orange and green pixel becomes a wrong training label if we take the guesses as they are.

This cell asks SAM for a probability map on each training image from the stroke clicks and shows one image
(seconds per image on a T4, much slower on a CPU, where `RUN_SAM = False` loads our maps, made the same way
with clicks drawn at random).

In [ ]:
RUN_SAM = True  # False -> load our precomputed SAM maps for the training images
N_PSEUDO_CLICKS = 20  # clicks per kind (vessel / background) and image, taken from the strokes
if not isinstance(RUN_SAM, bool):
    raise SystemExit("RUN_SAM must be True or False.")

t0 = time.monotonic()
pseudo_probs, pseudo_clicks = [], []  # per training image: SAM's probability map, (vessel, background) clicks
if RUN_SAM:
    for it in scrib_train:
        pos, neg = stroke_points(it, 1, N_PSEUDO_CLICKS), stroke_points(it, 2, N_PSEUDO_CLICKS)
        prob = sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg))
        pseudo_probs.append(prob)
        pseudo_clicks.append((pos, neg))
    src = f"SAM 2.1 run here, {N_PSEUDO_CLICKS} + {N_PSEUDO_CLICKS} clicks per image"
else:
    ours = PRECOMPUTED / "pseudo_labels" / "specialised"
    if not (ours / "prompts.json").is_file():
        raise SystemExit(f"{ours} is missing. Run setup_check.ipynb first, or set RUN_SAM = True.")
    prompts = json.loads((ours / "prompts.json").read_text())["prompts"]
    for it in scrib_train:
        pseudo_probs.append(read_png(ours / f"{it['name']}.png") / 255)
        p = prompts[it["name"]]
        clicks = [
            [xy for xy, lab in zip(p["points_xy"], p["labels"], strict=True) if lab == k] for k in (1, 0)
        ]
        pseudo_clicks.append(tuple(clicks))
    src = "our precomputed maps (same recipe, clicks drawn at random from the strokes)"
pseudo_sec = round(time.monotonic() - t0, 1)

# diagnosis only: SAM's guesses at the usual 0.5 cut, scored against the hidden truth (never for training)
sam_quality = mean_scores([score(p >= 0.5, it) for p, it in zip(pseudo_probs, scrib_train, strict=True)])
sam_marked = float(
    np.mean([(p >= 0.5)[region(it)].mean() for p, it in zip(pseudo_probs, scrib_train, strict=True)])
)
# how sure SAM is, inside the fundus (no truth needed): the spread of its probabilities
spread = np.quantile(
    np.concatenate([p[region(it)] for p, it in zip(pseudo_probs, scrib_train, strict=True)]),
    [0.05, 0.5, 0.95],
)
print(
    f"SAM pseudo-labels on {len(scrib_train)} training images · {src} · {pseudo_sec} s on {DEVICE}\n"
    f"SAM's probability inside the fundus · 5 % of the pixels below {spread[0]:.2f} · median "
    f"{spread[1]:.2f} · 95 % below {spread[2]:.2f}\n"
    f"diagnosis (truth used to score only): at 0.5 they mark {sam_marked:.0%} of the fundus · IoU "
    f"{sam_quality['iou']:.3f} · clDice {sam_quality['cldice']:.3f}"
)
ex, (pos, neg) = scrib_train[0], pseudo_clicks[0]
show_clicks(
    [
        (f"{ex['name']} · vessel clicks", ex["image"], pos),
        ("background clicks", ex["image"], neg),
        ("SAM probability: purple 1, white 0.5, orange 0", prob_rgb(pseudo_probs[0]), None),
        ("pseudo-label at 0.5 vs truth", error_map(ex["image"], pseudo_probs[0] >= 0.5, ex), None),
    ],
    width=2.8,
)

**Q20.** 🟡 Look at the error map and the probability panel. Where does SAM's guess go wrong: around
the thick vessels, along the thin ones, in the background between the strokes? Is SAM sure of itself (deep
purple, deep orange) or unsure (pale)? What does that mean for a cut at exactly 0.5?

<details><summary>What to look for</summary>

SAM looks for *objects* that contain the positive clicks and avoid the negative ones.

- Forty clicks scattered over a vessel tree do not describe one object. SAM settles for the object that
  holds most of them, often the whole fundus. Its edge is careful, but the inside is a region (orange), not
  a tree. Thin branches between the clicks can drop out (green).
- If most of the probability panel is pale, the values sit just around 0.5 and the exact cut decides
  almost everything: a little below and the whole fundus is "vessel", a little above and nothing is. The
  binary mask hides that doubt, because it looks just as crisp as a confident one.
- Whatever you see on your run, the threshold is your choice, not a property of SAM.

</details>

### 6.2 From a probability map to training targets

A target is the label the head trains on: 1 vessel, 0 background, −1 ignored (the loss skips it). The cut
turns SAM's probability into a target, and an unsure band lets us ignore the doubtful pixels instead of
guessing.

This cell draws the rule once, for a cut of 0.5 and a band of 0.1.

In [ ]:
# @title The cut and the unsure band, drawn { display-mode: "form" }
cut, band_demo = 0.5, 0.1
fig, ax = plt.subplots(figsize=(7.2, 1.5))
for lo, hi, colour, label in [
    (0, cut - band_demo, THEME["background"], "0 · background"),
    (cut - band_demo, cut + band_demo, "white", "−1 · ignored"),
    (cut + band_demo, 1, THEME["works"], "1 · vessel"),
]:
    ax.barh(0, hi - lo, left=lo, color=colour, edgecolor="black", height=0.5)
    text_colour = "black" if colour == "white" else "white"
    ax.text((lo + hi) / 2, 0, label, ha="center", va="center", fontsize=8, color=text_colour)
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xlabel("SAM's probability for a pixel")
ax.set_title(
    f"cut {cut}, band {band_demo}: below {cut - band_demo} → 0 · between → −1 · from {cut + band_demo} → 1",
    fontsize=9,
)
plt.tight_layout()
plt.show()

Two rules sit on top of the cut: a person's stroke always wins over SAM, and pixels outside the fundus are
always −1.

This helper cell holds our solution of the next exercise. Open it only if you are stuck.

In [ ]:
# @title Reference solution (open only if stuck) { display-mode: "form" }
def _ref_pseudo_targets(
    prob: np.ndarray, scribbles: np.ndarray, region: np.ndarray, threshold: float = 0.5, band: float = 0.0
) -> np.ndarray:
    """Reference: sure guesses become labels, doubtful ones are ignored, strokes and region come last."""
    targets = np.full(prob.shape, -1, np.int8)
    targets[prob >= threshold + band] = 1
    targets[prob < threshold - band] = 0
    targets[scribbles == 1] = 1
    targets[scribbles == 2] = 0
    targets[~np.asarray(region, bool)] = -1
    return targets

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 3 min)</b><p>Fill <code>my_pseudo_targets</code> in the next cell.</p>
<ul>
<li><strong>In:</strong> SAM's <code>prob</code> map (floats in [0, 1]), the <code>scribbles</code> map (0 unlabelled, 1 vessel stroke, 2
  background stroke), the boolean <code>region</code>, a <code>threshold</code> and an unsure <code>band</code>.</li>
<li><strong>Out:</strong> int8 targets, 1 vessel, 0 background, −1 ignored.</li>
<li><strong>Rules:</strong> <code>prob ≥ threshold + band</code> is vessel, <code>prob &lt; threshold − band</code> is background, the pixels in
  between are ignored. Strokes always win. Pixels outside <code>region</code> are always −1.</li>
</ul>
<p>The cell checks your version on a 3 × 4 play map, without and with a band, and continues with ours if you
skip it.</p></div>"""))

In [ ]:
# ✍️ YOUR CODE
def my_pseudo_targets(
    prob: np.ndarray, scribbles: np.ndarray, region: np.ndarray, threshold: float = 0.5, band: float = 0.0
) -> np.ndarray:
    """Targets (int8) from a pseudo-label probability map: 1 where prob >= threshold + band, 0 where
    prob < threshold - band, -1 in between. Strokes overrule the map (1 vessel stroke -> 1, 2 background
    stroke -> 0); outside `region` -> -1.
    e.g. threshold 0.5, band 0.1, no strokes: prob [[0.9, 0.55, 0.3, 0.05]] -> [[1, -1, 0, 0]]"""
    ...


_play_prob = np.array([[0.95, 0.62, 0.45, 0.10], [0.55, 0.30, 0.71, 0.05], [0.20, 0.90, 0.40, 0.80]])
_play_scrib = np.zeros((3, 4), np.uint8)
_play_scrib[1, 1] = 1  # a vessel stroke where SAM says "not vessel"
_play_scrib[2, 1] = 2  # a background stroke where SAM says "vessel"
_play_region = np.ones((3, 4), bool)
_play_region[:, 3] = False  # the last column is outside the evaluation region
pseudo_targets = use(
    my_pseudo_targets,
    reference=_ref_pseudo_targets,
    check=lambda f: [
        np.asarray(f(_play_prob, _play_scrib, _play_region, 0.5, b)).astype(int) for b in (0.0, 0.2)
    ],
    name="pseudo_targets",
)
print("play example · targets with threshold 0.5 and band 0.2 (−1 = ignored):")
print(pseudo_targets(_play_prob, _play_scrib, _play_region, 0.5, 0.2))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The scribble head of section 4 heard about the strokes only. This head hears about every pixel of the
fundus, most of it in SAM's words. Will it mark more of the held-out fundus than the scribble head, or
less? Will its IoU be higher? Write two guesses.</p></div>"""))

This cell turns SAM's maps into targets at the usual cut (threshold 0.5, no band) and trains the section-2
head on them (`TRAIN_YOURSELF = False` loads our "pseudo" checkpoint instead, trained the same way).

In [ ]:
TRAIN_YOURSELF = True  # False -> load our precomputed "pseudo" head instead of training
if not isinstance(TRAIN_YOURSELF, bool):
    raise SystemExit("TRAIN_YOURSELF must be True or False.")

# the supervision of this path: the strokes, and SAM's guesses from clicks on the strokes (L-005)
sam_targets = [
    pseudo_targets(p, it["scribbles"], region(it), 0.5, 0.0)
    for p, it in zip(pseudo_probs, scrib_train, strict=True)
]
if TRAIN_YOURSELF:
    sam_head, sam_sec = train_head(scrib_feats, sam_targets, HEAD_STEPS)
    src = f"trained here on {len(scrib_train)} images, {HEAD_STEPS} steps, SAM pseudo-labels at 0.5"
else:
    sam_head, entry = load_head(SPECIALISED, "pseudo", side)
    sam_sec = entry["train_sec"]
    src = f"our checkpoint {entry['file']} ({len(entry['train_names'])} images)"
print(f"SAM pseudo-label head @ {side} px · {src} · {sam_sec:.1f} s on {DEVICE}")
if sam_sec > 60:
    print("ℹ️ Slow session? Set TRAIN_YOURSELF = False above to load our checkpoint next time.")

This cell scores the new head on the held-out images with the numbers of section 4 and prints it under the
scribble head, which saw the same strokes but kept every other pixel unknown.

In [ ]:
# @title Score the SAM pseudo-label head on the held-out images { display-mode: "form" }
t0 = time.monotonic()
path2_preds = {
    "scribbles": ladder_preds["scribbles"],
    "sam_pseudo": [
        predict_head(sam_head, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
    ],
}
sam_infer = round(time.monotonic() - t0, 1)
path2_report = {k: rung_report(v) for k, v in path2_preds.items()}
RESULTS["pseudo_label_iou"] = path2_report["sam_pseudo"]["iou"]  # held-out IoU of the head (SPEC §9)
RESULTS["sam_pseudo_head"] = path2_report["sam_pseudo"] | {
    "long_side": side,
    "n_train": len(scrib_train),
    "threshold": 0.5,
    "band": 0.0,
    "pseudo_label_iou_fit": sam_quality["iou"],
    "pseudo_sec": pseudo_sec,
    "sam_run_here": RUN_SAM,
    "trained_here": TRAIN_YOURSELF,
    "train_sec": round(float(sam_sec), 2),
    "infer_sec": sam_infer,
}


def print_path2(reports: dict) -> None:
    """One line per head with the numbers of section 4."""
    print(f"{'approach':<46}{COLUMNS}")
    for k, v in reports.items():
        name = APPROACH_NAMES.get(k, k)
        print(
            f"{name:<46}{v['iou']:>6.3f}{v['cldice']:>8.3f}{v['recall']:>8.3f}{v['precision']:>8.3f}"
            f"{v['marked']:>8.0%}"
        )


print_path2(path2_report)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Question</b><p>Run the next cell, then move the cut.</p>
<p><strong>Q21.</strong> 🟡 Tune the pseudo-labels on the <strong>training images</strong>, never on the test images. Raise or lower
<code>PSEUDO_THRESHOLD</code> and give the doubtful pixels an <code>UNSURE_BAND</code> to ignore, picking values from the spread
printed in 6.1 and judging them by how many training targets are wrong (the two error columns). Then look
at the held-out line once: does it move the same way? Is there a setting that beats 0.5 on your run?</p>
<ul>
<li><code>PSEUDO_THRESHOLD</code>: try 0.55, 0.6, 0.7 (or the median printed in 6.1).</li>
<li><code>UNSURE_BAND</code>: try 0.03; pixels within 0.03 of the threshold are ignored.</li>
</ul>
<p>Both are sliders in the next cell: move one, then run it again.</p></div>"""))

This cell trains the same head on targets made with your setting and prints it under the head at 0.5, with
three diagnosis columns from the hidden truth on the training images: **ignored** (the share of the fundus
set to −1), **vessel → bg** (the share of true vessel pixels that got label 0) and **bg → vessel** (the
share of true background pixels that got label 1).

In [ ]:
# Q21 — tune the pseudo-label threshold and the unsure band
PSEUDO_THRESHOLD = 0.5  # @param {type:"slider", min:0.45, max:0.75, step:0.025}
UNSURE_BAND = 0.0  # @param {type:"slider", min:0, max:0.1, step:0.01}
ok = all(isinstance(v, (int, float)) for v in (PSEUDO_THRESHOLD, UNSURE_BAND))
if not ok or UNSURE_BAND < 0 or PSEUDO_THRESHOLD - UNSURE_BAND <= 0 or PSEUDO_THRESHOLD + UNSURE_BAND > 1:
    raise SystemExit(
        "PSEUDO_THRESHOLD must lie between 0 and 1, and UNSURE_BAND ≥ 0 must keep both edges inside."
    )


def target_errors(targets: list[np.ndarray]) -> dict:
    """Diagnosis only (uses the hidden truth): share of the fundus ignored, of the true vessel pixels called
    background, and of the true background pixels called vessel, over the training images."""
    pairs = list(zip(targets, scrib_train, strict=True))
    return {
        "ignored": float(np.mean([(t < 0)[region(it)].mean() for t, it in pairs])),
        "vessel_as_bg": float(
            np.mean([((t == 0) & it["mask"]).sum() / max(it["mask"].sum(), 1) for t, it in pairs])
        ),
        "bg_as_vessel": float(
            np.mean(
                [
                    ((t == 1) & ~it["mask"] & region(it)).sum() / max((~it["mask"] & region(it)).sum(), 1)
                    for t, it in pairs
                ]
            )
        ),
    }


tuned_targets = [
    pseudo_targets(p, it["scribbles"], region(it), PSEUDO_THRESHOLD, UNSURE_BAND)
    for p, it in zip(pseudo_probs, scrib_train, strict=True)
]
tuned_head, tuned_sec = train_head(scrib_feats, tuned_targets, HEAD_STEPS)
tuned_preds = [
    predict_head(tuned_head, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
]
tuned_report = rung_report(tuned_preds)
RESULTS["sam_pseudo_tuned"] = tuned_report | {"threshold": PSEUDO_THRESHOLD, "band": UNSURE_BAND}
print(f"{'targets on the training images':<46}{'ignored':>8}{'vessel → bg':>13}{'bg → vessel':>13}")
for name, t in {"threshold 0.5, no band": sam_targets, "your setting": tuned_targets}.items():
    e = target_errors(t)
    print(f"{name:<46}{e['ignored']:>8.0%}{e['vessel_as_bg']:>13.0%}{e['bg_as_vessel']:>13.0%}")
print(f"\nyour head: threshold {PSEUDO_THRESHOLD}, band {UNSURE_BAND} · trained in {tuned_sec:.1f} s")
print_path2({"threshold 0.5, no band": path2_report["sam_pseudo"], "your setting": tuned_report})

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>👥 Pair moment (optional, 4 min)</b><p>One of you tries a low cut (<code>PSEUDO_THRESHOLD = 0.55</code>), the other a high one (<code>0.7</code>). Compare your <strong>vessel
→ bg</strong> and <strong>bg → vessel</strong> columns: which kind of wrong label did each of you buy, and what happened to the
held-out line?</p></div>"""))

<details><summary>What to look for</summary>

The threshold trades one kind of wrong label for the other.

- **Low cut:** more of SAM's region becomes "vessel" (bg → vessel rises) and the head learns to mark more.
- **High cut:** only SAM's surest pixels stay vessel and the rest becomes background (vessel → bg rises).
  That is the section-3 mistake again, with SAM as the annotator who stopped early.
- **Crowded probabilities:** when they crowd around one value, a step of 0.05 can flip most of the fundus
  from one side to the other, so there may be no comfortable middle.
- **The band:** it removes the doubtful pixels instead of guessing, so both error columns can fall at once.
  The price is fewer labelled pixels (the "ignored" column).

Whether the held-out numbers follow the training diagnosis depends on your run. In a real project you pick
the setting on a small fully labelled check set, never on the test images.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: over-trusting SAM&#x27;s confident masks</b><ol>
<li>SAM draws a crisp edge and fills the whole region. <strong>Consequence:</strong> it looks finished, with no "I don't
   know" anywhere.</li>
<li>The head cannot tell a guess from a drawing. <strong>Consequence:</strong> it learns SAM's habit (a region where you
   wanted a line) as faithfully as the vessels.</li>
</ol></div>"""))

This simulation invents a teacher that is right about the strokes but confidently calls the brightest blob
of the fundus (usually the optic disc) vessel, trains one head on the strokes alone and one with that blind
spot, and shows both on the same held-out image (no ground truth, seconds per head).

In [ ]:
# @title Playful pitfall: a teacher with one confident blind spot { display-mode: "form" }
def bright_blob(item: dict) -> np.ndarray:
    """The brightest smooth blob of the fundus (usually the optic disc), found from the image alone."""
    g = ndi.gaussian_filter(item["image"][..., 1].astype(float), sigma=max(item["mask"].shape) / 60)
    g[~region(item)] = 0
    peak = np.unravel_index(int(g.argmax()), g.shape)
    lab, _ = ndi.label(g >= 0.9 * g.max())
    return ndi.binary_dilation(lab == lab[peak], iterations=max(1, max(item["mask"].shape) // 100))


def stroke_targets(item: dict, blind_spot: bool) -> np.ndarray:
    """The strokes only (1 vessel, 0 background, -1 unknown); with `blind_spot`, the teacher also calls the
    bright blob vessel wherever no stroke says otherwise."""
    t = np.full(item["mask"].shape, -1, np.int8)
    t[item["scribbles"] == 1] = 1
    t[item["scribbles"] == 2] = 0
    if blind_spot:
        t[bright_blob(item) & (item["scribbles"] == 0)] = 1
    return t


clean_head, _ = train_head(scrib_feats, [stroke_targets(it, False) for it in scrib_train], HEAD_STEPS)
biased_head, _ = train_head(scrib_feats, [stroke_targets(it, True) for it in scrib_train], HEAD_STEPS)
feat_demo = features_for(SPECIALISED, demo, side)
p_clean = predict_head(clean_head, feat_demo, demo["mask"].shape)
p_biased = predict_head(biased_head, feat_demo, demo["mask"].shape)
blob_demo = bright_blob(demo)
print(
    f"blind spot ({blob_demo.mean():.1%} of the image) marked as vessel · head on strokes alone "
    f"{p_clean[blob_demo].mean():.0%} · head on the biased teacher {p_biased[blob_demo].mean():.0%}\n"
    f"fundus marked as vessel · strokes alone {p_clean[region(demo)].mean():.0%} · "
    f"biased teacher {p_biased[region(demo)].mean():.0%}"
)
show_masks(
    [
        ("the teacher's blind spot", overlay(demo["image"], blob_demo, THEME["fails"])),
        ("head on the strokes alone", overlay(demo["image"], p_clean, THEME["works"])),
        ("head on the biased teacher", overlay(demo["image"], p_biased, THEME["works"])),
    ],
    width=3.0,
)

<details><summary>What to look for</summary>

Compare the blind-spot share in the printout and the violet area inside the orange blob. If the biased head
marks the blob far more than the clean head, it copied the teacher's mistake onto an image it never saw
labelled. It may also mark other bright patches, because it learnt "bright and round means vessel", not
"this exact blob". If both heads mark about the same share on your run, the strokes alone already
outweighed the blind spot. Try another `demo` image in section 2 and compare.

</details>

### 6.3 Grad-CAM: where a classifier looks

The second colleague does not segment at all.

1. We train a small classifier to answer one yes/no question.
2. We ask it *where* in the image its "yes" came from. Grad-CAM turns that answer into a heat map.
3. A threshold turns the heat map into a pseudo-mask.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: an awkward fit, said plainly</b><p>Grad-CAM needs an image-level question with a real "no" class, such as "is there a defect in this image?".</p>
<ul>
<li>Every retina image contains vessels, so there is no "no" at image level.</li>
<li>We fall back to a patch-level question: "does this 64 px tile contain a vessel?", with tile labels read
  off the strokes.</li>
<li>Most tiles of a fundus crop do contain a vessel (the next cell counts them), and a map at patch
  resolution cannot outline a line a few pixels wide, so it fits this task poorly.</li>
<li>We look at it anyway: on a task with real image-level labels (defects, tumours, "contains a cell") it is
  often the cheapest supervision there is, and seeing it fail here shows what to check.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: Grad-CAM</b><p>Grad-CAM answers: which parts of the input made the classifier say "yes"?</p>
<ul>
<li><strong>How:</strong> each of the classifier's feature maps Aₖ gets a weight αₖ (how much the "yes" score grows when Aₖ
  grows). The heat map is the weighted sum of the maps, keeping only the positive evidence.</li>
<li><strong>Pets:</strong> for "is there a cat?" the map lights up the face, not the whole cat.</li>
<li><strong>Here:</strong> one value per 16 px patch, because the features have that resolution.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Grad-CAM formula, CAM and the package</b></summary><p>The formula: <code>αₖ = mean over (i, j) of ∂score/∂Aₖ[i, j]</code>, then <code>map = ReLU(Σₖ αₖ·Aₖ)</code>, rescaled to [0, 1].</p>
<p>Our classifier ends in an average over patches. For such a classifier Grad-CAM reduces to the older CAM: the
classifier's own evidence for "vessel", patch by patch. We use the <code>grad-cam</code> package.</p>
<ul>
<li>Grad-CAM, Selvaraju et al., ICCV 2017: <a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/1610.02391">arXiv:1610.02391</a></li>
<li>CAM, Zhou et al., CVPR 2016: <a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/1512.04150">arXiv:1512.04150</a></li>
</ul></details></div>"""))

This helper cell defines the tile classifier, the tiling, the tile labels from the strokes, its training and
its Grad-CAM map on a whole image.

In [ ]:
# @title Grad-CAM helpers: tile classifier, tiles, labels from strokes, the map { display-mode: "form" }
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


class TileClassifier(torch.nn.Module):
    """'Does this tile contain a vessel?' on frozen DINOv3 features: 3 x 3 conv (384 -> 32), ReLU, one logit
    per patch, averaged. Fully convolutional, so it also runs on a whole image."""

    def __init__(self, dim: int, hidden: int = 32):
        super().__init__()
        self.hidden = torch.nn.Conv2d(dim, hidden, 3, padding=1)
        self.act = torch.nn.ReLU()
        self.out = torch.nn.Conv2d(hidden, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.out(self.act(self.hidden(x))).mean((2, 3))  # [B, 1]


def to_tiles(grid: torch.Tensor, t: int) -> torch.Tensor:
    """[C, h, w] -> [n, C, t, t]: non-overlapping t x t tiles, row by row (a ragged edge is dropped)."""
    c, h, w = grid.shape
    nh, nw = h // t, w // t
    g = grid[:, : nh * t, : nw * t].reshape(c, nh, t, nw, t)
    return g.permute(1, 3, 0, 2, 4).reshape(nh * nw, c, t, t)


def tile_labels(item: dict, grid: tuple[int, int], t: int) -> torch.Tensor:
    """Per tile, from the strokes only: 1 = a vessel stroke crosses it, 0 = only background strokes,
    -1 = no stroke (not used)."""
    has = [to_tiles(patch_share(item["scribbles"] == v, grid)[None], t).amax((1, 2, 3)) > 0 for v in (1, 2)]
    return torch.where(has[0], 1, torch.where(has[1], 0, -1))


def train_tile_classifier(x: torch.Tensor, y: torch.Tensor, steps: int, lr: float = 0.01):
    """Full-batch Adam with the balanced cross-entropy of every head; returns the classifier and seconds."""
    torch.manual_seed(SEED)
    clf = TileClassifier(x.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=lr)
    target = (y == 1).float()
    w = torch.where(y == 1, 0.5 / max(int((y == 1).sum()), 1), 0.5 / max(int((y == 0).sum()), 1))
    t0 = time.monotonic()
    for _ in range(steps):
        opt.zero_grad()
        bce = F.binary_cross_entropy_with_logits(clf(x)[:, 0], target, reduction="none")
        (bce * w).sum().backward()
        opt.step()
    return clf.eval(), time.monotonic() - t0


def cam_map(clf: TileClassifier, feat: torch.Tensor) -> np.ndarray:
    """Grad-CAM of the 'contains a vessel' score on a whole (standardised) feature grid: [h, w] in [0, 1]."""
    with GradCAM(model=clf, target_layers=[clf.act]) as cam:
        return cam(input_tensor=feat[None], targets=[ClassifierOutputTarget(0)])[0]

This cell cuts the five training images into 64 px tiles, labels each from the strokes (a vessel stroke
crosses it → "vessel", only background strokes → "no vessel", no stroke → unused), trains the classifier in
seconds and counts the tiles, with the held-out truth showing how rare a vessel-free tile really is.

In [ ]:
# @title Train the tile classifier and count the tiles { display-mode: "form" }
TILE_PATCHES = 1 if SMOKE else 4  # 64 px of the 1024 px crop either way (smoke thumbnails are 4× smaller)
CLS_STEPS = 30 if SMOKE else 200
all_feats = torch.cat([f.flatten(1) for f in scrib_feats], 1)
cls_mu, cls_sd = all_feats.mean(1).view(-1, 1, 1), all_feats.std(1).view(-1, 1, 1) + 1e-6
std_feats = [(f - cls_mu) / cls_sd for f in scrib_feats]
tile_x, tile_y = [], []
for it, f in zip(scrib_train, std_feats, strict=True):
    lab = tile_labels(it, tuple(f.shape[1:]), TILE_PATCHES)
    tile_x.append(to_tiles(f, TILE_PATCHES)[lab >= 0])
    tile_y.append(lab[lab >= 0])
tile_x, tile_y = torch.cat(tile_x), torch.cat(tile_y)
clf, cls_sec = train_tile_classifier(tile_x, tile_y, CLS_STEPS)
with torch.no_grad():
    fit_acc = float(((clf(tile_x)[:, 0] > 0).long() == tile_y).float().mean())
# evaluation only: the share of held-out tiles that truly contain a vessel
held_tiles = []
for it in SPEC_EVAL:
    grid = tuple(features_for(SPECIALISED, it, side).shape[1:])
    share = to_tiles(patch_share(it["mask"] & region(it), grid)[None], TILE_PATCHES).amax((1, 2, 3))
    inside = to_tiles(patch_share(region(it), grid)[None], TILE_PATCHES).mean((1, 2, 3)) > 0.5
    held_tiles.append(float((share[inside] > 0).float().mean()))
vessel_tile_share = float(np.mean(held_tiles))
print(
    f"tile classifier · {int((tile_y == 1).sum())} 'vessel' tiles, {int((tile_y == 0).sum())} 'no vessel' "
    f"tiles from the strokes · fit accuracy {fit_acc:.0%} · {cls_sec:.1f} s on {DEVICE}"
)
print(f"held-out truth (evaluation only): {vessel_tile_share:.0%} of the fundus tiles contain a vessel")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The classifier only ever answered "vessel in this tile, yes or no". Will its heat map look like a vessel
tree, like blobs, or like noise? Write one guess.</p></div>"""))

This cell cuts each training image's Grad-CAM map at half its peak into a pseudo-mask (strokes still win),
trains the same head on it in seconds, scores it on the held-out images and shows one training image.

In [ ]:
# @title Grad-CAM pseudo-labels: map, cut, head, score { display-mode: "form" }
CAM_THRESHOLD = 0.5  # share of the map's peak; Grad-CAM maps are rescaled to [0, 1] per image
t0 = time.monotonic()
cam_probs = []
for it, f in zip(scrib_train, std_feats, strict=True):
    cam = torch.from_numpy(cam_map(clf, f))[None, None]
    cam_probs.append(
        F.interpolate(cam, size=it["mask"].shape, mode="bilinear", align_corners=False)[0, 0].numpy()
    )
cam_sec = round(time.monotonic() - t0, 1)
# the supervision: strokes, and the tile classifier's heat map (trained on stroke-derived tile labels)
cam_targets = [
    pseudo_targets(p, it["scribbles"], region(it), CAM_THRESHOLD, 0.0)
    for p, it in zip(cam_probs, scrib_train, strict=True)
]
cam_quality = mean_scores([score(t == 1, it) for t, it in zip(cam_targets, scrib_train, strict=True)])
cam_head, cam_head_sec = train_head(scrib_feats, cam_targets, HEAD_STEPS)
t0 = time.monotonic()
path2_preds["gradcam_pseudo"] = [
    predict_head(cam_head, features_for(SPECIALISED, it, side), it["mask"].shape) for it in SPEC_EVAL
]
cam_infer = round(time.monotonic() - t0, 1)
path2_report["gradcam_pseudo"] = rung_report(path2_preds["gradcam_pseudo"])
RESULTS["gradcam_iou"] = path2_report["gradcam_pseudo"]["iou"]  # held-out IoU of the head (SPEC §9)
RESULTS["gradcam_note"] = (
    f"Patch-level proxy '{SPECIALISED['gradcam']['patch_level_task']}' (tiles labelled from the strokes), "
    f"because every image contains the structure and there is no image-level negative class: an awkward "
    f"fit, {vessel_tile_share:.0%} of the held-out tiles contain foreground"
)
RESULTS["gradcam_head"] = path2_report["gradcam_pseudo"] | {
    "long_side": side,
    "n_train": len(scrib_train),
    "tile_patches": TILE_PATCHES,
    "n_tiles": [int((tile_y == 1).sum()), int((tile_y == 0).sum())],
    "classifier_fit_acc": fit_acc,
    "held_vessel_tile_share": vessel_tile_share,
    "threshold": CAM_THRESHOLD,
    "pseudo_label_iou_fit": cam_quality["iou"],
    "train_sec": round(float(cls_sec + cam_head_sec), 2),
    "pseudo_sec": cam_sec,
    "infer_sec": cam_infer,
}
cam_marked = float(
    np.mean([(t == 1)[region(it)].mean() for t, it in zip(cam_targets, scrib_train, strict=True)])
)
print(
    f"Grad-CAM pseudo-labels · {cam_sec} s · diagnosis (truth used to score only): they mark "
    f"{cam_marked:.0%} of the fundus · IoU {cam_quality['iou']:.3f} · clDice {cam_quality['cldice']:.3f} "
    f"on the training images (SAM at 0.5: {sam_marked:.0%}, IoU {sam_quality['iou']:.3f})"
)
print(f"Grad-CAM head trained in {cam_head_sec:.1f} s on {DEVICE}")
print_path2(path2_report)
show_masks(
    [
        (f"{ex['name']}", ex["image"]),
        ("Grad-CAM: purple = strongest evidence", prob_rgb(cam_probs[0])),
        ("Grad-CAM pseudo-label vs truth", error_map(ex["image"], cam_targets[0] == 1, ex)),
        ("SAM pseudo-label vs truth", error_map(ex["image"], sam_targets[0] == 1, ex)),
    ],
    width=2.8,
)

**Q22.** 🟡 Compare the two pseudo-labels of the same training image. What shape does the Grad-CAM
evidence have, and where does it sit? Which of the two would you rather correct by hand, and why?

<details><summary>What to look for</summary>

- **Shape:** blobs one or a few patches wide, where the classifier found the most "vessel-like" patches:
  usually the thick, dark vessels, sometimes dark background. It does not follow a line, because its
  resolution is the 16 px patch and because its question ("is there a vessel somewhere in this tile?")
  does not need a line.
- **After the cut:** fat regions around the vessels the classifier found most convincing, with the thin
  ones mostly missing. SAM's guess fails differently, as one big region with a careful edge.
- **Which is better:** neither is the vessel map. A coarse but *local* guess can still teach the head more
  than a sharp but global one, so compare the held-out lines above before you pick.
- **Other tasks:** where the class really is a blob (a defect, a nucleus) and some images contain none,
  the same recipe fits much better.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Go deeper · how the heat map is made</b></summary><p>Two cells for the curious. The first runs the four steps of Grad-CAM by hand on one training image, and
checks the result against the package. The second draws the three shapes a heat map could take (a vessel
tree, blobs, noise), to compare with the real one.</p></details></div>"""))

In [ ]:
# @title Grad-CAM step by step on one training image { display-mode: "form" }
cam_feat = std_feats[0]
with torch.enable_grad():
    act = clf.act(clf.hidden(cam_feat[None]))  # the feature maps A_k: [1, 32, h, w]
    yes_score = clf.out(act).mean((2, 3))[0, 0]  # the "vessel" score
    grads = torch.autograd.grad(yes_score, act)[0]  # how the score changes with each A_k pixel
maps_a = act[0].detach().cpu().numpy()
alpha = grads[0].mean((1, 2)).cpu().numpy()  # step 1: one weight per feature map
weighted = np.tensordot(alpha, maps_a, axes=1)  # step 2: sum of alpha_k * A_k
positive = np.maximum(weighted, 0)  # step 3: keep only the evidence FOR "vessel"
positive = positive - positive.min()
steps_heat = positive / (1e-7 + positive.max())  # step 4: rescale to [0, 1]
k_hi, k_lo = int(alpha.argmax()), int(alpha.argmin())
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(ex["image"])
axes[0].set_title("training image", fontsize=9)
axes[1].imshow(maps_a[k_hi], cmap="Greys")
axes[1].set_title(f"A_k with the largest α ({alpha[k_hi]:+.2f})", fontsize=8)
axes[2].imshow(maps_a[k_lo], cmap="Greys")
axes[2].set_title(f"A_k with the smallest α ({alpha[k_lo]:+.2f})", fontsize=8)
sorted_alpha = np.sort(alpha)
axes[3].bar(
    range(len(alpha)), sorted_alpha, color=[THEME["works"] if a > 0 else THEME["fails"] for a in sorted_alpha]
)
axes[3].set_title("1 · weights α_k (one per map)", fontsize=8)
limit = np.abs(weighted).max() + 1e-9
axes[4].imshow(weighted, cmap="PuOr", vmin=-limit, vmax=limit)
axes[4].set_title("2 · Σ α_k·A_k (purple +, orange −)", fontsize=8)
axes[5].imshow(prob_rgb(steps_heat))
axes[5].set_title("3 + 4 · ReLU, rescale: the heat map", fontsize=8)
for a in axes:
    if a is not axes[3]:
        a.axis("off")
plt.tight_layout()
plt.show()
print(
    f"{(alpha > 0).sum()} of {len(alpha)} feature maps have a positive weight · "
    f"max difference to the grad-cam package: {np.abs(steps_heat - cam_map(clf, cam_feat)).max():.4f}"
)

In [ ]:
# @title Sketch: three shapes a heat map could take { display-mode: "form" }
sketch_rng = np.random.default_rng(SEED)
tree = np.zeros((48, 48))
for x0, y0, x1, y1 in [(24, 47, 24, 18), (24, 30, 8, 8), (24, 26, 42, 6), (24, 38, 40, 30), (24, 40, 10, 32)]:
    for s in np.linspace(0, 1, 80):
        tree[int(y0 + s * (y1 - y0)), int(x0 + s * (x1 - x0))] = 1
tree = ndi.gaussian_filter(tree, 0.8)
yy, xx = np.mgrid[0:48, 0:48]
centres = [(14, 14), (32, 28), (18, 38)]
blobs = sum(np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * 5**2)) for cx, cy in centres)
noise = ndi.gaussian_filter(sketch_rng.random((48, 48)), 0.7)
shapes = [("vessel tree: thin lines", tree), ("blobs: a few patches", blobs), ("noise: no structure", noise)]
show_masks(
    [(title, prob_rgb(0.5 + 0.5 * (m - m.min()) / (m.max() - m.min()))) for title, m in shapes],
    width=2.0,
)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Question (optional)</b><p>Run the next cell to use the heat map as a SAM prompt instead of a mask.</p>
<p><strong>Q23.</strong> 🔴 Instead of cutting the heat map, let it pick SAM clicks: the hottest patches as "vessel", the
coldest as "not vessel" (the trick of section 4.2, with Grad-CAM in place of the prototype). Do SAM's
masks from these clicks look more like the SAM pseudo-labels of 6.1 or like the Grad-CAM blobs?</p>
<pre><code class="language-python">N_CAM_CLICKS = 5   # try 2, 10
</code></pre></div>"""))

This cell picks SAM clicks on each training image's heat map, asks SAM for a mask and scores the three kinds
of pseudo-label on the training images (diagnosis only, no training).

In [ ]:
# Q23 — Grad-CAM as a SAM prompt: hottest patches = vessel clicks, coldest = background clicks
N_CAM_CLICKS = 5  # try 2, 10
if not isinstance(N_CAM_CLICKS, int) or N_CAM_CLICKS < 1:
    raise SystemExit("N_CAM_CLICKS must be a whole number of at least 1.")
cam_sam_masks, cam_clicks = [], []
for it, f in zip(scrib_train, std_feats, strict=True):
    s = torch.from_numpy(cam_map(clf, f)).to(DEVICE)
    keep = patch_share(region(it), tuple(s.shape))
    pos = top_points(s, keep, N_CAM_CLICKS, it["mask"].shape, largest=True)
    neg = top_points(s, keep, N_CAM_CLICKS, it["mask"].shape, largest=False)
    cam_sam_masks.append(
        sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg)) >= 0.5
    )
    cam_clicks.append(pos)
cam_sam_quality = mean_scores([score(m, it) for m, it in zip(cam_sam_masks, scrib_train, strict=True)])
RESULTS["gradcam_sam_prompt_iou_fit"] = cam_sam_quality["iou"]
print("pseudo-labels on the training images (diagnosis only: scored against the hidden truth)")
for name, q in {
    "SAM from stroke clicks (6.1, at 0.5)": sam_quality,
    f"Grad-CAM cut at {CAM_THRESHOLD}": cam_quality,
    f"SAM from {N_CAM_CLICKS} + {N_CAM_CLICKS} Grad-CAM clicks": cam_sam_quality,
}.items():
    print(f"{name:<44} IoU {q['iou']:.3f} · clDice {q['cldice']:.3f}")
show_clicks(
    [
        ("Grad-CAM clicks: vessel", ex["image"], cam_clicks[0]),
        ("SAM from Grad-CAM clicks vs truth", error_map(ex["image"], cam_sam_masks[0], ex), cam_clicks[0]),
    ],
    width=3.4,
)

<details><summary>What to look for</summary>

- The clicks land where the heat map is hottest, usually on or next to a thick vessel. SAM's mask then
  looks like its masks of 6.1: sharp edges around whatever object contains the clicks, often a region
  rather than a line.
- The prompt route swaps Grad-CAM's coarse blobs for SAM's sharp edges, but it brings back SAM's idea of an
  object. A click that lands on dark background drags the mask there.
- Two weak signals combined are not automatically a strong one: their mistakes can add up as easily as
  cancel out.

</details>

### 6.4 Path 2, side by side

This cell puts the scribble head of section 4 and the SAM and Grad-CAM pseudo-label heads side by side
(bars in the approach colours, error maps on the held-out demo image) and writes the annotation effort
for section 8: the same strokes for all three.

In [ ]:
# @title Path 2 side by side: table, bars and error maps { display-mode: "form" }
RESULTS["path2_effort_min"] = {
    k: round(effort_minutes(n_strokes=n_strokes), 2) for k in ("sam_pseudo", "gradcam_pseudo")
}
print_path2(path2_report)
print(
    f"{'everything is foreground':<46}{RESULTS['specialised_everything_iou']:>6.3f}\n"
    f"annotation for all three: {n_strokes} strokes on {len(scrib_train)} images "
    f"(≈ {effort_minutes(n_strokes=n_strokes):.1f} min); the pseudo-labels add compute, not annotation"
)
metrics = ["iou", "cldice", "recall", "precision"]
x, width = np.arange(len(metrics)), 0.8 / len(path2_report)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for n, (k, v) in enumerate(path2_report.items()):
    shift = (n - (len(path2_report) - 1) / 2) * width
    ax.bar(x + shift, [v[m] for m in metrics], width, color=APPROACH_COLOURS[k], label=APPROACH_NAMES[k])
ax.axhline(
    RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--", label='IoU of "everything"'
)
ax.set_xticks(x, ["IoU", "clDice", "recall", "precision"])
ax.set_ylim(0, 1)
ax.set_title("Path 2: same strokes, three ways to fill in the rest")
ax.legend(fontsize=7, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()
show_masks(
    [
        (
            f"{APPROACH_NAMES[k].split(':')[-1].strip()} · IoU {score(p[k_demo], demo)['iou']:.2f}",
            error_map(demo["image"], p[k_demo], demo),
        )
        for k, p in path2_preds.items()
    ]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]))],
    width=2.8,
)

**Q24.** 🟡 All three heads cost the same strokes. You can add one more afternoon (about 4 hours) of work.
Suppose a pilot (invented numbers) says:

- **A, more strokes:** a second round on the same five images would lift the scribble head's held-out IoU
  by about 0.04.
- **B, a better teacher:** a SAM checkpoint fine-tuned on retinas would lift the pseudo-labels' IoU on the
  training images from 0.45 to 0.65, and nobody has measured the head yet.

Which would you pick? What does the table suggest, and what would you want to measure first?

<details><summary>A possible answer</summary>

- **Compare like with like:** A's +0.04 is on the head itself, B's +0.20 is on the pseudo-labels only. A
  better teacher helps the head only as far as its mistakes stop being systematic.
- **The cap:** a head trained on pseudo-labels is capped by how well the second model understands *your*
  structures. SAM thinks in objects and Grad-CAM in tiles, and neither thinks in thin lines, so better
  pseudo-labels need a better teacher for this domain, not a cleverer threshold.
- **More strokes:** they add true information, but no extra coverage between them.
- **How to choose:** if the pseudo-labels are close to the truth on a few fully labelled images, a better
  teacher (or a cleaned-up pseudo-label) is the lever. If they are far, spend the afternoon on strokes.
- **First measurement:** the diagnosis line of 6.1 on a handful of images you labelled completely.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway: Path 2</b><ul>
<li><strong>You are about to train on a teacher's guesses</strong> → score them on a few fully labelled images first, and
  keep the person's strokes stronger than the teacher.</li>
<li><strong>Its probabilities crowd around 0.5</strong> → do not trust a hard cut. Tune the threshold on training images
  and ignore the doubtful band instead of guessing.</li>
<li><strong>Every image has the structure, so there is no "no" class</strong> → Grad-CAM is a poor fit. Use it when you
  have real image-level labels (defects, tumours).</li>
<li><strong>You need more labels but cannot pay for strokes</strong> → pseudo-labels add compute, not annotation, and
  bring the teacher's bias with them.</li>
</ul></div>"""))

## 7. Path 3 · Skip specialist training

Paths 1 and 2 trained a head once from strokes on five images, and that head then segmented every new image
on its own. Path 3 trains nothing: the strokes a person draws on an image become SAM's prompts on *that
same image*, and SAM's answer is the result.

It is the difference between teaching an intern once and pointing at every image yourself: no training time
and no training data, but the pointing never stops.

Section 2 gave SAM clicks on the vessels and one big box. Here we add what it lacked: "not here" clicks from
the background strokes, and many small boxes, one per stroke.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>Prompt the foundation model directly with scribble points and boxes, no training, and know when that beats
training a head.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: what a SAM prompt says</b><p>Sections 2 and 6.1 showed that forty scattered clicks ask SAM for one object, and it answers with a region.
What is new here:</p>
<ul>
<li>Split the prompt into many small ones, each its own object (pets: one dog, one mask), and combine the
  masks. SAM then answers many local questions instead of one global one.</li>
<li>Nothing is trained. The image encoder runs once per image, and every extra prompt costs only a fast
  decoder pass.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 SAM papers</b></summary><ul>
<li>SAM, Kirillov et al., ICCV 2023: <a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/2304.02643">arXiv:2304.02643</a></li>
<li>SAM 2, Ravi et al. 2024: <a style="color:inherit;text-decoration:underline" href="https://arxiv.org/abs/2408.00714">arXiv:2408.00714</a></li>
</ul></details></div>"""))

This helper cell adds a colour for each direct approach and defines the prompts we take from the strokes:
the separate strokes of an image, a click at the centre of a stroke, SAM on many (box, click) objects at
once, the two direct recipes on a list of images, and a figure that draws boxes.

In [ ]:
# @title Path 3 helpers: colours, strokes, SAM on many prompts, the two recipes { display-mode: "form" }
from matplotlib.patches import Rectangle

APPROACH_COLOURS |= {
    "sam_clicks": "#546E7A",
    "sam_boxes": "#880E4F",
}
APPROACH_NAMES |= {
    "sam_clicks": "direct: SAM from stroke clicks",
    "sam_boxes": "direct: SAM, one box + click per stroke",
}


def stroke_masks(item: dict, value: int = 1) -> list[np.ndarray]:
    """The separate strokes of one kind (1 vessel, 2 background) as boolean masks: the 8-connected pieces
    of the scribble map."""
    lab, n = ndi.label(item["scribbles"] == value, structure=np.ones((3, 3)))
    return [lab == k for k in range(1, n + 1)]


def stroke_click(stroke: np.ndarray) -> list[float]:
    """(x, y) of the stroke pixel nearest the stroke's centre: a click that is always on the stroke."""
    ys, xs = np.nonzero(stroke)
    k = int(np.argmin((xs - xs.mean()) ** 2 + (ys - ys.mean()) ** 2))
    return [float(xs[k]), float(ys[k])]


@torch.no_grad()
def sam_many(image: np.ndarray, boxes: list[list[float]], clicks: list[list[float]] | None = None, chunk=32):
    """SAM with one object per box (plus one positive click inside it, if given), decoded in chunks after a
    single image encoding. Returns the highest probability over the objects: a pixel is foreground as soon
    as one object covers it."""
    best = np.zeros(image.shape[:2], np.float32)
    for s in range(0, len(boxes), chunk):
        kw = {"input_boxes": [boxes[s : s + chunk]]}
        if clicks is not None:
            c = clicks[s : s + chunk]
            kw |= {"input_points": [[[p] for p in c]], "input_labels": [[[1]] * len(c)]}
        inputs = sam.proc(images=image, return_tensors="pt", **kw)
        if sam._cache is None or sam._cache[0] is not image:  # the same image cache as Sam.probability
            sam._cache = (image, sam.model.get_image_embeddings(inputs["pixel_values"].to(sam.device)))
        out = sam.model(
            image_embeddings=sam._cache[1],
            input_points=inputs["input_points"].to(sam.device) if clicks is not None else None,
            input_labels=inputs["input_labels"].to(sam.device) if clicks is not None else None,
            input_boxes=inputs["input_boxes"].to(sam.device),
            multimask_output=False,
        )
        logits = sam.proc.post_process_masks(out.pred_masks.cpu(), inputs["original_sizes"], binarize=False)[
            0
        ]
        best = np.maximum(best, torch.sigmoid(logits[:, 0]).amax(0).numpy())
    return best


def direct_clicks(items: list[dict], n: int) -> tuple[list[np.ndarray], list[tuple]]:
    """Recipe 1: n vessel clicks and n background clicks spread along the strokes of each image, one SAM
    prompt per image. Returns the masks and each image's (vessel, background) clicks."""
    preds, clicks = [], []
    for it in items:
        pos, neg = stroke_points(it, 1, n), stroke_points(it, 2, n)
        preds.append(
            sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg)) >= 0.5
        )
        clicks.append((pos, neg))
    return preds, clicks


def direct_boxes(items: list[dict], pad: int, with_click: bool) -> tuple[list[np.ndarray], list[tuple]]:
    """Recipe 2: one box around every vessel stroke (grown by `pad` px; `stroke_box` is set in 7.2), plus a
    click at the stroke's centre if `with_click`; the masks of all strokes are combined. Returns the masks
    and each image's (boxes, clicks)."""
    preds, prompts = [], []
    for it in items:
        strokes = stroke_masks(it, 1)
        boxes = [[float(v) for v in stroke_box(s, pad)] for s in strokes]
        clicks = [stroke_click(s) for s in strokes] if with_click else None
        preds.append(
            sam_many(it["image"], boxes, clicks) >= 0.5 if boxes else np.zeros(it["mask"].shape, bool)
        )
        prompts.append((boxes, clicks))
    return preds, prompts


def show_boxes(panels: list[tuple[str, np.ndarray, list | None]], width: float = 3.2):
    """One row of panels (title, RGB image, boxes or None); boxes are drawn as teal rectangles."""
    fig, axes = plt.subplots(1, len(panels), figsize=(width * len(panels), width))
    for ax, (title, img, boxes) in zip(np.atleast_1d(axes), panels, strict=True):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
        for x0, y0, x1, y1 in boxes or []:
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=1, color=THEME["scribble"]))
    plt.tight_layout()
    plt.show()

### 7.1 Clicks on the vessels, and "not here"

The news is **fairness**: the heads of sections 4 to 6 never saw a stroke on the held-out images, but Path 3
gets strokes on every one of them. It gets more help per image, and pays for it at use time.

The first recipe is section 6.1 without the head: twenty clicks along the vessel strokes and twenty along
the background strokes of each *held-out* image, and SAM's mask is the answer. The next cell prints that
one row.

In [ ]:
# @title SAM from twenty vessel and twenty background clicks per image { display-mode: "form" }
N_DIRECT_CLICKS = 20  # clicks per kind (vessel / background) and image, taken from the strokes
t0 = time.monotonic()
path3_preds = {"zero_shot": ladder_preds["zero_shot"], "scribbles": ladder_preds["scribbles"]}
path3_preds["sam_clicks"], click_prompts = direct_clicks(SPEC_EVAL, N_DIRECT_CLICKS)
clicks_sec = round(time.monotonic() - t0, 1)
path3_report = {k: rung_report(v) for k, v in path3_preds.items()}
print(
    f"SAM from {N_DIRECT_CLICKS} + {N_DIRECT_CLICKS} stroke clicks on {len(SPEC_EVAL)} images · "
    f"{clicks_sec} s on {DEVICE}"
)
print_path2({"sam_clicks": path3_report["sam_clicks"]})
print(f"{'section 2: SAM, 20 vessel clicks only':<46}{RESULTS['sam_prompt_ladder']['20 points']:>6.3f}")

**Q25.** 🔴 (optional) Did the "not here" clicks pull SAM's mask onto the vessels (compare the row with
section 2's vessel-only clicks)? Change the number of clicks per kind and run the next cell: does SAM do
better with fewer clicks or with more? Why would that be, given that every prompt set describes one object?

```python
N_CLICKS_TRY = 5   # try 1, 5, 50
```

This cell runs the same recipe with your number of clicks on every evaluation image and prints it under the
default of 20.

In [ ]:
# Q25 — fewer or more stroke clicks for SAM
N_CLICKS_TRY = 5  # try 1, 5, 50
if not isinstance(N_CLICKS_TRY, int) or N_CLICKS_TRY < 1:
    raise SystemExit("N_CLICKS_TRY must be a whole number of at least 1.")
try_preds, _ = direct_clicks(SPEC_EVAL, N_CLICKS_TRY)
RESULTS["sam_direct_clicks_try"] = rung_report(try_preds) | {"n_clicks": N_CLICKS_TRY}
print_path2(
    {
        f"{N_DIRECT_CLICKS} + {N_DIRECT_CLICKS} clicks (default)": path3_report["sam_clicks"],
        f"{N_CLICKS_TRY} + {N_CLICKS_TRY} clicks (yours)": RESULTS["sam_direct_clicks_try"],
    }
)

<details><summary>What to look for</summary>

SAM looks for one object that contains every positive click and none of the negative ones.

- A vessel tree is one connected structure, but it is a few pixels wide and spread over the whole crop. The
  simplest object that contains clicks all over it is a large region around it.
- The negative clicks, which also sit all over the crop, mostly cut holes in that region rather than
  turning it into a line.
- A few clicks can leave SAM a more local object to settle on. Many clicks push it towards "the whole
  fundus minus a few holes". The marked column tells you which happened on your run.
- More prompts in *one* set are not more information to SAM if they describe something it does not see as
  one object.

</details>

### 7.2 One small box per stroke

The second recipe splits the question:

1. Each vessel stroke gets its own box, a few pixels larger than the stroke, and a click at its centre.
2. SAM answers each box as a separate object.
3. The vessel map is the union of all the answers.

One stroke is a short piece of one vessel, so each box asks a local question SAM can answer: "the thing
along this line, inside this rectangle".

This helper cell defines `stroke_box`, the box around one stroke: the min and max of its columns and rows,
grown by a margin and clipped to the image.

In [ ]:
# @title stroke_box: the box around one stroke { display-mode: "form" }
def stroke_box(stroke: np.ndarray, pad: int) -> list[float]:
    """The box [x0, y0, x1, y1] around the True pixels of `stroke`, grown by `pad` pixels on every side and
    clipped to the image: 0 <= x <= width - 1, 0 <= y <= height - 1. np.nonzero returns rows (y) first."""
    ys, xs = np.nonzero(stroke)
    h, w = stroke.shape
    return [
        float(max(xs.min() - pad, 0)),
        float(max(ys.min() - pad, 0)),
        float(min(xs.max() + pad, w - 1)),
        float(min(ys.max() + pad, h - 1)),
    ]

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction</b><p>The strokes cover only some of the vessels (section 4 dropped 70 % of the segments), and each box sits on a
stroke. Two guesses:</p>
<ul>
<li>Will this recipe mark more of the fundus than the clicks of 7.1, or less?</li>
<li>Will it beat simply thickening the strokes by a few pixels, with no model at all?</li>
</ul></div>"""))

This cell asks SAM for one object per vessel stroke (a box with 4 px of margin plus a click at the
centre) on every evaluation image and prints it next to the strokes thickened by 3 px, the "no model"
reference.

In [ ]:
# @title SAM with one box and one click per stroke, next to the thickened strokes { display-mode: "form" }
BOX_PAD = 1 if SMOKE else 4  # px around each stroke (smoke thumbnails are 4× smaller)
THICKEN = 1 if SMOKE else 3  # px: the "no model" reference, the vessel strokes thickened
t0 = time.monotonic()
path3_preds["sam_boxes"], box_prompts = direct_boxes(SPEC_EVAL, BOX_PAD, with_click=True)
boxes_sec = round(time.monotonic() - t0, 1)
path3_report["sam_boxes"] = rung_report(path3_preds["sam_boxes"])
strokes_alone = [ndi.binary_dilation(it["scribbles"] == 1, iterations=THICKEN) for it in SPEC_EVAL]
strokes_report = rung_report(strokes_alone)
n_boxes = [len(b) for b, _ in box_prompts]
print(
    f"SAM, one box + click per vessel stroke · {int(np.mean(n_boxes))} boxes per image on average · "
    f"{boxes_sec} s on {DEVICE}"
)
print_path2(path3_report | {f"no model: the strokes thickened by {THICKEN} px": strokes_report})
boxes, _ = box_prompts[k_demo]
show_boxes(
    [
        (f"{demo['name']} · {len(boxes)} stroke boxes", demo["image"], boxes),
        (
            "SAM, one object per box vs truth",
            error_map(demo["image"], path3_preds["sam_boxes"][k_demo], demo),
            None,
        ),
        ("strokes thickened vs truth", error_map(demo["image"], strokes_alone[k_demo], demo), None),
    ],
    width=3.0,
)

Is the demo image typical? This cell scores the box recipe on each evaluation image and shows the worst,
median and best side by side.

In [ ]:
# @title Box recipe: worst, typical and best evaluation image { display-mode: "form" }
box_iou = [score(p, it)["iou"] for p, it in zip(path3_preds["sam_boxes"], SPEC_EVAL, strict=True)]
rank_box = np.argsort(box_iou)
mid_box = rank_box[len(rank_box) // 2]
box_picks = [("worst", rank_box[0]), ("typical (median)", mid_box), ("best", rank_box[-1])]
show_masks(
    [
        (
            f"{label} · IoU {box_iou[k]:.2f}",
            error_map(SPEC_EVAL[k]["image"], path3_preds["sam_boxes"][k], SPEC_EVAL[k]),
        )
        for label, k in box_picks
    ],
    width=3.0,
)

**Q26.** 🟡 Compare the box recipe with the clicks of 7.1 and with the thickened strokes: which column
moved, recall or precision? Then grow the boxes and drop the centre click in the next cell. What does a
larger box let SAM find, and what does it cost?

```python
BOX_PAD_TRY = 16      # slider: try 0, 16, 32 (px of the 1024 px crop)
WITH_CLICK_TRY = True  # tick box: untick for the box alone
```

This cell runs the box recipe with your margin and click setting on every evaluation image and prints it
under the default.

In [ ]:
# Q26 — larger or smaller boxes, with or without the centre click
BOX_PAD_TRY = 16  # @param {type:"slider", min:0, max:48, step:8}
WITH_CLICK_TRY = True  # @param {type:"boolean"}
if not isinstance(BOX_PAD_TRY, int) or BOX_PAD_TRY < 0 or not isinstance(WITH_CLICK_TRY, bool):
    raise SystemExit("BOX_PAD_TRY must be a whole number ≥ 0 and WITH_CLICK_TRY True or False.")
pad_try = max(BOX_PAD_TRY // 4, 0) if SMOKE else BOX_PAD_TRY  # smoke thumbnails are 4× smaller
try_preds, _ = direct_boxes(SPEC_EVAL, pad_try, WITH_CLICK_TRY)
RESULTS["sam_direct_boxes_try"] = rung_report(try_preds) | {"pad": BOX_PAD_TRY, "with_click": WITH_CLICK_TRY}
print_path2(
    {
        f"pad {BOX_PAD} px, with click (default)": path3_report["sam_boxes"],
        f"pad {BOX_PAD_TRY} px, {'with' if WITH_CLICK_TRY else 'no'} click (yours)": RESULTS[
            "sam_direct_boxes_try"
        ],
    }
)

<details><summary>What to look for</summary>

- **Recall is capped.** Each box finds only what lies inside it, so recall is limited by how much of the
  tree the strokes touch: the vessels a person stroked, plus whatever branches cross their boxes. That is
  why the box recipe marks a small share of the fundus, unlike the clicks of 7.1.
- **A larger box** reaches more branches (recall can rise) but also more background that SAM may take for
  the object (precision falls). At some size it turns back into section 2's big box.
- **The centre click** tells SAM which thing inside the box is meant. It helps most when the box holds more
  than one candidate.
- **The baseline:** if the thickened strokes come close to or beat SAM here, that is a real finding, not a
  bug. On thin structures a cheap baseline built from the hints alone is the first thing to compare against.

</details>

### 7.3 Path 3, side by side

This cell puts the two direct recipes next to the section-2 box, the scribble head and the SAM
pseudo-label head (bars and error maps on the demo image), and writes the numbers section 8 compares: no
training time, but annotation on *every* image you segment, at 2 s per stroke.

In [ ]:
# @title Path 3 side by side: table, annotation cost, bars and error maps { display-mode: "form" }
path3_preds["sam_pseudo"] = path2_preds["sam_pseudo"]
path3_report["sam_pseudo"] = path2_report["sam_pseudo"]
eval_meta = SPECIALISED["eval"]
strokes_per_image = {
    "sam_clicks": float(np.mean([m.get("n_fg_strokes", 0) + m.get("n_bg_strokes", 0) for m in eval_meta])),
    "sam_boxes": float(np.mean([m.get("n_fg_strokes", 0) for m in eval_meta])),  # vessel strokes only
}
effort = {k: effort_minutes(n_strokes=round(v * len(eval_meta))) for k, v in strokes_per_image.items()}
sec = {"sam_clicks": clicks_sec, "sam_boxes": boxes_sec}
result_key = {"sam_clicks": "sam_direct_clicks", "sam_boxes": "sam_direct_boxes"}
for k in ("sam_clicks", "sam_boxes"):
    per_image = [score(p, it)["iou"] for p, it in zip(path3_preds[k], SPEC_EVAL, strict=True)]
    RESULTS[result_key[k]] = path3_report[k] | {
        "iou_std": float(np.std(per_image)),
        "train_sec": 0.0,
        "infer_sec": sec[k],
        "n_images": len(SPEC_EVAL),
        "strokes_per_image": strokes_per_image[k],
        "effort_min_per_image": round(effort_minutes(n_strokes=1) * strokes_per_image[k], 2),
        "effort_min": round(effort[k], 2),
    }
RESULTS["sam_direct_boxes"] |= {"pad": BOX_PAD, "with_click": True}
RESULTS["sam_direct_clicks"] |= {"n_clicks": N_DIRECT_CLICKS}
RESULTS["sam_direct_iou"] = path3_report["sam_boxes"]["iou"]  # headline: points + boxes (SPEC §6, 07)
RESULTS["sam_direct_clicks_iou"] = path3_report["sam_clicks"]["iou"]
RESULTS["sam_direct_strokes_alone"] = strokes_report | {"thicken_px": THICKEN}
RESULTS["path3_effort_min"] = {k: round(v, 2) for k, v in effort.items()}

order = ["zero_shot", "scribbles", "sam_pseudo", "sam_clicks", "sam_boxes"]
print_path2({k: path3_report[k] for k in order})
print(
    f"{'no model: strokes thickened':<46}{strokes_report['iou']:>6.3f}\n"
    f"{'everything is foreground':<46}{RESULTS['specialised_everything_iou']:>6.3f}"
)
print(
    f"\nannotation · the heads: {n_strokes} strokes on {len(scrib_train)} training images, once "
    f"(≈ {effort_minutes(n_strokes=n_strokes):.1f} min)"
)
for k in ("sam_clicks", "sam_boxes"):
    print(
        f"annotation · {APPROACH_NAMES[k]}: ≈ {RESULTS[result_key[k]]['effort_min_per_image']:.1f} min on "
        f"every image you segment ({effort[k]:.0f} min for these {len(SPEC_EVAL)})"
    )
metrics = ["iou", "cldice", "recall", "precision"]
x, width = np.arange(len(metrics)), 0.8 / len(order)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
for n, k in enumerate(order):
    shift = (n - (len(order) - 1) / 2) * width
    ax.bar(
        x + shift,
        [path3_report[k][m] for m in metrics],
        width,
        color=APPROACH_COLOURS[k],
        label=APPROACH_NAMES[k],
    )
ax.axhline(
    RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--", label='IoU of "everything"'
)
ax.set_xticks(x, ["IoU", "clDice", "recall", "precision"])
ax.set_ylim(0, 1)
ax.set_title("Path 3: SAM prompted directly, next to the trained heads")
ax.legend(fontsize=7, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()
show_masks(
    [
        (
            f"{APPROACH_NAMES[k].split(':')[-1].strip()} · "
            f"IoU {score(path3_preds[k][k_demo], demo)['iou']:.2f}",
            error_map(demo["image"], path3_preds[k][k_demo], demo),
        )
        for k in order
    ]
    + [("ground truth", overlay(demo["image"], demo["mask"], THEME["truth"]))],
    width=2.6,
)

### 7.4 🔴 Optional: SAM 3 from a word

SAM 3 segments every instance of a typed noun phrase, with no click at all. Nothing later depends on it.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 About SAM 3</b></summary><p>SAM 3 (Meta, 2025) adds <em>concept</em> prompts: type a short noun phrase and it segments every instance of it.</p>
<ul>
<li>It is gated on Hugging Face: accept the terms on its model page with your account to use it.</li>
<li>The workshop never requires it. <code>setup_check.ipynb</code> caches it only if your account has access.</li>
<li>If it is not in your cache, the next cell says so and stops.</li>
<li>The cell loads it from your cache only (never from the internet), in fp16 on a GPU, runs your phrase on
  every evaluation image and keeps the union of the instances it finds.</li>
</ul></details></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Question (optional)</b><p>Run the next cell if you have SAM 3, then try other words by editing <code>SAM3_TEXT</code> in its code (open the
collapsed cell).</p>
<p><strong>Q27.</strong> 🔴 Does a word find the vessels better than your strokes did? Try a few phrasings
(<code>"blood vessel"</code>, <code>"vessel"</code>, <code>"artery"</code>, <code>"dark line"</code>, <code>"retina"</code>). Which words work, and what does that
tell you about the images SAM 3 learnt its concepts from?</p></div>"""))

In [ ]:
# @title Q27 — 🔴 SAM 3 from a text prompt (optional, only if SAM 3 is in your cache) { display-mode: "form" }
SAM3_TEXT = "blood vessel"  # @param {type:"string"}
SAM3_ID, SAM3_LICENCE = "facebook/sam3", "SAM License (Meta)"
if not isinstance(SAM3_TEXT, str) or not SAM3_TEXT.strip():
    raise SystemExit('SAM3_TEXT must be a short phrase in quotes, e.g. "blood vessel".')
RESULTS["sam3_text_iou"] = None
sam3 = None
try:
    from transformers import Sam3Model, Sam3Processor

    sam3_dtype = torch.float16 if DEVICE == "cuda" else torch.float32  # fp16 on a T4, never bf16 (SPEC §5)
    sam3_src = str(model_folder(SAM3_ID) or SAM3_ID)  # plain files on Drive (setup_check), else the hub id
    sam3_proc = Sam3Processor.from_pretrained(sam3_src, local_files_only=True)
    sam3 = Sam3Model.from_pretrained(sam3_src, local_files_only=True, dtype=sam3_dtype).eval().to(DEVICE)
    print(f"SAM 3 · source {SAM3_ID} · licence {SAM3_LICENCE} · {str(sam3_dtype).split('.')[-1]} on {DEVICE}")
except Exception as err:  # noqa: BLE001 - not cached, no access, an older transformers: all mean "skip"
    RESULTS["sam3_note"] = f"SAM 3 not available in this session ({type(err).__name__}); it is optional"
    print(
        "ℹ️ SAM 3 is not in your cache, so this optional cell stops here. To try it later: request access on "
        f"huggingface.co/{SAM3_ID}, then run setup_check.ipynb with your HF token. Nothing else needs it."
    )
if sam3 is not None:
    try:
        t0, sam3_preds = time.monotonic(), []
        for it in SPEC_EVAL:
            inputs = sam3_proc(images=it["image"], text=SAM3_TEXT.strip(), return_tensors="pt").to(DEVICE)
            inputs["pixel_values"] = inputs["pixel_values"].to(sam3_dtype)
            with torch.no_grad():
                out = sam3(**inputs)
            found = sam3_proc.post_process_instance_segmentation(
                out, threshold=0.3, mask_threshold=0.5, target_sizes=[it["mask"].shape]
            )[0]
            masks = found["masks"].bool().cpu().numpy()
            sam3_preds.append(masks.any(0) if len(masks) else np.zeros(it["mask"].shape, bool))
        sam3_report = rung_report(sam3_preds)
        RESULTS["sam3_text_iou"] = sam3_report["iou"]
        RESULTS["sam3_text"] = sam3_report | {"text": SAM3_TEXT, "infer_sec": round(time.monotonic() - t0, 1)}
        print_path2(
            {
                f'SAM 3, "{SAM3_TEXT}"': sam3_report,
                "SAM 2.1, one box + click per stroke": path3_report["sam_boxes"],
            }
        )
        show_masks(
            [
                (f'SAM 3, "{SAM3_TEXT}" vs truth', error_map(demo["image"], sam3_preds[k_demo], demo)),
                (
                    "SAM 2.1 from stroke boxes vs truth",
                    error_map(demo["image"], path3_preds["sam_boxes"][k_demo], demo),
                ),
            ],
            width=3.2,
        )
    except Exception as err:  # noqa: BLE001 - an optional extension must never stop the notebook
        RESULTS["sam3_note"] = f"SAM 3 loaded but failed: {type(err).__name__}: {err}"
        print(f"⚠️ SAM 3 loaded but did not run here ({type(err).__name__}: {err}). It is optional; carry on.")

<details><summary>What to look for</summary>

- **Only known concepts work.** A concept prompt works for concepts SAM 3 met in training, named the way
  its captions named them. "Blood vessel" in a fundus photograph is a rare, specialist phrase next to "dog"
  or "car", so expect thick vessels, a blob, or nothing, and a lot of change with the wording.
- **Appearance beats meaning.** Generic words like "dark line" sometimes do better than the correct medical
  term, which hints that the model matches appearance more than meaning.
- **It is free.** A word costs no annotation, so even a weak result is a free starting point for the
  strokes of 7.2.

</details>

Want to try this on a photo of your own? Section 9 lets you draw the strokes yourself.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ Takeaway: Path 3</b><ul>
<li><strong>Your structure is thin and spread over the image</strong> → more clicks only make a bigger region: split the
  prompt into one small box and a centre click per stroke.</li>
<li><strong>Few images, or each one matters</strong> → prompt SAM directly and review every mask.</li>
<li><strong>Images keep coming</strong> → train a head once (Path 1 or 2); direct prompting pays strokes on every image.</li>
</ul></div>"""))

## 8. Compare everything

Every section since 2 ended with its own small table, each against a different neighbour. Here they all
sit in one table, scored on the same held-out images against the same hidden ground truth.

The question becomes the one you face on a real project: not "which approach has the highest number" but
"which approach would I pay for". A recipe can win on accuracy and lose on annotation hours, or win on
average and fail badly on one image in five.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>fill the comparison table, predicting the winner first.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: the five columns</b><p>Each row of the table is read through five columns:</p>
<ul>
<li><strong>Accuracy</strong>: mean IoU over the held-out images (0 = no overlap with the truth, 1 = perfect), plus
  clDice (does the mask follow the vessel centre lines?).</li>
<li><strong>Annotation effort</strong>: what a person had to draw, in minutes. A stroke or a box is seconds, a complete
  vessel map is hours.</li>
<li><strong>Reliability, or spread</strong>: the standard deviation of the per-image IoU, i.e. how far a typical image
  sits from the average. IoUs of 0.60, 0.62, 0.58 give a spread of 0.02: trust the average. IoUs of
  0.90, 0.30, 0.60 have the same average but a spread of 0.25: some images fail badly.</li>
<li><strong>Compute</strong>: seconds of training plus inference in this session.</li>
<li><strong>Scalability</strong>: a sentence, not a number. What does the next 1,000 images cost?</li>
</ul>
<p>The table also shows <strong>recall</strong> (the share of the true vessel pixels the mask finds) and <strong>worst</strong> (the
lowest IoU on any single image).</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFF8E1;border-left:6px solid #5D4037;border-radius:6px;padding:12px 14px;margin:6px 0;color:#5D4037;font-family:inherit;line-height:1.5"><b>🧾 Assumptions behind the minutes</b><ul>
<li>A scribble stroke, a click or a box takes about <strong>2 seconds</strong>.</li>
<li>A complete vessel map of one fundus image takes <strong>3 to 5 hours</strong> (FIVES paper, 2022). We count the
  midpoint, 4 hours, and a half-painted map as half of that.</li>
<li>Both numbers come from the dataset config, not from a measurement in this room. Change them there and
  every row moves.</li>
<li>Training minutes are paid once. The per-image column is what every <em>new</em> image costs.</li>
</ul></div>"""))

This helper cell collects each approach's held-out predictions, numbers, seconds, annotation minutes and a
sentence on how it scales.

In [ ]:
# @title Comparison helpers: every approach, its predictions, seconds and effort { display-mode: "form" }
APPROACH_COLOURS |= {"full_masks": THEME["works"]}
APPROACH_NAMES |= {"full_masks": "full masks: complete maps (section 2)"}
ALL_APPROACHES = [  # the short names Q28 accepts
    "zero_shot",
    "one_shot",
    "one_shot_sam",
    "scribbles",
    "naive",
    "masked",
    "pu",
    "sam_pseudo",
    "gradcam_pseudo",
    "sam_clicks",
    "sam_boxes",
    "full_masks",
]
COMPARE_LABEL = {  # readable names for figure titles and the table
    "zero_shot": "zero-shot box",
    "one_shot": "prototype",
    "one_shot_sam": "prototype → SAM",
    "scribbles": "scribble head",
    "naive": "naive (gaps = bg)",
    "masked": "masked loss",
    "pu": "PU loss",
    "sam_pseudo": "SAM pseudo-labels",
    "gradcam_pseudo": "Grad-CAM labels",
    "sam_clicks": "SAM from clicks",
    "sam_boxes": "SAM from boxes",
    "full_masks": "full masks",
}
# A 🔴 part you skipped must not break this section: its row is left out, with one note.
SKIPPED = {}
if "one_shot_sam" not in ladder_report or "proto_sam_sec" not in globals():
    SKIPPED["one_shot_sam"] = "the 'prototype → SAM clicks' cell of section 4 was not run"
if APPROACH_NAMES["pu"] not in path_reports or "pu_head" not in RESULTS:
    SKIPPED["pu"] = "the PU cell of section 5.3 was not run"
COMPARE_ORDER = [k for k in ALL_APPROACHES if k not in SKIPPED]
COMPARE_REPORTS = (  # mean IoU, clDice, recall, precision, marked share and held-out predictions per approach
    {k: ladder_report[k] | {"held_preds": ladder_preds[k]} for k in ladder_report}
    | {k: path_reports[APPROACH_NAMES[k]] for k in ("naive", "masked", "pu") if k not in SKIPPED}
    | {k: path2_report[k] | {"held_preds": path2_preds[k]} for k in ("sam_pseudo", "gradcam_pseudo")}
    | {k: path3_report[k] | {"held_preds": path3_preds[k]} for k in ("sam_clicks", "sam_boxes")}
    | {"full_masks": path_reports["full masks (section 2)"]}
)
COMPARE_SECONDS = {  # training + pseudo-labels + inference, as recorded by each section in this session
    "zero_shot": RESULTS["specialised_zero_shot_seconds"] / len(LADDER),  # the box: 1 of section 2's prompts
    "one_shot": proto_sec,
    "full_masks": spec_heads[side][1] + RESULTS["scribbles_head"]["infer_sec"],  # same head, same features
} | {
    k: sum(float(RESULTS[entry].get(part, 0.0)) for part in ("train_sec", "pseudo_sec", "infer_sec"))
    for k, entry in {
        "scribbles": "scribbles_head",
        "naive": "naive_head",
        "masked": "masked_head",
        "pu": "pu_head",
        "sam_pseudo": "sam_pseudo_head",
        "gradcam_pseudo": "gradcam_head",
        "sam_clicks": "sam_direct_clicks",
        "sam_boxes": "sam_direct_boxes",
    }.items()
    if k not in SKIPPED
}
if "one_shot_sam" not in SKIPPED:
    COMPARE_SECONDS["one_shot_sam"] = proto_sec + proto_sam_sec
COMPARE_EFFORT = (  # annotation minutes for this run (section 4's effort_minutes and its assumptions)
    RESULTS["ladder_effort_min"]
    | RESULTS["path1_effort_min"]
    | RESULTS["path2_effort_min"]
    | RESULTS["path3_effort_min"]
    | {"full_masks": round(effort_minutes(n_full_masks=len(spec_train)), 2)}
)
COMPARE_PER_IMAGE = {  # annotation minutes for every *new* image; trained heads need none
    k: 0.0 for k in COMPARE_ORDER
} | {
    "zero_shot": effort_minutes(n_strokes=1),  # one box
    "sam_clicks": RESULTS["sam_direct_clicks"]["effort_min_per_image"],
    "sam_boxes": RESULTS["sam_direct_boxes"]["effort_min_per_image"],
}
SCALES = {
    "zero_shot": "a box on every new image, SAM per image, nothing to train",
    "one_shot": "one complete mask once; new images cost one feature pass",
    "one_shot_sam": "one complete mask once; SAM per new image, clicks picked automatically",
    "scribbles": "minutes of strokes once; new images cost one feature pass",
    "naive": "half-painted maps once (hours); new images cost one feature pass",
    "masked": "half-painted maps plus a few strokes once; new images cost one feature pass",
    "pu": "half-painted maps once plus a guessed share; new images cost one feature pass",
    "sam_pseudo": "strokes once, SAM at training time only; new images cost one feature pass",
    "gradcam_pseudo": "strokes once, a small classifier at training time; new images cost one feature pass",
    "sam_clicks": "strokes on every new image, SAM per image; the clicking never stops",
    "sam_boxes": "strokes on every new image, SAM per box; the clicking never stops",
    "full_masks": "hours of painting per training image; new images cost one feature pass",
}
WITH_CLDICE = "cldice" in SPECIALISED.get("metric_extras", [])
print(f"{len(COMPARE_ORDER)} approaches, each scored on the same {len(SPEC_EVAL)} held-out images:")
for k in COMPARE_ORDER:
    print(f"  {k:<16}{APPROACH_NAMES[k]}")
for k, why in SKIPPED.items():
    print(f"note: '{k}' is left out of this section, {why}. Run it to add the row.")
cost =SPECIALISED["annotation_effort"]
print(
    f"effort assumptions ({SPECIALISED['name']}): {cost['seconds_per_stroke']} s per stroke, click or box · "
    f"{cost['hours_per_full_mask'][0]}–{cost['hours_per_full_mask'][-1]} h per complete mask "
    f"(we count {np.mean(cost['hours_per_full_mask']):.0f} h)"
)

### 8.1 Look first, then predict

This cell shows every approach on one held-out image next to its hidden ground truth (green), each title in
the approach's colour: look for who finds the thin vessels, who paints blobs and who misses most.

In [ ]:
# @title All approaches on one held-out image { display-mode: "form" }
k_show = 0  # which held-out image to look at
it_show = SPEC_EVAL[k_show]
truth_img = overlay(it_show["image"], it_show["mask"], THEME["truth"])
gallery = [
    (f"{it_show['name']} · image", it_show["image"], "#311B92"),
    ("ground truth (scoring only)", truth_img, THEME["truth"]),
]
for k in COMPARE_ORDER:
    pred = COMPARE_REPORTS[k]["held_preds"][k_show] & region(it_show)
    colour = APPROACH_COLOURS[k]
    gallery.append((f"{COMPARE_LABEL[k]}\n{k}", overlay(it_show["image"], pred, colour), colour))
g_cols = 5
g_rows = -(-len(gallery) // g_cols)
fig, axes = plt.subplots(g_rows, g_cols, figsize=(2.6 * g_cols, 2.9 * g_rows))
for ax in np.ravel(axes):
    ax.axis("off")
for ax, (title, img, colour) in zip(np.ravel(axes), gallery, strict=False):
    ax.imshow(img)
    ax.set_title(title, fontsize=8, color=colour)
plt.tight_layout()
plt.show()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction (1 min)</b><p><strong>Q28.</strong> 🟡 Before the table appears: which approach will have the <strong>highest mean IoU</strong>, and which will be
the <strong>steadiest</strong> (the smallest spread of IoU from image to image)?</p>
<ul>
<li>Choose your two picks from the dropdowns in the next cell and run it; leave them blank to skip. The
  short names are the second line of each picture title above. The table further down shows where
  they landed.</li>
</ul></div>"""))

In [ ]:
# Q28 — your two predictions (pick from the dropdowns, or leave blank to skip)
MY_WINNER = ""  # @param ["", "zero_shot", "one_shot", "one_shot_sam", "scribbles", "naive", "masked", "pu", "sam_pseudo", "gradcam_pseudo", "sam_clicks", "sam_boxes", "full_masks"]
MY_STEADIEST = ""  # @param ["", "zero_shot", "one_shot", "one_shot_sam", "scribbles", "naive", "masked", "pu", "sam_pseudo", "gradcam_pseudo", "sam_clicks", "sam_boxes", "full_masks"]
for label, pick in (("MY_WINNER", MY_WINNER), ("MY_STEADIEST", MY_STEADIEST)):
    if not isinstance(pick, str) or (pick and pick not in ALL_APPROACHES):
        raise SystemExit(f'{label} must be "" or one of: {", ".join(ALL_APPROACHES)}.')
print(f"your picks · highest IoU: {MY_WINNER or '(none)'} · steadiest: {MY_STEADIEST or '(none)'}")

This helper cell defines `spread`, the population standard deviation of one approach's per-image IoUs
(divided by the number of images, as in the theory box above).

In [ ]:
# @title spread: the standard deviation of the per-image IoU { display-mode: "form" }
def spread(per_image_iou: list[float]) -> float:
    """Reference: numpy's standard deviation, which divides by n by default.
    e.g. [0.1, 0.3, 0.2, 0.6] -> 0.19"""
    return float(np.std(np.asarray(per_image_iou, dtype=float)))

### 8.2 The table

This cell scores every approach image by image against the hidden ground truth and shows the table sorted
by IoU (best value of each column in bold), then where your two picks landed.

In [ ]:
# @title Score every approach and show the table { display-mode: "form" }
comparison = []
PER_IMAGE_IOU = {}  # approach -> its IoU on each held-out image (drawn as violins below)
for k in COMPARE_ORDER:
    rep = COMPARE_REPORTS[k]
    pairs = zip(rep["held_preds"], SPEC_EVAL, strict=True)
    per_image = [iou(p & region(it), it["mask"] & region(it)) for p, it in pairs]
    PER_IMAGE_IOU[k] = per_image
    comparison.append(
        {
            "approach": k,
            "name": APPROACH_NAMES[k],
            "iou": float(np.mean(per_image)),
            **({"cldice": float(rep["cldice"])} if WITH_CLDICE else {}),
            "recall": float(rep["recall"]),
            "precision": float(rep["precision"]),
            "effort_min": float(COMPARE_EFFORT[k]),
            "effort_min_per_new_image": round(float(COMPARE_PER_IMAGE[k]), 2),
            "reliability": float(spread(per_image)),
            "worst_iou": float(np.min(per_image)),
            "seconds": round(float(COMPARE_SECONDS[k]), 1),
            "scalability": SCALES[k],
        }
    )
RESULTS["comparison_table"] = comparison
RESULTS["comparison_assumptions"] = dict(SPECIALISED["annotation_effort"]) | {
    "reliability": f"standard deviation of the IoU over {len(SPEC_EVAL)} held-out images",
    "seconds": f"training + pseudo-labels + inference in this session, on {DEVICE}",
}

from html import escape

ranked = sorted(comparison, key=lambda r: r["iou"], reverse=True)
THICK_IOU = RESULTS["sam_direct_strokes_alone"]["iou"]
TABLE_COLS = [  # header, row key, number format, which value is the best one
    ("IoU", "iou", "{:.3f}", max),
    *([("clDice", "cldice", "{:.3f}", max)] if WITH_CLDICE else []),
    ("recall", "recall", "{:.3f}", max),
    ("± spread", "reliability", "{:.3f}", min),
    ("worst", "worst_iou", "{:.3f}", max),
    ("effort min", "effort_min", "{:.1f}", min),
    ("/new img", "effort_min_per_new_image", "{:.2f}", min),
    ("sec", "seconds", "{:.1f}", min),
]


def comparison_html(rows: list[dict]) -> str:
    """The table as HTML: a colour chip per approach, the best value of each column in bold."""
    best = {}
    for _, key, fmt, pick in TABLE_COLS:
        top = fmt.format(pick(r[key] for r in rows))
        ties = sum(fmt.format(r[key]) == top for r in rows)
        best[key] = top if ties <= 3 else None  # when many rows tie, nobody is "the best"
    th = "padding:4px 8px;border-bottom:2px solid #311B92;color:#311B92;text-align:"
    td = "padding:3px 8px;border-bottom:1px solid #D1C4E9;text-align:"
    head = f'<th style="{th}left">approach</th>'
    head += "".join(f'<th style="{th}right">{h}</th>' for h, *_ in TABLE_COLS)
    lines = []
    for r in rows:
        chip = (
            '<span style="display:inline-block;width:11px;height:11px;border-radius:3px;margin-right:6px;'
            f'background:{APPROACH_COLOURS[r["approach"]]}"></span>'
        )
        cells = f'<td style="{td}left;white-space:nowrap">{chip}{escape(COMPARE_LABEL[r["approach"]])} '
        cells += f'<span style="color:#757575">{r["approach"]}</span></td>'
        for _, key, fmt, _pick in TABLE_COLS:
            text = fmt.format(r[key])
            text = f"<b>{text}</b>" if text == best[key] else text
            cells += f'<td style="{td}right">{text}</td>'
        lines.append(f"<tr>{cells}</tr>")
    for label, value in (
        ("no model: strokes thickened (section 7)", THICK_IOU),
        ("everything is foreground", RESULTS["specialised_everything_iou"]),
    ):
        grey = "color:#757575;font-style:italic"
        lines.append(
            f'<tr><td style="{td}left;{grey}">{label}</td><td style="{td}right;{grey}">{value:.3f}</td>'
            f'<td colspan="{len(TABLE_COLS) - 1}" style="{td}left"></td></tr>'
        )
    scales = "".join(f"<li><b>{r['approach']}</b>: {escape(r['scalability'])}</li>" for r in rows)
    return (
        '<table style="border-collapse:collapse;font-size:13px;margin:6px 0">'
        f"<tr>{head}</tr>{''.join(lines)}</table>"
        '<div style="font-size:12px;color:#616161;margin:2px 0 6px">recall = share of the true vessel pixels '
        "found · ± spread = std of the per-image IoU · worst = lowest per-image IoU · effort min = "
        "annotation minutes once · /new img = annotation minutes per new image · bold = best of its column"
        "</div>"
        '<details style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:8px 12px;'
        'color:#616161;font-size:13px"><summary><b>🔍 How each one scales</b></summary>'
        f'<ul style="margin:6px 0">{scales}</ul></details>'
    )


try:
    from IPython.display import HTML, display

    display(HTML(comparison_html(ranked)))
except ImportError:  # no IPython: a plain list with the same numbers
    for r in ranked:
        nums = " · ".join(f"{h} {fmt.format(r[key])}" for h, key, fmt, _ in TABLE_COLS)
        print(f"{r['approach']:<16}{nums}")
    print(f"no model: strokes thickened (section 7) {THICK_IOU:.3f}")
    print(f"everything is foreground {RESULTS['specialised_everything_iou']:.3f}")

winner = ranked[0]["approach"]
steadiest = min(comparison, key=lambda r: r["reliability"])["approach"]
by_spread = [r["approach"] for r in sorted(comparison, key=lambda r: r["reliability"])]
print(f"highest IoU: {winner} · steadiest: {steadiest}")
if MY_WINNER in by_spread:
    rank = [r["approach"] for r in ranked].index(MY_WINNER) + 1
    print(f"your IoU pick {MY_WINNER} ranks {rank} of {len(ranked)}")
if MY_STEADIEST in by_spread:
    rank = by_spread.index(MY_STEADIEST) + 1
    print(f"your steadiness pick {MY_STEADIEST} ranks {rank} of {len(by_spread)}")
for pick in {MY_WINNER, MY_STEADIEST} - {"", *by_spread}:
    print(f"(your pick {pick} was left out: {SKIPPED.get(pick, 'not run')})")
if not (MY_WINNER or MY_STEADIEST):
    print("no prediction written in Q28; fine, but next time guess first: it is the cheapest lesson here")

### 8.3 The trade-off in one picture

This cell draws the table: IoU against annotation minutes on the left (log scale, with a bar for each
spread), compute seconds on the right.

In [ ]:
# @title Trade-off plot: accuracy against annotation, and compute { display-mode: "form" }
fig, (ax, bx) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [1.25, 1]})
top = min(1.0, 1.25 * max(r["iou"] + r["reliability"] for r in comparison) + 0.05)
gap = 0.045 * top  # labels of approaches with about the same effort are stacked at least this far apart
groups, last = [], None
for r in sorted(comparison, key=lambda r: r["effort_min"]):
    lx = np.log10(max(r["effort_min"], 0.01))
    if last is None or lx - last > 0.35:
        groups.append([])
    groups[-1].append(r)
    last = lx
label_y = {}
for group in groups:
    y_prev = -1.0
    for r in sorted(group, key=lambda r: r["iou"]):
        label_y[r["approach"]] = y_prev = max(r["iou"], y_prev + gap)
for r in comparison:
    x, colour = max(r["effort_min"], 0.01), APPROACH_COLOURS[r["approach"]]
    ax.errorbar(x, r["iou"], yerr=r["reliability"], fmt="o", ms=7, capsize=3, color=colour)
    ax.annotate(
        r["approach"],
        (x, r["iou"]),
        xytext=(x * 1.25, label_y[r["approach"]]),
        fontsize=7,
        color=colour,
        va="center",
    )
ax.axhline(
    RESULTS["specialised_everything_iou"], color=THEME["background"], ls="--", label='IoU of "everything"'
)
ax.set_xscale("log")
ax.set_xlim(right=ax.get_xlim()[1] * 4)  # room for the labels of the right-most points
ax.set_xlabel("annotation effort, ≈ minutes (log scale)")
ax.set_ylabel("mean IoU on the hidden ground truth (± spread)")
ax.set_ylim(0, top)
ax.set_title("Accuracy against annotation")
ax.legend(fontsize=7, frameon=False, loc="upper left")
names = [r["approach"] for r in ranked][::-1]
bx.barh(names, [r["seconds"] for r in ranked][::-1], color=[APPROACH_COLOURS[k] for k in names])
bx.set_xlabel(f"seconds, training + inference ({DEVICE})")
bx.tick_params(axis="y", labelsize=7)
bx.set_title("Compute (sorted by IoU, best at the top)")
plt.tight_layout()
plt.show()

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFF8E1;border-left:6px solid #5D4037;border-radius:6px;padding:12px 14px;margin:6px 0;color:#5D4037;font-family:inherit;line-height:1.5"><b>🧾 Reading the plot</b><ul>
<li><strong>Top-left</strong> would be the dream (accurate and nearly free; on thin structures nobody is there),
  <strong>bottom-left</strong> is cheap and weak, and the <strong>far right</strong> is the complete-mask head, hours away.</li>
<li><strong>The vertical bar is the spread</strong>: a slightly lower dot with a much shorter bar can be the safer choice.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 More on reading the trade-off plot</b></summary><ul>
<li><strong>Same x position</strong> is the fairest comparison: same annotation, different use of it (the scribble head,
  the two pseudo-label heads and the direct SAM recipes all start from the same kind of strokes).</li>
<li><strong>Far right</strong>: the log axis hides how far away the complete-mask head is.</li>
<li><strong>Right panel</strong>: seconds matter little for the heads and a lot for anything that runs SAM on every image.</li>
</ul></details></div>"""))

**Q29.** 🟡 Pick one approach for each situation and name the column that decided it.

- (a) A screening study of 1,000 new images next month.
- (b) One difficult image a clinician needs today.
- (c) A project where a missed vessel is much worse than a false alarm.

Columns you can use: IoU, recall, spread, worst, effort min, /new img. Did the IoU winner win any of them?

<details><summary>A possible answer</summary>

- (a) A thousand images make the **/new img** column decisive. Strokes or a box on every image multiply
  their minutes by a thousand; a trained head pays its annotation once and then costs a feature pass per
  image. Among the heads, pick by IoU and spread.
- (b) For one image today the per-image rows are the natural candidates: no training, a minute of
  strokes, and you check the result yourself. Compare them with the thickened-strokes line before
  trusting SAM's share of the work.
- (c) When misses cost more than false alarms, IoU is the wrong column. Look at **recall** and at
  **worst**, and accept a head that over-marks if a person checks its output.

The IoU winner can win none of the three. That is the point of a table with more than one column.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>👥 Pairs, 4 min: defend a row</b><p>Use your Q29 answers. <strong>A</strong> defends one row for situation (a), <strong>B</strong> defends another row for situation
(c), one minute each, naming the column that decides. Then swap: attack the other's row.</p>
<ul>
<li>Which column did the defence rest on, and which column of the same row would hurt it most?</li>
<li>If you changed any setting earlier (threshold, number of clicks, box size), did a row of your table
  move compared with your neighbour's?</li>
</ul></div>"""))

<details><summary>What pairs usually find</summary>

- A good defence names a column, not a row: **/new img** for (a), **recall** or **worst** for (c).
- Every row has a weak column (hours of painting, clicking on every image, an over-marking habit), and the
  attacker's job is to find it.
- Two rows whose IoUs differ by less than their spread are a tie: choose by the cheaper column.

</details>

### 8.4 Who is steady, and who fails on some images

This cell draws one dot per held-out image and a violin around the dots for each approach (a wide violin
means the IoU varies a lot; the short bar is the median image).

In [ ]:
# @title Violin plot: each approach's IoU over the held-out images { display-mode: "form" }
fig, ax = plt.subplots(figsize=(10, 3.8))
for x, r in enumerate(ranked):
    vals = np.asarray(PER_IMAGE_IOU[r["approach"]], float)
    colour = APPROACH_COLOURS[r["approach"]]
    if len(vals) >= 2 and np.ptp(vals) > 1e-9:  # a violin needs some spread to draw
        body = ax.violinplot(vals, positions=[x], widths=0.8, showextrema=False)["bodies"][0]
        body.set_facecolor(colour)
        body.set_edgecolor(colour)
        body.set_alpha(0.3)
    ax.scatter(np.full(len(vals), x), vals, s=12, color=colour, zorder=3)
    ax.plot([x - 0.3, x + 0.3], [np.median(vals)] * 2, color=colour, lw=2.5, zorder=4)
line_all = RESULTS["specialised_everything_iou"]
ax.axhline(line_all, color=THEME["background"], ls="--", label='mean IoU of "everything"')
ax.set_xticks(range(len(ranked)), [r["approach"] for r in ranked], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("IoU on one held-out image")
ax.set_title("Per-image IoU, best mean on the left (dot = one image, bar = median)")
ax.legend(fontsize=7, frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

This cell shows the IoU winner's **worst**, **typical** (median) and **best** held-out image as error maps
(violet = found, orange = false alarm, green = missed).

In [ ]:
# @title Worst, typical and best image of the IoU winner { display-mode: "form" }
w_iou = np.asarray(PER_IMAGE_IOU[winner], float)
w_order = np.argsort(w_iou)
panels = []
w_picks = (("worst", w_order[0]), ("typical (median)", w_order[len(w_order) // 2]), ("best", w_order[-1]))
for tag, n in w_picks:
    it_n, pred_n = SPEC_EVAL[n], COMPARE_REPORTS[winner]["held_preds"][n]
    panels.append((f"{winner} · {tag} · IoU {w_iou[n]:.2f}", error_map(it_n["image"], pred_n, it_n)))
show_masks(panels, width=3.4)

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ When you see X, do Y: choosing an approach</b><ul>
<li><strong>When</strong> two approaches differ by less than their spread <strong>→</strong> call it a tie and pick the cheaper one.</li>
<li><strong>When</strong> the best approach needs strokes or a box on every new image and you have thousands of images
  <strong>→</strong> compare the /new img column first; a trained head pays its annotation once.</li>
<li><strong>When</strong> a missed structure costs more than a false alarm <strong>→</strong> rank by recall and worst image, not by
  mean IoU.</li>
<li><strong>When</strong> the IoU winner is also the most expensive row <strong>→</strong> ask what the next-cheapest row loses
  before you pay for it.</li>
</ul>
<p>Section 9 runs the cheapest rungs on your own images; section 10 turns this table into rules.</p></div>"""))

## 9. Your own images

Everything so far ran on our images, where a hidden ground truth told us who was right. This section is
the dress rehearsal for your next problem.

Pick one to three photos of your own (a leaf, a crack in a wall, cells, a satellite tile, a pet) and draw a
few strokes on what you want and on what you do not want. Then run three rungs of the ladder on them.

There is no ground truth here. You judge the masks the way you would on a real project: by eye, against
your strokes, and by where the three approaches disagree.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>scribble on your own images and run the ladder.</p></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: privacy</b><ul>
<li>Your images stay in your Colab session: they go to this session's <code>/content</code> folder, and nothing is
  written to Drive.</li>
<li>The canvas runs in your browser and sends your strokes only to your own Colab kernel.</li>
<li>We never use a public share link (such as Gradio's <code>share=True</code>): that would put your images on a
  public URL.</li>
<li>Even so, pick images you are allowed to process in a cloud notebook: no patient data, nothing under
  an NDA.</li>
</ul></div>"""))

This helper cell defines the tools of this section: shrinking an image, our example images, a drawing
canvas with two pens, and the code that turns a drawing (or a PNG you painted elsewhere) into a stroke
map: 0 nothing, 1 the thing, 2 background, as in section 4.

In [ ]:
# @title Your images: example, canvas, stroke decoding { display-mode: "form" }
import base64
import io
import uuid
from string import Template

from IPython.display import HTML, display

OWN_STROKES: dict[int, np.ndarray] = {}  # image number -> strokes saved from the canvas (0 / 1 thing / 2 bg)
CANVAS_PX = 640  # long side of the canvas on screen; your strokes are scaled back to the image
BRUSH_PX = 6  # pen width on screen
_CANVAS = Template("""
<div><b>$title</b><br>
<div style="position:relative;width:${cw}px;height:${ch}px">
<img src="data:image/jpeg;base64,$b64" width="$cw" height="$ch" style="position:absolute;left:0;top:0">
<canvas id="fmc$uid" width="$cw" height="$ch"
 style="position:absolute;left:0;top:0;cursor:crosshair;touch-action:none"></canvas>
</div>
<button id="fmt$uid">🖍️ the thing</button> <button id="fmb$uid">🖍️ background</button>
<button id="fmx$uid">clear</button> <button id="fms$uid">💾 save strokes</button>
<span id="fmm$uid">pen: the thing</span></div>
<script>
(function () {
  var c = document.getElementById("fmc$uid"), ctx = c.getContext("2d");
  var msg = document.getElementById("fmm$uid"), pen = "$thing", down = false;
  ctx.lineWidth = $brush; ctx.lineCap = "round"; ctx.lineJoin = "round";
  function xy(e) { var r = c.getBoundingClientRect(); return [e.clientX - r.left, e.clientY - r.top]; }
  c.addEventListener("pointerdown", function (e) {
    down = true; c.setPointerCapture(e.pointerId);
    var p = xy(e); ctx.strokeStyle = pen; ctx.beginPath(); ctx.moveTo(p[0], p[1]);
    ctx.lineTo(p[0], p[1]); ctx.stroke();
  });
  c.addEventListener("pointermove", function (e) {
    if (!down) return; var p = xy(e); ctx.lineTo(p[0], p[1]); ctx.stroke();
  });
  c.addEventListener("pointerup", function () { down = false; });
  document.getElementById("fmt$uid").onclick = function () {
    pen = "$thing"; msg.textContent = "pen: the thing";
  };
  document.getElementById("fmb$uid").onclick = function () {
    pen = "$bg"; msg.textContent = "pen: background";
  };
  document.getElementById("fmx$uid").onclick = function () {
    ctx.clearRect(0, 0, c.width, c.height); msg.textContent = "cleared (press save to send the empty canvas)";
  };
  document.getElementById("fms$uid").onclick = function () {
    if (!(window.google && google.colab && google.colab.kernel)) {
      msg.textContent = "saving needs Colab: use the PNG fallback below"; return;
    }
    msg.textContent = "saving...";
    google.colab.kernel.invokeFunction("fm_workshop.save_strokes", [$k, c.toDataURL("image/png")], {})
      .then(function () { msg.textContent = "✅ saved, now run the cell below the canvases"; })
      .catch(function (err) { msg.textContent = "❌ " + err; });
  };
})();
</script>
""")


def shrink(arr: np.ndarray, max_side: int) -> np.ndarray:
    """Resize so the long side is at most `max_side`: bilinear for an RGB image, nearest for a 2-D map."""
    h, w = arr.shape[:2]
    if max(h, w) <= max_side:
        return arr
    s = max_side / max(h, w)
    size = (max(1, round(w * s)), max(1, round(h * s)))
    resample = Image.BILINEAR if arr.ndim == 3 else Image.NEAREST
    return np.array(Image.fromarray(arr.astype(np.uint8)).resize(size, resample)).astype(arr.dtype)


def own_item(name: str, image: np.ndarray, source: str, auto=None, mask=None) -> dict:
    """One image of this section: `auto` = generated strokes (example only), `mask` = its ground truth
    (example only, for scoring, never for training: L-005)."""
    return {
        "name": name,
        "image": image,
        "source": source,
        "auto": auto,
        "mask": mask,
        "fov": np.ones(image.shape[:2], bool),  # no border to ignore: the whole image counts
        "scribbles": None,  # filled in 9.2: 0 nothing, 1 the thing, 2 background
    }


def shrink_strokes(strokes: np.ndarray, max_side: int) -> np.ndarray:
    """Like `shrink` for strokes, but thin strokes survive: a pixel keeps a stroke if its block has one"""
    if max(strokes.shape) <= max_side:
        return strokes
    out = np.zeros(shrink(strokes, max_side).shape, np.uint8)
    for v in (2, 1):  # the thing wins over the background where they meet
        layer = Image.fromarray(((strokes == v) * 255).astype(np.uint8))
        out[np.array(layer.resize((out.shape[1], out.shape[0]), Image.BOX)) > 0] = v
    return out


def example_items(max_side: int) -> list[dict]:
    """Our example: two held-out pets (compact) and one held-out retina image (thin vessels), with the strokes
    our scribble generator drew from their masks."""
    out = []
    for it in [load_item(PETS, meta) for meta in PETS["held"][:2]] + [SPEC_EVAL[0]]:
        item = own_item(
            it["name"],
            shrink(it["image"], max_side),
            "example",
            auto=shrink_strokes(it["scribbles"], max_side),
            mask=shrink(it["mask"], max_side),
        )
        if it.get("fov") is not None:  # the retina's black border is not scored
            item["fov"] = shrink(it["fov"], max_side)
        out.append(item)
    return out


def show_canvas(k: int, image: np.ndarray, title: str) -> None:
    """A drawing canvas over image k: two pens, clear, save. Saving calls `save_strokes` in this kernel."""
    small = shrink(image, CANVAS_PX)
    buf = io.BytesIO()
    Image.fromarray(small).save(buf, format="JPEG", quality=85)
    s = CANVAS_PX / max(image.shape[:2])
    ch, cw = round(image.shape[0] * s), round(image.shape[1] * s)
    html = _CANVAS.substitute(
        title=title,
        k=k,
        cw=cw,
        ch=ch,
        b64=base64.b64encode(buf.getvalue()).decode(),
        uid=uuid.uuid4().hex[:8],
        thing=THEME["scribble"],
        bg=THEME["background"],
        brush=BRUSH_PX,
    )
    display(HTML(html))


def decode_strokes(png: bytes, shape: tuple[int, int]) -> np.ndarray:
    """A stroke map (0 nothing, 1 the thing, 2 background) at `shape` from a PNG: either the canvas layer
    (transparent, teal = thing, grey = background) or a copy of your image painted elsewhere with pure red
    (thing) and pure blue (background)."""
    img = Image.open(io.BytesIO(png)).convert("RGBA").resize((shape[1], shape[0]), Image.NEAREST)
    r, g, b, a = (np.asarray(img, int)[..., i] for i in range(4))
    red, blue = (r > 200) & (g < 60) & (b < 60), (b > 200) & (r < 60) & (g < 60)
    out = np.zeros(shape, np.uint8)
    if (a < 255).any():  # a transparent layer: every opaque pixel is a stroke
        on = a > 127
        out[on] = 2  # grey (or blue)
        out[on & (((r < 79) & (g > 79)) | red)] = 1  # teal (or red)
    else:  # a painted copy of the photo: only the pure pen colours count
        out[red], out[blue] = 1, 2
    return out


def strokes_png(strokes: np.ndarray) -> bytes:
    """The other way round, as the canvas sends it: a transparent PNG with teal and grey strokes."""
    rgba = np.zeros((*strokes.shape, 4), np.uint8)
    for v, colour in ((1, THEME["scribble"]), (2, THEME["background"])):
        rgba[strokes == v] = [*np.round(hex_rgb(colour) * 255).astype(int), 255]
    buf = io.BytesIO()
    Image.fromarray(rgba).save(buf, format="PNG")
    return buf.getvalue()


def save_strokes(k, data_url: str) -> None:
    """Called by a canvas's 💾 button: decode the strokes of image k and keep them in this session."""
    k = int(k)
    OWN_STROKES[k] = decode_strokes(base64.b64decode(data_url.split(",", 1)[1]), OWN[k]["image"].shape[:2])


def show_strokes(items: list[dict]) -> None:
    """Each image next to its strokes: the thing in teal, background in grey, thickened for the figure."""
    panels = []
    for it in items:
        grow = max(1, max(it["image"].shape[:2]) // 256)
        thick = {v: ndi.binary_dilation(it["scribbles"] == v, iterations=grow) for v in (1, 2)}
        drawn = overlay(it["image"], thick[2], THEME["background"], 0.9)
        drawn = overlay(drawn, thick[1], THEME["scribble"], 0.9)
        panels += [(it["name"], it["image"]), (f"strokes · {it['stroke_source']}", drawn)]
    show_masks(panels, width=2.6)


if IN_COLAB:
    from google.colab import output

    output.register_callback("fm_workshop.save_strokes", save_strokes)  # the canvas talks to this kernel only

### 9.1 Pick one to three images

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 2 min): choose your images</b><p>By default the next cell uses our example: two held-out pets and one retina image, with strokes drawn by
our scribble generator. To use your own images:</p>
<ol>
<li>Tick <code>ASK_UPLOAD</code> in the next cell and run it.</li>
<li>Pick one to three JPEG or PNG files in the upload button that appears (press Cancel to keep the example).</li>
</ol>
<ul>
<li>Choose images that show the <em>same kind of thing</em>: the head is trained on all of them together.</li>
<li>Already uploaded them with the folder icon on the left? Put their paths in <code>MY_IMAGES</code> instead.</li>
</ul>
<p>Nothing here is mandatory.</p></div>"""))

This cell uploads your images only if `ASK_UPLOAD` is ticked (in Colab), shrinks each one to at most
1024 px on its long side, and shows them.

In [ ]:
ASK_UPLOAD = False  # @param {type:"boolean"}
# ASK_UPLOAD: set True to upload your own images; False uses the bundled example images.
USE_EXAMPLE = False  # True -> use our example images even if you uploaded or listed your own
MY_IMAGES = []  # or paths uploaded with the folder icon, e.g. ["/content/leaf1.jpg", "/content/leaf2.jpg"]
OWN_MAX_SIDE = 256 if SMOKE else 1024  # long side, in pixels, after shrinking
if not isinstance(ASK_UPLOAD, bool) or not isinstance(USE_EXAMPLE, bool) or not isinstance(MY_IMAGES, list):
    raise SystemExit("ASK_UPLOAD, USE_EXAMPLE must be True or False; MY_IMAGES a list of paths in quotes.")
missing = [p for p in MY_IMAGES if not Path(p).is_file()]
if missing:
    raise SystemExit(f"not found: {missing}. Upload with the folder icon on the left and copy each path.")

seed_everything()
OWN_STROKES.clear()  # a fresh start: strokes from an earlier run belong to earlier images
own_paths = [Path(p) for p in MY_IMAGES]
if ASK_UPLOAD and IN_COLAB and not USE_EXAMPLE and not own_paths:  # opt-in: Run all never waits here
    from google.colab import files

    print("Pick 1 to 3 images (JPEG or PNG), or press Cancel to use our example.")
    upload_dir = Path("/content/own_images")  # this session's disk only; gone when the session ends
    upload_dir.mkdir(exist_ok=True)
    try:
        uploaded = files.upload()
    except Exception as err:  # noqa: BLE001 - a cancelled upload raises in some Colab versions
        print(f"(upload stopped: {type(err).__name__})")
        uploaded = {}
    for fname, data in uploaded.items():
        (upload_dir / fname).write_bytes(data)
        own_paths.append(upload_dir / fname)
if len(own_paths) > 3:
    print(f"ℹ️ {len(own_paths)} images: keeping the first 3, so every step stays short.")
    own_paths = own_paths[:3]

OWN = []
for p in own_paths if not USE_EXAMPLE else []:
    try:
        OWN.append(own_item(p.stem, shrink(np.array(Image.open(p).convert("RGB")), OWN_MAX_SIDE), "yours"))
    except Exception as err:  # noqa: BLE001 - any unreadable file gets the same plain message
        print(f"⚠️ {p.name}: could not read it as an image ({type(err).__name__}), skipped. JPEG or PNG work.")
if not OWN:
    OWN = example_items(OWN_MAX_SIDE)
    print("ℹ️ Using our example: two held-out pets and one retina image, with generated strokes.")
sizes = ", ".join(f"{it['name']} {it['image'].shape[1]}×{it['image'].shape[0]}" for it in OWN)
print(f"{len(OWN)} image(s) · {sizes} · {OWN[0]['source']}")
show_masks([(it["name"], it["image"]) for it in OWN], width=3.0)

### 9.2 Draw your strokes 🖍️

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 4 min): scribble</b><p>Run the next cell: a canvas appears over each image.</p>
<ol>
<li>With the <strong>thing</strong> pen (teal), draw two to five short strokes <em>on</em> what you want segmented, on
   different parts of it.</li>
<li>With the <strong>background</strong> pen (grey), draw two to five strokes on what you do not want, especially near
   the boundary and on anything that looks similar.</li>
<li>Press <strong>💾 save strokes</strong> under each canvas, then run the "collect strokes" cell further down.</li>
</ol>
<p>Strokes, not outlines: a few seconds per image, the annotation budget of section 4. On the example,
drawing is optional: an image you do not save keeps its generated strokes.</p></div>"""))

This cell draws one canvas per image; the save button sends your strokes, as a small PNG, to this Colab
kernel, where they become a stroke map.

In [ ]:
# @title Draw your strokes on a canvas { display-mode: "form" }
for k, it in enumerate(OWN):
    show_canvas(k, it["image"], f"image {k + 1} of {len(OWN)} · {it['name']}")
if not IN_COLAB:
    print("ℹ️ Not in Colab: the canvases show, but saving needs Colab. Use the PNG fallback below.")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FDECEA;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;margin:6px 0;color:#8B1A10;font-family:inherit;line-height:1.5"><b>⚠️ Pitfall: no canvas, or the save button does nothing?</b><p>Some browsers or extensions block the canvas. The fallback:</p>
<ol>
<li>Open a copy of your image in any paint program (Preview, Paint, a phone's markup tool).</li>
<li>Draw the thing strokes in <strong>pure red</strong> (255, 0, 0) and the background strokes in <strong>pure blue</strong>
   (0, 0, 255), and save it as PNG.</li>
<li>Upload it with the folder icon on the left and put its path in <code>MY_STROKE_PNGS</code> below, one per image,
   in the order above (<code>""</code> skips one). Any size works.</li>
</ol></div>"""))

This "collect strokes" cell gathers the strokes of every image (your PNGs, then the canvas, then the
generated strokes for the example), counts them and shows them.

In [ ]:
MY_STROKE_PNGS = []  # fallback, e.g. ["/content/leaf1_strokes.png", ""]: one path (or "") per image, in order
if not isinstance(MY_STROKE_PNGS, list) or len(MY_STROKE_PNGS) > len(OWN):
    raise SystemExit(f"MY_STROKE_PNGS must be a list of at most {len(OWN)} paths, one per image, in order.")
for k, p in enumerate(MY_STROKE_PNGS):
    if p and not Path(p).is_file():
        raise SystemExit(f"{p} not found. Upload it with the folder icon on the left and copy its path.")
    if p:
        OWN_STROKES[k] = decode_strokes(Path(p).read_bytes(), OWN[k]["image"].shape[:2])

for k, it in enumerate(OWN):
    if k in OWN_STROKES:
        it["scribbles"], it["stroke_source"] = OWN_STROKES[k], "drawn by you"
    elif it["auto"] is not None:  # the example, not redrawn: generated strokes through the canvas decoder
        it["scribbles"] = decode_strokes(strokes_png(it["auto"]), it["image"].shape[:2])
        it["stroke_source"] = "generated"
    else:
        it["scribbles"], it["stroke_source"] = np.zeros(it["image"].shape[:2], np.uint8), "none yet"
no_thing = [it["name"] for it in OWN if not (it["scribbles"] == 1).any()]
if no_thing:
    print(f"⚠️ no 'thing' stroke on {no_thing}: skipped. Draw one, press 💾 save, and run this cell again.")
LADDER_OWN = [it for it in OWN if (it["scribbles"] == 1).any()]
if not any((it["scribbles"] == 2).any() for it in LADDER_OWN):
    print("⚠️ No background stroke on any image, and the head needs both kinds. Draw at least one background")
    print("   stroke and save. Until then the ladder runs on our example, so the notebook keeps going.")
    LADDER_OWN = []
if not LADDER_OWN:
    LADDER_OWN = example_items(OWN_MAX_SIDE)
    for it in LADDER_OWN:
        it["scribbles"] = decode_strokes(strokes_png(it["auto"]), it["image"].shape[:2])
        it["stroke_source"] = "generated"

own_n_strokes = {v: sum(len(stroke_masks(it, v)) for it in LADDER_OWN) for v in (1, 2)}
own_effort = effort_minutes(n_strokes=own_n_strokes[1] + own_n_strokes[2])
print(
    f"{len(LADDER_OWN)} image(s) · {own_n_strokes[1]} thing strokes, {own_n_strokes[2]} background strokes · "
    f"≈ {own_effort:.1f} min at {SPECIALISED['annotation_effort']['seconds_per_stroke']} s per stroke"
)
show_strokes(LADDER_OWN)

### 9.3 The ladder on your images

Three rungs, the same recipes as before, nothing new to learn.

- **Zero-shot**: SAM with one box, the box a person would draw around the thing (here: around your thing
  strokes, grown by 5 % of the image).
- **Your scribble head**: the section-4 head, a linear layer on DINOv3 patch features. It is trained with
  the partial cross-entropy on your strokes only (every unlabelled pixel ignored), on all your images
  together.
- **SAM from your strokes**: ten clicks along your thing strokes and ten along your background strokes,
  one SAM prompt per image (Path 3, section 7.1).

This helper cell defines the three rungs on a list of images, plus the only check we have without a ground
truth: how much of your thing strokes a mask covers, and how much of your background strokes it spares.

In [ ]:
# @title The ladder on your images: box, head on your strokes, SAM from clicks { display-mode: "form" }
OWN_APPROACHES = ("zero_shot", "scribbles", "sam_clicks")  # the colours and names of sections 4 and 7


def strokes_box(item: dict, pad: float) -> list[float]:
    """Zero-shot's box: around every 'thing' stroke, grown by `pad` × the long side, inside the image."""
    ys, xs = np.nonzero(item["scribbles"] == 1)
    h, w = item["scribbles"].shape
    g = pad * max(h, w)
    x0, y0 = max(0.0, xs.min() - g), max(0.0, ys.min() - g)
    return [float(x0), float(y0), float(min(w - 1.0, xs.max() + g)), float(min(h - 1.0, ys.max() + g))]


def own_ladder(items: list[dict], n_clicks: int, box_pad: float, feat_side: int, steps: int):
    """The three rungs on `items`. Returns masks per approach (one per image), seconds per approach, and the
    seconds the head took to train."""
    preds, sec = {k: [] for k in OWN_APPROACHES}, dict.fromkeys(OWN_APPROACHES, 0.0)
    for it in items:  # SAM: the image is encoded once, then both prompts reuse it
        t0 = time.monotonic()
        preds["zero_shot"].append(sam.probability(it["image"], box=strokes_box(it, box_pad)) >= 0.5)
        t1 = time.monotonic()
        pos, neg = stroke_points(it, 1, n_clicks), stroke_points(it, 2, n_clicks)
        prob = sam.probability(it["image"], points=pos + neg, labels=[1] * len(pos) + [0] * len(neg))
        preds["sam_clicks"].append(prob >= 0.5)
        sec["zero_shot"] += t1 - t0  # includes the image encoding
        sec["sam_clicks"] += time.monotonic() - t1
    t0 = time.monotonic()
    feats = [dino.features(it["image"], min(feat_side, max(it["image"].shape[:2]))) for it in items]
    # the rung's supervision: your strokes only; every unlabelled pixel is ignored (-1)
    labels = [scribble_targets(it["scribbles"], region(it)) for it in items]
    head, train_sec = train_head(feats, labels, steps)
    pairs = zip(feats, items, strict=True)
    preds["scribbles"] = [predict_head(head, f, it["image"].shape[:2]) for f, it in pairs]
    sec["scribbles"] = time.monotonic() - t0
    return preds, sec, train_sec


def stroke_agreement(pred: np.ndarray, item: dict) -> tuple[float, float]:
    """Share of the thing-stroke pixels the mask covers, and share of the background-stroke pixels it leaves
    out (nan if the image has no background stroke). Both are 1.0 for a mask that respects every stroke."""
    s = item["scribbles"]
    spared = 1.0 - float(pred[s == 2].mean()) if (s == 2).any() else float("nan")
    return float(pred[s == 1].mean()), spared

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>🤔 Your prediction (1 min)</b><p>Look at your images and strokes above. Is your thing an "object" in SAM's sense (compact, a clear inside
and outside), or more like the vessels (thin, scattered, a texture)?</p>
<ul>
<li>Write down which of the three rungs you expect to give the cleanest mask, and which the worst.</li>
<li>Then run the next cell (seconds on a T4, much slower on CPU): it shows each image with the three masks
  in the approach colours of sections 4 and 7, and the IoU on the example.</li>
</ul>
<p><strong>Q30.</strong> 🟡 Which rung gave the mask you would keep, image by image, and did it match your prediction?
Where a rung failed, was it:</p>
<ul>
<li>SAM's idea of an object (the box or the clicks grabbed the wrong thing, or too much)?</li>
<li>the 16 px patches of the head (blocky edges, small parts lost)?</li>
<li>your strokes (a part of the thing you never touched)?</li>
</ul></div>"""))

In [ ]:
# Q30 — the ladder on your images: zero-shot SAM, your scribble head, SAM from your strokes
N_OWN_CLICKS = 10  # clicks per kind (thing / background) and image, spread along your strokes
OWN_BOX_PAD = 0.05  # zero-shot box: your thing strokes' bounding box, grown by this share of the long side
OWN_FEAT_SIDE = 256 if SMOKE else 512  # DINOv3 resolution for the head (long side, pixels)
if not isinstance(N_OWN_CLICKS, int) or not 1 <= N_OWN_CLICKS <= 50:
    raise SystemExit("N_OWN_CLICKS must be a whole number from 1 to 50.")
if not isinstance(OWN_BOX_PAD, (int, float)) or not 0 <= OWN_BOX_PAD <= 0.5:
    raise SystemExit("OWN_BOX_PAD must be a number from 0 to 0.5 (a share of the image's long side).")

seed_everything()
own_preds, own_sec, own_train_sec = own_ladder(
    LADDER_OWN, N_OWN_CLICKS, OWN_BOX_PAD, OWN_FEAT_SIDE, HEAD_STEPS
)
if own_train_sec > 60:
    print("ℹ️ Slow session? The head trained for over a minute: fewer or smaller images make it faster.")
has_truth = all(it["mask"] is not None for it in LADDER_OWN)
own_report = {}
print(f"{'approach':<36}{'marks':>7}{'thing ✓':>9}{'bg ✓':>7}{'s':>6}" + (f"{'IoU':>7}" if has_truth else ""))
for k in OWN_APPROACHES:
    agree = [stroke_agreement(p, it) for p, it in zip(own_preds[k], LADDER_OWN, strict=True)]
    own_report[k] = {
        "marked": float(np.mean([p.mean() for p in own_preds[k]])),
        "thing_strokes_covered": float(np.mean([a[0] for a in agree])),
        "bg_strokes_spared": float(np.nanmean([a[1] for a in agree])),
        "seconds": round(own_sec[k], 1),
    }
    if has_truth:  # the example only: its ground truth scores, it never trains (L-005)
        pairs = zip(own_preds[k], LADDER_OWN, strict=True)
        own_report[k]["iou"] = float(
            np.mean([iou(p & region(it), it["mask"] & region(it)) for p, it in pairs])
        )
    v = own_report[k]
    print(
        f"{APPROACH_NAMES[k]:<36}{v['marked']:>7.0%}{v['thing_strokes_covered']:>9.0%}"
        f"{v['bg_strokes_spared']:>7.0%}{v['seconds']:>6.1f}" + (f"{v['iou']:>7.3f}" if has_truth else "")
    )
print("thing ✓ = share of your thing-stroke pixels inside the mask (100 % = all covered)")
print("bg ✓ = share of your background-stroke pixels outside the mask (100 % = none marked)")
for n, it in enumerate(LADDER_OWN):
    panels = [(f"{it['name']} · strokes", overlay(it["image"], it["scribbles"] > 0, THEME["scribble"], 0.9))]
    for k in OWN_APPROACHES:
        r_n = region(it)
        tag = f" · IoU {iou(own_preds[k][n] & r_n, it['mask'] & r_n):.2f}" if has_truth else ""
        title = f"{APPROACH_NAMES[k].split(':')[0]}{tag}"
        panels.append((title, overlay(it["image"], own_preds[k][n], APPROACH_COLOURS[k])))
    if has_truth:
        panels.append(("ground truth · scoring only", overlay(it["image"], it["mask"], THEME["truth"])))
    show_masks(panels, width=2.6)

RESULTS["own_images"] = {
    "source": LADDER_OWN[0]["source"],
    "n_images": len(LADDER_OWN),
    "n_strokes": own_n_strokes[1] + own_n_strokes[2],
    "effort_min": round(own_effort, 2),
    "head_train_sec": round(float(own_train_sec), 2),
    "approaches": own_report,
}
RESULTS["own_images_ok"] = all(len(own_preds[k]) == len(LADDER_OWN) for k in OWN_APPROACHES)  # SPEC §9

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 How to read the stroke checks and the masks</b></summary><ul>
<li>The two percentages are the only "score" you get on a real problem before anyone labels a test set, and
  they are weak. SAM's clicks sit <em>on</em> your strokes and the head was trained on them, so both should
  respect them. A low number means the approach ignored what you drew.</li>
<li>The masks matter more. For a compact, photo-like thing, expect the three rungs to agree, as on the
  pets in section 1: the box alone is often enough.</li>
<li>For a thin, scattered or textural thing, expect section 2 again. SAM's box grabs a region, the clicks
  give one blob or several, and the head draws the thing at patch resolution but in all the right places.</li>
<li>If your strokes covered only one part of the thing, both the head and the clicks tend to stop at that
  part. Information you did not give is information no rung has.</li>
</ul></details></div>"""))

### 9.4 No ground truth: look where they disagree

Without a ground truth, run two or three different approaches and look at the pixels where they
disagree. That is where the image is hard, and where one more stroke is worth the most.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#EDE7F6;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;margin:6px 0;color:#311B92;font-family:inherit;line-height:1.5"><b>📘 Theory: the disagreement map</b><p>The map is <code>True</code> on every pixel that <strong>some</strong> of the masks mark and <strong>others</strong> do not. Three masks over
four pixels:</p>
<pre><code>mask A      1 1 0 0
mask B      1 0 1 0
mask C      1 1 1 0
disagree    0 1 1 0
</code></pre>
<ul>
<li><strong>Low, a thin rim</strong> along the thing: the rungs agree on <em>what</em> it is and differ only on the exact edge.</li>
<li><strong>High, whole patches</strong>: they disagree on what belongs to the thing. That is where the next stroke pays.</li>
</ul></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (optional, 2 min): the disagreement map</b><p>Fill <code>my_disagreement</code>.</p>
<ul>
<li><strong>In:</strong> a list of boolean masks of the same image.</li>
<li><strong>Out:</strong> a boolean map, <code>True</code> where they do not all agree (some mark the pixel, some do not).</li>
<li><strong>Hint:</strong> <code>np.stack</code> puts the masks on a new first axis; <code>.any(0)</code> and <code>.all(0)</code> reduce along it.</li>
</ul>
<p>The cell checks your version on the three play masks above and continues with ours if you skip it.</p></div>"""))

In [ ]:
# @title Reference solution (open only if stuck) { display-mode: "form" }
def _ref_disagreement(masks: list[np.ndarray]) -> np.ndarray:
    """Reference: marked by some, not by all."""
    stack = np.stack([np.asarray(m, bool) for m in masks])
    return stack.any(0) & ~stack.all(0)

In [ ]:
# ✍️ YOUR CODE
def my_disagreement(masks: list[np.ndarray]) -> np.ndarray:
    """Boolean map, same shape as each mask: True where the masks disagree (at least one marks the pixel and
    at least one does not).
    e.g. three 1×4 masks [1, 1, 0, 0], [1, 0, 0, 0], [1, 1, 0, 1] -> [False, True, False, True]"""
    ...


_play_masks = [np.array([[1, 1, 0, 0]], bool), np.array([[1, 0, 1, 0]], bool), np.array([[1, 1, 1, 0]], bool)]
disagreement = use(
    my_disagreement,
    reference=_ref_disagreement,
    check=lambda f: np.asarray(f(_play_masks)).astype(int),
    name="disagreement",
)
print("play example · three masks over four pixels → disagreement:", disagreement(_play_masks).astype(int))

This cell computes the disagreement map of the three rungs on each of your images, prints the share of
the image it covers, and shows it in orange next to your strokes.

In [ ]:
# @title Disagreement map of the three rungs { display-mode: "form" }
own_disagree = [disagreement([own_preds[k][n] for k in OWN_APPROACHES]) for n in range(len(LADDER_OWN))]
RESULTS["own_images"]["disagreement"] = float(np.mean([d.mean() for d in own_disagree]))
panels = []
for it, d in zip(LADDER_OWN, own_disagree, strict=True):
    print(f"{it['name']}: the three rungs disagree on {d.mean():.0%} of the image")
    panels += [
        (f"{it['name']} · strokes", overlay(it["image"], it["scribbles"] > 0, THEME["scribble"], 0.9)),
        (f"disagreement · {d.mean():.0%}", overlay(it["image"], d, THEME["fails"])),
    ]
show_masks(panels, width=2.8)

**Q31.** 🟡 Where is the orange?

- Along the boundary of the thing?
- On whole parts of it?
- On something in the background that looks similar?

If you could add only two more strokes per image, where would you put them, and with which pen? On the
example, compare the retina's orange with the pets'.

<details><summary>What to look for</summary>

- A thin orange rim means the rungs agree on *what* it is and differ only on the exact edge: SAM's sharp
  boundary against the head's 16 px squares. That is a resolution question, and more strokes will not
  fix it.
- Whole orange parts mean they disagree on what belongs to the thing: usually a part you never stroked,
  or a background region that looks like it. That is exactly where the next stroke pays.
- Put a thing stroke on a missed part, or a background stroke on a false alarm. This is the idea behind
  interactive annotation and active learning: let the models show you where to look.

</details>

**Q32.** 🔴 Test your answer to Q31.

1. Draw your two extra strokes per image on the canvas of 9.2. Draw over the old ones; they stay until
   you press clear. On the example, a saved canvas replaces the generated strokes, so draw a full set.
2. Save, and run the cells from the stroke collection to here again.
3. Did the disagreement shrink where you drew? Did the head change elsewhere too?

<details><summary>What to look for</summary>

- SAM from clicks changes only on the image you drew on, near the new clicks.
- The head changes on every image, because it learns one rule from all the strokes together. A
  background stroke on a false alarm in one image can remove similar false alarms in another, or, if the
  two regions really are the same kind of thing, remove true parts too.
- That is the trade-off of section 8 in miniature: a trained head pays annotation once and spreads it
  everywhere, for better and for worse; direct prompting stays local.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>👥 Pairs, 5 min: compare your images</b><p>Show your neighbour your strokes, your three masks and your disagreement map.</p>
<ul>
<li>Whose thing was easier for the foundation model, and what made it easier: its shape, its contrast, how
  "photo-like" it is?</li>
<li>Did the same rung win on both your images?</li>
<li>Agree on one sign you would look for, on a new problem, that says "a box will do" versus "I need
  strokes and a head".</li>
</ul></div>"""))

<details><summary>What pairs usually find</summary>

- Compact, everyday objects (a mug, a pet, a car, a leaf on a plain background) make all three rungs
  agree, and the zero-shot box is often already the mask you would keep.
- Thin, repeated, textural or specialist things (cracks, cells, vessels, roads from above, defects) make
  them disagree, and the trained head or many small prompts start to matter.
- The sign to look for is the one from section 2: if the thing is not an "object" with a clear inside
  and outside, or is much smaller than a 16 px patch, plan for strokes.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E0F2F1;border-left:6px solid #004D40;border-radius:6px;padding:12px 14px;margin:6px 0;color:#004D40;font-family:inherit;line-height:1.5"><b>✅ When you see X, do Y: on your own images</b><ul>
<li><strong>When</strong> your thing is compact and photo-like and the three rungs agree <strong>→</strong> the zero-shot box is
  probably enough; stop there.</li>
<li><strong>When</strong> the rungs disagree on whole parts of the thing <strong>→</strong> add a thing stroke on the missed part or
  a background stroke on the false alarm, then rerun.</li>
<li><strong>When</strong> only a thin rim disagrees <strong>→</strong> that is the 16 px patch against SAM's sharp edge; more strokes
  will not fix it.</li>
<li><strong>When</strong> there is no ground truth <strong>→</strong> use your strokes and the disagreement map as a sanity check,
  then label a few images completely so the next decision is measured.</li>
</ul>
<p>Section 10 turns all of this into rules you can take to the next problem.</p></div>"""))

## 10. Guiding principles & wrap-up

A table is good for one decision; a rule is what you take to the next problem. This last section reads the
numbers *your* session produced in sections 1 to 8 and writes them up as a handful of principles, each
with its evidence.

They are not general truths about foundation models. They are what this run, on this task, with these
settings, showed. If your neighbour's session ran with other settings, some sentences may read
differently, and that is fine.

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8F5E9;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1B5E20;font-family:inherit;line-height:1.5"><b>🎯 Objective</b><p>turn the results into rules you can reuse.</p></div>"""))

This cell draws the whole story in one picture: the pets one box got almost right, the same box on the
specialised task, and the best approach of every path on the same hidden ground truth.

In [ ]:
# @title The story in one picture { display-mode: "form" }
TABLE = {r["approach"]: r for r in RESULTS["comparison_table"]}  # section 8's rows, by short name
EVERYTHING = RESULTS["specialised_everything_iou"]
PATHS = {  # the rows of section 8, grouped the way the notebook introduced them
    "ladder (section 4)": ("one_shot", "one_shot_sam", "scribbles"),
    "path 1 (section 5)": ("naive", "masked", "pu"),
    "path 2 (section 6)": ("sam_pseudo", "gradcam_pseudo"),
    "path 3 (section 7)": ("sam_clicks", "sam_boxes"),
}
story = [("pets · one box (section 1)", RESULTS["pets_zero_shot_iou"], THEME["works"])]
story.append(
    (f"{SPECIALISED['name']} · one box (section 2)", TABLE["zero_shot"]["iou"], APPROACH_COLOURS["zero_shot"])
)
for path, keys in PATHS.items():
    best = max(keys, key=lambda k: TABLE[k]["iou"])
    story.append((f"best of {path}: {best}", TABLE[best]["iou"], APPROACH_COLOURS[best]))
story.append(("complete masks (section 2)", TABLE["full_masks"]["iou"], APPROACH_COLOURS["full_masks"]))

fig, ax = plt.subplots(figsize=(8, 3.4))
labels, values, colours = zip(*story[::-1], strict=True)
ax.barh(labels, values, color=colours)
for y, v in enumerate(values):
    ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=8, color="#311B92")
ax.axvline(EVERYTHING, color=THEME["background"], ls="--", label='IoU of "everything" (specialised task)')
ax.set_xlim(0, 1.08)
ax.set_xlabel("mean IoU on the held-out images")
ax.set_title("Where it worked, where it fell short, how far each path got back")
ax.tick_params(axis="y", labelsize=8)
ax.legend(fontsize=7, frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

### 10.1 The principles, from your numbers

The next cell writes six rules as cards, each with the evidence from your session (a rule with your own
number attached tells you how strongly to believe it), and says so where a lesson did not hold in your run.

In [ ]:
# @title Principles from this run { display-mode: "form" }
import textwrap


def p_check_first() -> str:
    """Outcome 1: is the foundation model good enough out of the box?"""
    pets, spec = RESULTS["pets_zero_shot_iou"], RESULTS["specialised_zero_shot_iou"]
    rule = "Test the foundation model on your own task first, and read its score against marking everything."
    fact = (
        f"One box scored IoU {pets:.2f} on pets and {spec:.2f} on {SPECIALISED['name']}, where a mask over "
        f"the whole image scores {EVERYTHING:.2f}."
    )
    if spec < EVERYTHING + 0.1:
        return f"{rule} {fact} Out of the box it was close to doing nothing useful here."
    return f"{rule} {fact} It did part of the job out of the box; adapt only if the rest matters."


def p_resolution() -> str:
    """Section 2: the patch is 16 px; does a larger input help?"""
    res = sorted((int(k.split()[0]), v) for k, v in RESULTS["resolution_effect"].items())
    (lo, a), (hi, b) = res[0], res[-1]
    if SPECIALISED.get("thin_structures"):
        rule = "Thin structures defeat patch features: check the resolution first."
    else:
        rule = "Check the resolution first: a patch feature describes a 16 px block, not a pixel."
    fact = (
        f"The same head on complete masks scored IoU {a['iou']:.2f} at {lo} px and {b['iou']:.2f} at {hi} px "
        f"(clDice {a['cldice']:.2f} → {b['cldice']:.2f}); on pets the same kind of head scored "
        f"{RESULTS['pets_head_iou']:.2f}."
    )
    if b["iou"] - a["iou"] < 0.05:
        return f"{rule} {fact} More pixels helped little, so the limit is the patch: plan for cheap labels."
    return f"{rule} {fact} More pixels helped, so try the larger input before anything more expensive."


def p_missing_is_unknown() -> str:
    """Sections 3 and 5: missing labels are unknown, not background."""
    naive, masked = RESULTS["naive_recall_hidden_fg"], RESULTS["masked_recall_hidden_fg"]
    rule = "An unlabelled pixel is unknown, not background: tell the loss so."
    fact = (
        f"On the same half-painted maps, the head that read the gaps as background found {naive:.0%} of the "
        f"hidden foreground, the masked loss {masked:.0%}."
    )
    if masked <= naive:
        return f"{rule} {fact} Here ignoring the gaps did not recover more; check what the strokes cover."
    pn, pm = TABLE["naive"]["precision"], TABLE["masked"]["precision"]
    if pm < pn:
        return f"{rule} {fact} The price was precision ({pn:.2f} → {pm:.2f}): fewer misses, more alarms."
    return f"{rule} {fact} And it kept its precision ({pn:.2f} → {pm:.2f})."


def p_cheapest_baseline() -> str:
    """Section 7: compare SAM with the hints alone."""
    alone = RESULTS["sam_direct_strokes_alone"]
    sam_rows = [k for k in ("zero_shot", "one_shot_sam", "sam_clicks", "sam_boxes") if k in TABLE]
    best = max(sam_rows, key=lambda k: TABLE[k]["iou"])
    gain = TABLE[best]["iou"] - alone["iou"]
    rule = "Compare the foundation model with the cheapest baseline built from the same hints."
    fact = (
        f"The strokes thickened by {alone['thicken_px']} px, no model at all, scored IoU {alone['iou']:.2f}; "
        f"the best SAM recipe, {best}, scored {TABLE[best]['iou']:.2f}."
    )
    if gain <= 0:
        return f"{rule} {fact} The model added nothing the strokes did not already say."
    if gain < 0.05:
        return f"{rule} {fact} The model added only {gain:.2f}."
    return f"{rule} {fact} The model earned its place, by {gain:.2f}."


def p_one_number() -> str:
    """Section 8: no single column decides."""
    rows = RESULTS["comparison_table"]
    by_iou = max(rows, key=lambda r: r["iou"])
    by_recall = max(rows, key=lambda r: r["recall"])
    steady = min(rows, key=lambda r: r["reliability"])
    rule = "Choose by what a mistake costs, not by one number."
    fact = (
        f"Highest IoU: {by_iou['approach']} ({by_iou['iou']:.2f}); highest recall: {by_recall['approach']} "
        f"({by_recall['recall']:.2f}); smallest spread: {steady['approach']} ({steady['reliability']:.2f})."
    )
    leaders = {by_iou["approach"], by_recall["approach"], steady["approach"]}
    if len(leaders) == 1:
        return f"{rule} {fact} This time one approach led all three; check that before assuming it."
    lead = f"{len(leaders)} different approaches lead the three columns"
    return f"{rule} {fact} {lead}, so the right row depends on which mistake costs most."


def p_minutes_before_hours() -> str:
    """Section 8: annotation effort, once and per new image."""
    full = TABLE["full_masks"]
    cheap_rows = [r for r in RESULTS["comparison_table"] if r["effort_min"] < full["effort_min"] / 10]
    cheap = max(cheap_rows or RESULTS["comparison_table"], key=lambda r: r["iou"])
    per_image = max(RESULTS["comparison_table"], key=lambda r: r["effort_min_per_new_image"])
    rule = "Spend minutes before hours: start with strokes, and buy complete masks only if the table says so."
    fact = (
        f"Complete masks cost {full['effort_min']:.0f} min of annotation for IoU {full['iou']:.2f}; "
        f"{cheap['approach']} needed {cheap['effort_min']:.1f} min for IoU {cheap['iou']:.2f}."
    )
    if cheap["iou"] >= full["iou"]:
        verdict = "The cheap labels matched or beat the expensive ones."
    else:
        ratio = full["effort_min"] / max(cheap["effort_min"], 0.01)
        verdict = f"The hours bought {full['iou'] - cheap['iou']:.2f} IoU for {ratio:.0f}× the effort."
    scale = (
        f"And count per new image: {per_image['approach']} costs "
        f"{per_image['effort_min_per_new_image']:.2f} min on every one, a trained head none."
    )
    return f"{rule} {fact} {verdict} {scale}"


PRINCIPLES = [
    p_check_first(),
    p_resolution(),
    p_missing_is_unknown(),
    p_cheapest_baseline(),
    p_one_number(),
    p_minutes_before_hours(),
]
RESULTS["principles"] = PRINCIPLES
headline = f"Principles from this session ({SPECIALISED['name']}, {len(SPEC_EVAL)} held-out images, {DEVICE})"
try:
    from html import escape

    from IPython.display import HTML, display

    cards = ""
    for n, text in enumerate(PRINCIPLES, start=1):
        rule, _, evidence = text.partition(". ")  # the rule is the first sentence, the rest is the evidence
        cards += (
            f'<div style="flex:1 1 300px;background:#EDE7F6;border-left:6px solid {THEME["works"]};'
            'border-radius:6px;padding:10px 12px;color:#311B92;line-height:1.45">'
            f'<div style="font-weight:bold;font-size:15px">{n}. {escape(rule.rstrip("."))}.</div>'
            f'<div style="font-size:12px;margin-top:5px;color:#4527A0">{escape(evidence)}</div></div>'
        )
    display(
        HTML(
            f'<div style="font-weight:bold;color:#311B92;margin:4px 0">{escape(headline)}</div>'
            f'<div style="display:flex;flex-wrap:wrap;gap:8px">{cards}</div>'
        )
    )
except ImportError:  # no IPython: plain text
    print(headline + ":\n")
    for n, text in enumerate(PRINCIPLES, start=1):
        print(textwrap.fill(text, width=100, initial_indent=f"{n}. ", subsequent_indent="   "), end="\n\n")

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#FFE8D6;border-left:6px solid #7A2E00;border-radius:6px;padding:12px 14px;margin:6px 0;color:#7A2E00;font-family:inherit;line-height:1.5"><b>✏️ Your turn (2 min)</b><p><strong>Q33.</strong> 🟢 Think of one segmentation or detection problem of your own (work, research, or the images of
section 9). Which of the principles above would have changed what you did first, and what number would you
measure to check it?</p>
<ul>
<li>Write the problem and your first step on Monday, as a rule with a number, in the next cell.</li>
<li>The cell prints them next to the principle you picked and checks nothing.</li>
</ul></div>"""))

In [ ]:
# Q33 — your problem and your first step (leave "" to skip)
MY_PROBLEM = ""  # e.g. "cracks in drone photos of bridges"
MY_PRINCIPLE = 0  # the number of the principle above that changes your plan (1–6), or 0 for none
MY_FIRST_STEP = ""  # e.g. "one box on 20 images, compared with the 'everything' line"
if not isinstance(MY_PRINCIPLE, int) or not 0 <= MY_PRINCIPLE <= len(PRINCIPLES):
    raise SystemExit(f"MY_PRINCIPLE must be a whole number from 0 to {len(PRINCIPLES)}.")
if not (isinstance(MY_PROBLEM, str) and isinstance(MY_FIRST_STEP, str)):
    raise SystemExit('MY_PROBLEM and MY_FIRST_STEP must be text in quotes, e.g. "my problem".')
if MY_PROBLEM or MY_FIRST_STEP:
    print(f"problem: {MY_PROBLEM or '(not written)'}\nfirst step: {MY_FIRST_STEP or '(not written)'}")
    if MY_PRINCIPLE:
        print(textwrap.fill(PRINCIPLES[MY_PRINCIPLE - 1], width=100, initial_indent="because: "))
else:
    print("nothing written; fine. Keep the question for the train home: what would you measure first?")

<details><summary>A possible answer</summary>

Take cracks in drone photos of bridges. They are thin, like vessels, so principle 2 changes the plan.

1. Run one box and one feature head on ten images at two input sizes.
2. See whether the larger size moves clDice at all.
3. If it does not, the patch is the limit: spend the first afternoon on strokes for twenty images, not
   on a bigger model.

The number to measure is the gap between the head and the "everything" line on a small test set you
label completely, a few images, once. A different problem gives a different first step; what carries
over is writing the step with the number that would prove it wrong.

</details>

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F3E5F5;border-left:6px solid #4A148C;border-radius:6px;padding:12px 14px;margin:6px 0;color:#4A148C;font-family:inherit;line-height:1.5"><b>❓ Show of hands</b><ul>
<li>Which principle would you have got wrong this morning, before section 1?</li>
<li>Which one would you most like to check on your own data next week?</li>
</ul></div>"""))

### 10.2 What you can do now

At the end you can:

1. **Judge when a vision foundation model is good enough out of the box**, and recognise the signs that it
   is not (sections 1 and 2).
2. **Adapt it with cheap, incomplete supervision**, and explain what each approach uses and assumes
   (sections 3 to 7).
3. **Compare approaches** on accuracy, annotation effort, reliability and compute, and decide what to do
   next on a new problem, including your own images (sections 8 and 9).

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Go deeper · six questions for any task</b></summary><ul>
<li>Is the out-of-the-box result above the "everything" line?</li>
<li>Is the structure smaller than a 16 px patch?</li>
<li>What do the gaps in the labels mean?</li>
<li>Does the model beat the hints alone?</li>
<li>Which mistake costs most?</li>
<li>How many minutes does every new image cost?</li>
</ul></details></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Go deeper · where to go next</b></summary><ul>
<li><strong>Label a small test set first</strong>: a few fully annotated images turn every later decision into a
  measurement, as the hidden ground truth did here.</li>
<li><strong>Let the model choose the next stroke</strong>: draw where the approaches disagree (section 9), retrain, repeat.</li>
<li><strong>A light decoder instead of a linear head</strong>: a small upsampling decoder on the same frozen features
  comes before any fine-tuning.</li>
<li><strong>The 🔴 parts you skipped</strong>: PU loss (5), Grad-CAM peaks as clicks (6), SAM 3 from a word (7), extra
  strokes (9).</li>
<li><strong>Sources</strong>: <a style="color:inherit;text-decoration:underline" href="https://github.com/facebookresearch/dinov3">DINOv3</a> ·
  <a style="color:inherit;text-decoration:underline" href="https://github.com/facebookresearch/sam2">SAM 2</a> ·
  <a style="color:inherit;text-decoration:underline" href="https://github.com/jacobgil/pytorch-grad-cam">pytorch-grad-cam</a></li>
</ul></details></div>"""))

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#E8EAF6;border-left:6px solid #1A237E;border-radius:6px;padding:12px 14px;margin:6px 0;color:#1A237E;font-family:inherit;line-height:1.5"><b>🔀 Across approaches, one last time</b><p>The question on the first screen was what modern AI solves out of the box, when its limits start to
matter, and what to do next.</p>
<p>Your session's answer is in the six principles above, with numbers. The foundation model was a starting
point, not an answer, and every way out paid for its gain in a different currency: minutes of strokes, a
wrong assumption about the gaps, a second model's mistakes, or a person pointing at every image.</p>
<p>On your next problem, measure first, spend minutes before hours, and keep the table. Thank you for taking
part.</p></div>"""))

In [ ]:
# @title Credits: models and datasets of this session { display-mode: "form" }
cfg = SPECIALISED
print(f"DINOv3 ViT-S/16 · {DINO_SOURCE} · {DINO_LICENCE}")
print(f"SAM 2.1 hiera-small · {SAM_ID} · {SAM_LICENCE}")
print(f"{PETS['name']} · {PETS['licence']}")
print(f"{cfg['name']} · {cfg['licence']} ({cfg['licence_url']})")
print(textwrap.fill(cfg["citation"], width=100, initial_indent="  ", subsequent_indent="  "))
RESULTS["credits"] = {
    "models": {DINO_SOURCE: DINO_LICENCE, SAM_ID: SAM_LICENCE},
    "datasets": {PETS["name"]: PETS["licence"], SPECIALISED["name"]: SPECIALISED["licence"]},
}

In [ ]:
# @title
from IPython.display import HTML, display
display(HTML(r"""<div style="background:#F5F5F5;border-left:6px solid #616161;border-radius:6px;padding:12px 14px;margin:6px 0;color:#616161;font-family:inherit;line-height:1.5"><details><summary><b>🔍 Credits &amp; licences</b></summary><ul>
<li><strong>DINOv3</strong> ViT-S/16 (Meta), via the <code>timm</code> re-host
  <a style="color:inherit;text-decoration:underline" href="https://huggingface.co/timm/vit_small_patch16_dinov3.lvd1689m">timm/vit_small_patch16_dinov3.lvd1689m</a>.</li>
<li>DINOv3 is licensed under the
  <a style="color:inherit;text-decoration:underline" href="https://huggingface.co/timm/vit_small_patch16_dinov3.lvd1689m">DINOv3 License</a>, Copyright © Meta
  Platforms, Inc. All Rights Reserved. It allows research and commercial use under Meta's conditions and
  acceptable-use policy; keep this notice if you redistribute the weights or anything derived from them.
  SAM 2.1 is Apache-2.0.</li>
<li><strong>SAM 2.1</strong> hiera-small (Meta),
  <a style="color:inherit;text-decoration:underline" href="https://huggingface.co/facebook/sam2.1-hiera-small">facebook/sam2.1-hiera-small</a>. SAM 3, if you used
  it, comes under Meta's own SAM 3 licence, which you accepted on Hugging Face.</li>
<li><strong>Oxford-IIIT Pet</strong>: Parkhi, Vedaldi, Zisserman and Jawahar, <em>Cats and Dogs</em>, CVPR 2012,
  <a style="color:inherit;text-decoration:underline" href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>. The subset shipped with this workshop
  is resized and shared under the same licence.</li>
<li><strong>pytorch-grad-cam</strong> (Jacob Gildenblat and contributors), MIT.</li>
<li>The specialised dataset is printed by the credits cell above, read from the dataset config.</li>
<li>Every derived file (scribbles, hidden labels, pseudo-labels, features) carries its source's licence.
  The full list is in <code>data/ATTRIBUTION.md</code>.</li>
</ul></details></div>"""))